# NB06V3 v1 -- Geospatial Statistics & Feature Analysis (V3 per-point sample unit)

> Copyright (C) 2024-2026 Marco Heinzen - SPDX-License-Identifier: AGPL-3.0-or-later
> Part of the Master Thesis "Building Damage Assessment with Multimodal Satellite Time Series and Machine Learning in the Russia-Ukraine War 2022-2026"
> Code hosted at https://github.com/marcoheinzen/bda
> Parts of this code were written or improved with the assistance of Claude (Anthropic); all other code, and the concept, research, architecture, design, execution, testing and validation throughout, are the author's work.



| Field | Value |
|---|---|
| Notebook | `06_bda_Satellite_Statistics_V3_v1.ipynb` |
| Version | `V3 v1` |
| Pipeline position | After NB05bV3 v6, parallel to NB6V2 v9. Consumes V3 (per-point) parquets. |
| Last validated | `2026-04-28` |

## NB06V3 v1 vs NB6V2 v9

Same statistical analysis pipeline (descriptive + inferential statistics, AUC/Mann-Whitney
per feature, multicollinearity, PCA, separability, transition matrices, KDE damage hotspots,
per-parquet AUC + VIF + separability, manifest-driven AUC comparison, rolling+block product
analysis, single-scene + accumulator analysis), same 19 + 13 = 32 cells of analysis, same
output structure and registry. The single change is the sample unit:

- **NB6V2 (footprint-based):** one row per (city, building_id), feature value is the v29
  statistic-aware zonal aggregate (mean, p10, p50, p90, std, min, max, max_abs_delta) over
  the Overture footprint pixels. Sample-unit parquet: `bda_buildings_t{tier}.parquet`.
- **NB06V3 (point-based):** one row per (city, point_id), feature value is the single-pixel
  sample at the point's (row, col). Sample-unit parquet: `bda_points_t{tier}.parquet`.
  Same `damage_binary` target.

Source for the architectural distinction: `DECISION_V3.md` and
`METHODOLOGY_Pivot_PointFirst_and_Aggregation.md`.

### Why a parallel V3 statistics notebook

The V3 audit (NB08cV3 v1) produced honest per-point AUCs and revealed the matched-filter
spatial-distribution effect: V3 wins on raw per-scene optical parquets (where V2 footprint
mean attenuated the per-pixel anomaly) but collapses on every wide accumulator and SAR
parquet. NB06V3 runs the same depth of statistical analysis at per-point granularity that
NB6V2 ran at per-footprint granularity, producing a parallel set of:

- per-feature AUC distributions (Cell 4)
- VIF + Spearman multicollinearity (Cell 9)
- PCA per modality (Cell 10)
- J-M + Bhattacharyya + chi-square separability (Cell 11)
- per-parquet AUC + VIF + separability deep dives (Cells P1-P23)
- temporal AUC trajectories on long parquets (Cells P10-P23)
- manifest-driven AUC comparison across all V3 parquets (Cell 16)
- rolling + block + COH drop feature analysis (Cell 17)
- single-scene + accumulator product analysis (Cell 18)

### Changes from NB6V2 v9 (purely surgical)

- Sample-unit column: `'building_id'` -> `'point_id'` everywhere it is a column accessor
  (load helper merges, get_analysis_df, P1b/P1c rolling+block analysis joins).
- Dataset path: `STACK_DIR / 'dataset' / 'v2'` -> `STACK_DIR / 'dataset' / 'v3'`.
- Variable names: `DATASET_ROOT_V2` -> `DATASET_ROOT_V3`, `V2_DIR` -> `V3_DIR`,
  `load_v2_parquet` -> `load_v3_parquet`, `V2_PARQUET_INFO` -> `V3_PARQUET_INFO`.
- Sample-unit parquet: `bda_buildings_t{tier}.parquet` -> `bda_points_t{tier}.parquet`,
  `df_buildings` -> `df_points`. The sample-unit dataframe naming reflects what it
  actually holds.
- Output dir: `RESULTS_ROOT / 'nb06'` -> `RESULTS_ROOT / 'nb06v3'` (parallel directory
  so V2 and V3 statistics outputs do not collide).
- Registry tag: `notebook='NB06 v2'` -> `notebook='NB06V3 v1'`.

### What this notebook does NOT change

- It does not modify the V3 dataset builder (NB05bV3 v6). V3 parquets must already exist in
  `STACK_DIR/dataset/v3/` with a valid `parquet_manifest.json`.
- It does not touch any V2 outputs. NB6V2 v9 outputs remain in `RESULTS_ROOT/nb06/`.
- It does not propose to merge V2 and V3 statistics. They run on different sample units.
- Internal variable names like `df_pq`, `merged`, `is_w` remain unchanged per the
  no-rename project convention.

# CELL 1: NB06 CONFIG

In [1]:
# @title CELL 1: NB06V3 v1 CONFIG
TIER_SELECTION = [0,1,2]
CITY_SELECTION = None
REQUIRE_DAMAGE = True
REQUIRE_ML_READY = False
TARGET_COL = 'damage_binary'
MIN_SAMPLES = 20
MIN_FEATURE_COVERAGE = 0.3
FILTER_UNOSAT_ONLY = True
BALANCE_CLASSES = False

# V3: which parquets to analyze (None = all from manifest)
PARQUET_SELECTION = None  # e.g. ['scene_ms', 'block_stats', 'fusion_ms_card']

# CELL 2: GLOBAL SETUP

In [2]:
# @title CELL 2: GLOBAL SETUP
import platform, os
if platform.system() == 'Windows':
    _setup = r'F:\PROJECTS\masterthesis\gdrive\masterthesis\notebooks\global_setup.py'
elif os.path.exists('/content/drive_f'):
    _setup = '/content/drive_f/masterthesis/notebooks/global_setup.py'
else:
    _setup = '/mnt/f/PROJECTS/masterthesis/gdrive/masterthesis/notebooks/global_setup.py'
with open(_setup) as f:
    exec(f.read())


BDA GLOBAL SETUP
Started: 2026-04-29 08:00:45
Python: 3.12.12

[1/7] Directory Structure
----------------------------------------------------------------------


/home/alpineobotics/miniconda3/envs/bda/lib/python3.12/site-packages/pyproj/network.py:59: UserWarning: pyproj unable to set PROJ database path.
  _set_context_ca_bundle_path(ca_bundle_path)


  GDrive (G:):       /content/drive_f/masterthesis OK
  GDrive (F:):       /content/drive_f/masterthesis OK
  Local data (G:):   /content/masterthesis_local/data OK
  Data stack (F:):   /mnt/f/PROJECTS/masterthesis/data_stack OK

  TIER_SELECTION: [0, 1, 2]
  CITY_SELECTION: None (tier filter)
  REQUIRE_UNOSAT: False
  CITIES_TO_PROCESS: 21 cities

[2/7] Credentials
----------------------------------------------------------------------
  Copernicus: inf***
  OpenTopography: OK
  Earthdata: marcoheinzen

[3/7] Python Packages
----------------------------------------------------------------------


<string>:564: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.



  Already installed: 23
  Newly installed:   0
  Failed:            0

[4/7] Global Imports & Configuration
----------------------------------------------------------------------
  All imports loaded

[5/7] Processing Config & SNAP
----------------------------------------------------------------------
  GPT: Usage:
  Temporal baseline: 10-24 days
  Wavelength: 0.0555

[6/7] GPU Status
----------------------------------------------------------------------
  CUDA available: NVIDIA GeForce RTX 2070 SUPER
    CUDA version: 12.8

[7/7] Disk Space
----------------------------------------------------------------------
  GDrive (G:)     909.8/7452.0 GB (6542.2 GB free)
  GDrive (F:)     1402.2/3726.0 GB (2323.8 GB free)
  Local data      11557.6/14901.9 GB (3344.3 GB free)
  Data stack      1402.2/3726.0 GB (2323.8 GB free)
  WSL ext4        68.6/1006.9 GB (887.1 GB free)

GLOBAL SETUP COMPLETE
  Torch device: cuda
  Cities: 21, CITY=Avdiivka
  Functions: load_aoi(), load_aoi_gdf(), load_aoi_

# CELL 2B: DATASET PROFILES


In [3]:
# @title CELL 2B: DATASET PROFILES
import json as _json
from pathlib import Path
print("=" * 70)
print("CELL 2B: DATASET PROFILES")
print("=" * 70)
DATASET_ROOT_V3 = STACK_DIR / 'dataset' / 'v3'
PROFILE_DIR = DATASET_ROOT_V3 / 'dataset_profiles'
RAW_PROFILES = []
if PROFILE_DIR.exists():
    for pf in sorted(PROFILE_DIR.glob('*.json')):
        if pf.name.endswith('_features.json'):
            continue
        with open(pf) as f:
            RAW_PROFILES.append(_json.load(f))
if RAW_PROFILES:
    PROFILES_DF = pd.DataFrame([
        {
            'parquet': p['parquet_name'],
            'rows': p['n_rows'],
            'cols': p['n_cols'],
            'features': p['n_features'],
            'cities': p['n_cities'],
            'NaN%': f"{p['nan_rate_overall']*100:.1f}",
            'timestamp': p['timestamp'][:16],
        }
        for p in RAW_PROFILES
    ])
else:
    PROFILES_DF = pd.DataFrame()
if len(PROFILES_DF) == 0:
    print("  No dataset profiles available.")
    print("  (profiles are created when log_dataset_profile() is called in NB05b)")
else:
    print(f"\n  {len(PROFILES_DF)} parquets profiled:\n")
    print(PROFILES_DF.to_string(index=False))
    for prof in RAW_PROFILES:
        pq = prof.get('parquet_name', '?')
        print(f"\n  --- {pq} ---")
        print(f"    rows={prof.get('n_rows')}, features={prof.get('n_features')}, "
              f"cities={prof.get('n_cities')}, NaN={prof.get('nan_rate_overall', 0)*100:.1f}%")
        if 'points_per_city' in prof:
            for city, n in sorted(prof['points_per_city'].items(), key=lambda x: -x[1]):
                dr = prof.get('damage_rate_per_city', {}).get(city)
                dr_s = f" ({dr:.1%} damaged)" if dr is not None else ""
                nan_c = prof.get('nan_rate_per_city', {}).get(city)
                nan_s = f" NaN={nan_c*100:.1f}%" if nan_c is not None else ""
                print(f"      {city:25s}: {n:>7,} buildings{dr_s}{nan_s}")
# ---- load per-feature stats ----
_feat_rows = []
if PROFILE_DIR.exists():
    for pf in sorted(PROFILE_DIR.glob('*_features.json')):
        with open(pf) as f:
            rows = _json.load(f)
        pq_name = pf.name.replace('_features.json', '')
        for r in rows:
            r['parquet_name'] = pq_name
            _feat_rows.append(r)
FEATURE_STATS_DF = pd.DataFrame(_feat_rows) if _feat_rows else pd.DataFrame()
if len(FEATURE_STATS_DF) > 0:
    print(f"\n  FEATURE_STATS_DF: {len(FEATURE_STATS_DF)} feature records across "
          f"{FEATURE_STATS_DF['parquet_name'].nunique()} parquets")
else:
    print("\n  No feature stats available.")
    print("  (re-run NB05b to generate *_features.json profiles)")

CELL 2B: DATASET PROFILES

  108 parquets profiled:

                           parquet   rows  cols  features  cities  NaN%        timestamp
           bda_block_accum_card_t0  39997    18        15       4  33.3 2026-04-26T19:31
           bda_block_accum_card_t1  19020    13        10      11  30.0 2026-04-26T22:48
           bda_block_accum_card_t2   4226    88        85       6   3.5 2026-04-26T22:49
            bda_block_accum_coh_t0  39997     7         4       4   0.0 2026-04-26T19:31
            bda_block_accum_coh_t1  18429     7         4       9   0.0 2026-04-26T22:48
            bda_block_accum_coh_t2   1650    21        18       4   0.0 2026-04-26T22:48
             bda_block_accum_ms_t0  39997   171       168       4  39.9 2026-04-26T19:31
             bda_block_accum_ms_t1  18513   129       126      10  25.4 2026-04-26T22:50
             bda_block_accum_ms_t2   3533   402       399       5  59.1 2026-04-26T22:51
                bda_block_stats_t0  39997   109       106

# CELL 3: LOAD DATASET (from NB05 parquet via stack_loader)

Loads `bda_product_prepost` (period-aggregated features) + `bda_buildings` (metadata/labels).
Other datasets (COH drop, rolling, block, single-scene) are loaded in their own cells
(17, 18) to avoid cross-dataset NaN contamination. Each dataset is analyzed independently.
NB13 aggregates results across all datasets for model selection.


In [4]:
import gc
# @title CELL 3: LOAD V3 MANIFEST + POINTS (lazy parquet loading)
import importlib, gc
import pandas as pd
import numpy as np
import json as _json
from pathlib import Path

# Central column-role filter: single source of truth for id / label / metadata
# classification. Replaces the hand-maintained _NON_FEATURE set that previously
# lived here -- every item in that set is now covered (verified 47/47) by
# metadata_filter.is_non_feature(), plus the pattern rules catch was_observed_*,
# scenes_observed_*, qa__cloud_freq__*, visibility__*__freq__*, obs_count__*,
# block __count_* which the old set missed.
import metadata_filter
importlib.reload(metadata_filter)
from metadata_filter import is_non_feature

df_balanced = None

print("=" * 70)
print("CELL 3: LOAD V3 MANIFEST + POINTS ONLY")
print("  Parquets loaded on-demand per analysis cell, then freed.")
print("=" * 70)

# ---- V3 paths ----
DATASET_ROOT_V3 = STACK_DIR / 'dataset' / 'v3'
V3_DIR = DATASET_ROOT_V3
MANIFEST_PATH = V3_DIR / 'parquet_manifest.json'

if not MANIFEST_PATH.exists():
    raise FileNotFoundError(f"V3 manifest not found: {MANIFEST_PATH}\nRun NB05bV3 v6 first.")

with open(MANIFEST_PATH) as f:
    MANIFEST = _json.load(f)

print(f"  Manifest: {len(MANIFEST['parquets'])} parquets, version={MANIFEST['version']}")
print(f"  Created: {MANIFEST['created'][:16]} by {MANIFEST['created_by']}")

_tiers = TIER_SELECTION if TIER_SELECTION != "ALL" else [0, 1, 2, 3, 4, 5]

# ---- helper: load one parquet (concat tiers) ----
def load_v3_parquet(name):
    dfs = []
    for tier in _tiers:
        p = V3_DIR / f"bda_{name}_t{tier}.parquet"
        if p.exists():
            dfs.append(pd.read_parquet(p))
    if not dfs:
        return None
    return pd.concat(dfs, ignore_index=True)

# ---- points: always in memory (small, needed everywhere) ----
df_points = load_v3_parquet('points')
if df_points is None:
    raise FileNotFoundError("bda_points not found in v3/")
print(f"  Points: {len(df_points)} rows, {df_points['city'].nunique()} cities, "
      f"{df_points.memory_usage(deep=True).sum()/1e6:.1f} MB")

# ---- list available parquets (no loading) ----
V3_PARQUET_INFO = {}
print(f"\n  Available parquets (NOT loaded yet):")
for pq_name, pq_info in sorted(MANIFEST['parquets'].items()):
    if PARQUET_SELECTION and pq_name not in PARQUET_SELECTION:
        continue
    # check at least one tier exists on disk
    exists = any((V3_DIR / f"bda_{pq_name}_t{t}.parquet").exists() for t in _tiers)
    if exists:
        V3_PARQUET_INFO[pq_name] = pq_info
        fmt = pq_info.get('format', '?')
        n_feat = pq_info.get('n_features', 0)
        print(f"    {pq_name:<35s} [{pq_info['id']:>3s}] {fmt:>5s}  {n_feat:>4d} features")
    else:
        print(f"    {pq_name:<35s} [---] NOT ON DISK")

print(f"\n  {len(V3_PARQUET_INFO)} parquets available for on-demand loading")

# NON_FEATURE column set removed -- metadata_filter.is_non_feature() is now
# the single source of truth (verified to cover all 47 previously enumerated
# items plus pattern-matched leakage columns).

join_cols = ['point_id', 'city']

# ---- core helper: load parquet + merge buildings + filter ----
def get_analysis_df(pq_name):
    """Load one V3 parquet, merge points, filter cities.
    Returns (df, feat_cols, is_wide). Caller must gc.collect() after use."""
    if pq_name not in V3_PARQUET_INFO:
        print(f"  WARNING: {pq_name} not in manifest")
        return None, [], False
    df_pq = load_v3_parquet(pq_name)
    if df_pq is None:
        return None, [], False
    is_w = 'date' not in df_pq.columns
    if is_w:
        bex = [c for c in df_points.columns if c not in df_pq.columns]
        merged = df_pq.merge(df_points[join_cols + bex], on=join_cols, how='left')
    else:
        bex = [c for c in df_points.columns if c not in df_pq.columns and c not in ('date', 'timestep', 'period_label')]
        merged = df_pq.merge(df_points[join_cols + bex], on=join_cols, how='left')
    del df_pq
    if FILTER_UNOSAT_ONLY and TARGET_COL in merged.columns:
        merged = merged[merged[TARGET_COL].isin([0, 1])].reset_index(drop=True)
    if CITY_SELECTION is not None:
        merged = merged[merged['city'].isin(CITY_SELECTION)].reset_index(drop=True)
    feat = [c for c in merged.columns
            if not is_non_feature(c)
            and merged[c].dtype.kind in ('f', 'i', 'u')]
    mb = merged.memory_usage(deep=True).sum() / 1e6
    print(f"  Loaded {pq_name}: {len(merged)} rows, {len(feat)} features, {mb:.1f} MB")
    return merged, feat, is_w

# ---- load DEFAULT parquet for cells 4-15 (backward compat) ----
DEFAULT_PQ = 'composite_prepost_bands'
if DEFAULT_PQ not in V3_PARQUET_INFO:
    DEFAULT_PQ = 'block_stats' if 'block_stats' in V3_PARQUET_INFO else next(iter(V3_PARQUET_INFO))
    print(f"  WARNING: composite_prepost_bands not found, using {DEFAULT_PQ}")

df, FEATURE_COLS, _is_wide = get_analysis_df(DEFAULT_PQ)

from stack_catalog import parse_feature_name, BDACatalog

# catalog for cells that query feature groups by sensor/measurement
cat = None
try:
    cat = BDACatalog(CATALOG_DB, readonly=True)
    print(f"  Catalog: {CATALOG_DB} (readonly)")
except Exception as e:
    print(f"  Catalog unavailable ({e}), cat.* calls will use fallbacks")
FEATURE_GROUPS = {}
for col in FEATURE_COLS:
    info = parse_feature_name(col)
    gname = info.get('group_name', 'other')
    if gname != 'meta':
        FEATURE_GROUPS.setdefault(gname, []).append(col)
FEATURE_GROUPS = {k: sorted(v) for k, v in sorted(FEATURE_GROUPS.items()) if v}
META_COLS = [c for c in df.columns if is_non_feature(c)]

print(f"\n  Default parquet: {DEFAULT_PQ}")
print(f"  df: {len(df)} rows, {len(df.columns)} cols, {df.memory_usage(deep=True).sum()/1e6:.1f} MB")
print(f"  FEATURE_COLS: {len(FEATURE_COLS)}")
print(f"  FEATURE_GROUPS: {', '.join(f'{k}({len(v)})' for k, v in sorted(FEATURE_GROUPS.items()))}")

# ---- city filtering (on default df) ----
if REQUIRE_DAMAGE and TARGET_COL in df.columns:
    cities_with_damage = df.groupby('city')[TARGET_COL].apply(lambda s: (s == 1).sum() > 0)
    keep_cities = cities_with_damage[cities_with_damage].index.tolist()
    df = df[df['city'].isin(keep_cities)].reset_index(drop=True)

CITIES_TO_PROCESS = sorted(df['city'].unique())

# city_meta for cells that need battle dates (KDE SAR background, landuse transition)
city_meta = {}
for c in CITIES_TO_PROCESS:
    brow = df_points[df_points['city'] == c].iloc[0]
    city_meta[c] = {
        'battle_start': str(brow.get('battle_start', ''))[:10] if 'battle_start' in brow.index else '',
        'battle_stop': str(brow.get('battle_stop', '') or '')[:10] if 'battle_stop' in brow.index else '',
        'tier': int(brow.get('tier', 99)) if 'tier' in brow.index else 99,
    }
print(f"  Cities to process: {len(CITIES_TO_PROCESS)}")
for c in CITIES_TO_PROCESS:
    n = len(df[df['city'] == c])
    nd = (df[df['city'] == c][TARGET_COL] == 1).sum() if TARGET_COL in df.columns else 0
    print(f"    {c:<22s} {n:>7d} buildings, {nd:>5d} damaged")

# ---- optional balanced df ----
if BALANCE_CLASSES and TARGET_COL in df.columns:
    balanced_parts = []
    for city in CITIES_TO_PROCESS:
        cdf = df[df['city'] == city]
        n_dam = (cdf[TARGET_COL] == 1).sum()
        if n_dam < 5:
            continue
        dam = cdf[cdf[TARGET_COL] == 1]
        ctrl = cdf[cdf[TARGET_COL] == 0].sample(n=min(n_dam, len(cdf[cdf[TARGET_COL] == 0])), random_state=42)
        balanced_parts.append(pd.concat([dam, ctrl]))
    df_balanced = pd.concat(balanced_parts, ignore_index=True) if balanced_parts else None

# backward compat
PP_CAT = f'bda_{DEFAULT_PQ}_t{_tiers[0]}'
BL_CAT = f'bda_points_t{_tiers[0]}'

import matplotlib.pyplot as plt
OUT_DIR = RESULTS_ROOT / 'nb06v3'
OUT_DIR.mkdir(parents=True, exist_ok=True)

def save_fig(fig, name, subdir=''):
    d = OUT_DIR / subdir if subdir else OUT_DIR
    d.mkdir(parents=True, exist_ok=True)
    fig.savefig(d / f'{name}.png', dpi=150, bbox_inches='tight')
    plt.close(fig)

# ---- save_result helper (saves DataFrame as CSV) ----
def save_result(df_result, name, subdir=''):
    d = OUT_DIR / subdir if subdir else OUT_DIR
    d.mkdir(parents=True, exist_ok=True)
    out = d / f'{name}.csv'
    df_result.to_csv(out, index=False)
    print(f"    saved -> {out.name} ({len(df_result)} rows)")

# ---- registry stub (v1 ResultRegistry compat, logs to NB06_RESULTS) ----
class _RegistryStub:
    def log_statistics(self, cell_id='', analysis_name='', parquet_name='', **kwargs):
        NB06_RESULTS.append({
            'cell': cell_id, 'analysis': analysis_name, 'parquet': parquet_name,
            **{k: str(v)[:200] for k, v in kwargs.items()}
        })
    def log_experiment(self, **kwargs):
        NB06_RESULTS.append(kwargs)
    def save(self):
        csv_path = OUT_DIR / 'nb06_results_registry.csv'
        pd.DataFrame(NB06_RESULTS).to_csv(csv_path, index=False)
        print(f"  Registry saved: {csv_path} ({len(NB06_RESULTS)} entries)")

registry = _RegistryStub()

# ---- results accumulator for NB13 ----
NB06_RESULTS = []

def log_nb06_result(parquet, cell, metric, value, city='ALL', feature='', extra=None):
    row = {'parquet': parquet, 'cell': cell, 'metric': metric,
           'value': value, 'city': city, 'feature': feature}
    if extra:
        row.update(extra)
    NB06_RESULTS.append(row)


CELL 3: LOAD V3 MANIFEST + POINTS ONLY
  Parquets loaded on-demand per analysis cell, then freed.
  Manifest: 36 parquets, version=v3
  Created: 2026-04-26T22:51 by NB05bV3 v6
  Points: 63243 rows, 21 cities, 12.7 MB

  Available parquets (NOT loaded yet):
    block_accum_card                    [A27]  wide    15 features
    block_accum_coh                     [A26]  wide     4 features
    block_accum_ms                      [A28]  wide   168 features
    block_stats                         [A15]  wide   106 features
    card_drop                           [A19]  wide     7 features
    coh_drop                            [A14]  wide     7 features
    composite_prepost_bands             [ A9]  wide    79 features
    composite_prepost_landuse           [A10]  wide     4 features
    composite_vs_scenes_landuse         [A12]  long     3 features
    fusion_card_cohdrop                 [ F3]  long     9 features
    fusion_composite_blockstats         [ F8]  wide   185 features
    fu

# CELL 3B: NaN STRUCTURE DIAGNOSTIC

Characterizes missing data BEFORE analysis cells consume it.
Uses scene parquets (with timestep) to show WHEN and WHERE data is missing.

Key outputs:
- Per-city, per-modality observation count by timestep
- Listwise vs pairwise NaN dropout quantification (PCA vs AUC impact)
- Feature-group NaN frequency per city (leakage risk diagnostic)
- Pre/post temporal balance after NaN filtering

Scientific motivation: NB09e proved NaN handling strategy > classifier choice
for SAR signal (coh_only AUC 0.47 -> 0.74 by not dropping NaN rows).
NaN patterns that vary by city encode geographic identity = leakage vector.

In [5]:
# @title CELL 3B: NaN STRUCTURE DIAGNOSTIC
import pandas as pd
import numpy as np
from pathlib import Path

print("=" * 70)
print("CELL 3B: NaN STRUCTURE DIAGNOSTIC")
print("=" * 70)

# ---- 1. PRODUCT_PREPOST: feature-group NaN frequency per city ----
print("\n[1/4] FEATURE-GROUP NaN FREQUENCY PER CITY (product_prepost)")
print("-" * 70)

nan_summary = []
for city_name in CITIES_TO_PROCESS:
    city_df = df[df['city'] == city_name]
    if len(city_df) == 0:
        continue
    row = {'city': city_name, 'n_buildings': len(city_df)}
    for gname, cols in sorted(FEATURE_GROUPS.items()):
        avail = [c for c in cols if c in city_df.columns]
        if avail:
            nan_frac = city_df[avail].isna().values.mean()
            row[f'{gname}_nan_pct'] = round(nan_frac * 100, 1)
        else:
            row[f'{gname}_nan_pct'] = 100.0
    nan_summary.append(row)

nan_df = pd.DataFrame(nan_summary)
group_nan_cols = [c for c in nan_df.columns if c.endswith('_nan_pct')]

print(f"  {'City':<22s} {'N':>6s}", end="")
for c in group_nan_cols:
    label = c.replace('_nan_pct', '')[:10]
    print(f" {label:>10s}", end="")
print()

for _, row in nan_df.iterrows():
    print(f"  {row['city']:<22s} {int(row['n_buildings']):>6d}", end="")
    for c in group_nan_cols:
        v = row[c]
        tag = f"{v:.0f}%" if v < 100 else "ALL NaN"
        print(f" {tag:>10s}", end="")
    print()

# variance across cities = leakage risk
if len(nan_df) > 1:
    print(f"\n  NaN variance across cities (higher = more leakage risk):")
    for c in group_nan_cols:
        std = nan_df[c].std()
        mn = nan_df[c].mean()
        label = c.replace('_nan_pct', '')
        risk = "HIGH" if std > 20 else "moderate" if std > 10 else "low"
        print(f"    {label:30s} mean={mn:5.1f}% std={std:5.1f}% risk={risk}")

# ---- 2. LISTWISE vs PAIRWISE DROPOUT ----
print(f"\n[2/4] LISTWISE vs PAIRWISE NaN DROPOUT")
print("-" * 70)
print("  Listwise = drop row if ANY feature NaN (PCA, VIF)")
print("  Pairwise = drop row if THIS feature NaN (AUC, Mann-Whitney)")

for city_name in CITIES_TO_PROCESS:
    city_df = df[df['city'] == city_name]
    if len(city_df) < MIN_SAMPLES:
        continue
    feat_cols_avail = [c for c in FEATURE_COLS if c in city_df.columns and city_df[c].notna().any()]
    n_total = len(city_df)

    # listwise: all features
    n_listwise_all = city_df[feat_cols_avail].dropna().shape[0]

    # listwise: per modality group
    modality_listwise = {}
    for gname, cols in sorted(FEATURE_GROUPS.items()):
        avail = [c for c in cols if c in city_df.columns and city_df[c].notna().any()]
        if avail:
            n_kept = city_df[avail].dropna().shape[0]
            modality_listwise[gname] = n_kept

    # pairwise: median across features
    pairwise_counts = [city_df[c].notna().sum() for c in feat_cols_avail]
    n_pairwise_median = int(np.median(pairwise_counts)) if pairwise_counts else 0

    pct_listwise = n_listwise_all / n_total * 100 if n_total > 0 else 0
    pct_pairwise = n_pairwise_median / n_total * 100 if n_total > 0 else 0

    print(f"\n  {city_name}: {n_total} buildings")
    print(f"    Listwise ALL features: {n_listwise_all}/{n_total} ({pct_listwise:.0f}%) retained")
    print(f"    Pairwise median:       {n_pairwise_median}/{n_total} ({pct_pairwise:.0f}%) retained")
    if pct_listwise < 50:
        print(f"    WARNING: >50% data lost in PCA/VIF (listwise deletion)")
    for gname, n_kept in sorted(modality_listwise.items()):
        pct = n_kept / n_total * 100
        flag = " <-- bottleneck" if pct < pct_pairwise * 0.8 else ""
        print(f"    Listwise {gname:20s}: {n_kept:>6d} ({pct:5.1f}%){flag}")

# ---- 3. TEMPORAL COVERAGE BY TIMESTEP + PERIOD (from scene parquets) ----
print(f"\n[3/4] TEMPORAL COVERAGE BY TIMESTEP + PERIOD (scene parquets)")
print("-" * 70)
print("  period_label derived at runtime: date vs battle_start/battle_stop")
print("  timestep: sequential index, t=0 = first obs >= battle_start")
print("  Rolling products: period_label = trailing edge of window (spans boundaries)")

scene_parquets = {
    'CARD': 'scene_card',
    'MS': 'scene_ms',
    'COH': 'scene_coh',
    'MS_derived': 'scene_indices',
}

for label, pq_fmt in scene_parquets.items():
    try:
        scene_df = load_v3_parquet(pq_fmt)
        if scene_df is None:
            print(f"  {label}: not found, skipping")
            continue
    except FileNotFoundError:
        print(f"  {label}: no tier parquets found, skipping")
        continue
    date_col = 'date2' if 'date2' in scene_df.columns else 'date'
    print(f"\n  {label}: {len(scene_df)} rows")

    for city_name in CITIES_TO_PROCESS:
        city_scene = scene_df[scene_df['city'] == city_name]
        if city_scene.empty:
            continue
        if 'timestep' not in city_scene.columns:
            print(f"    {city_name}: no timestep column")
            continue

        n_bldg = city_scene['point_id'].nunique()
        n_dates = city_scene[date_col].nunique()
        ts_range = city_scene['timestep'].agg(['min', 'max'])

        # period breakdown (from period_label column, derived at runtime by NB05b)
        period_counts = {}
        if 'period_label' in city_scene.columns:
            for period in ['prebattle', 'crossbattle', 'postbattle']:
                period_dates = city_scene[city_scene['period_label'] == period][date_col].nunique()
                period_counts[period] = period_dates
        else:
            n_pre = (city_scene['timestep'] < 0).sum() // max(n_bldg, 1)
            n_post = (city_scene['timestep'] >= 0).sum() // max(n_bldg, 1)
            period_counts = {'prebattle': n_pre, 'crossbattle': 0, 'postbattle': n_post}

        pre = period_counts.get('prebattle', 0)
        cross = period_counts.get('crossbattle', 0)
        post = period_counts.get('postbattle', 0)

        print(f"    {city_name:<22s} dates={n_dates:>3d}  t=[{int(ts_range['min']):+d}..{int(ts_range['max']):+d}]  pre={pre} cross={cross} post={post}", end="")
        if pre == 0:
            print("  WARNING: no prebattle obs", end="")
        if post == 0 and cross == 0:
            print("  WARNING: no post/cross obs", end="")
        if pre > 0 and (cross + post) > 0:
            ratio = (cross + post) / pre
            if ratio > 5 or ratio < 0.2:
                print(f"  IMBALANCED ratio={ratio:.1f}", end="")
        print()

# rolling window boundary check
print(f"\n  ROLLING WINDOW PERIOD BOUNDARIES:")
for window_label, pq_name in [('CARD', 'rolling_card'), ('COH', 'rolling_coh')]:
    for w in [3, 7, 13]:
        rdf = load_v3_parquet(pq_name)
        if rdf is None:
            continue
        if 'period_label' not in rdf.columns or 'timestep' not in rdf.columns:
            continue
        # find observations near t=0 that span the pre/cross boundary
        near_zero = rdf[(rdf['timestep'] >= -(w-1)) & (rdf['timestep'] <= 0)]
        if near_zero.empty:
            continue
        n_boundary = len(near_zero) // max(near_zero['point_id'].nunique(), 1)
        print(f"    {window_label} roll{w}: {n_boundary} timesteps near boundary (t={-(w-1)}..0) mix pre+cross data")

# ---- 4. NaN-ENCODES-CITY DIAGNOSTIC ----
print(f"\n[4/4] NaN-ENCODES-CITY DIAGNOSTIC")
print("-" * 70)
print("  Binary NaN mask per feature -> predict city with RF")
print("  If AUC >> 0.5 (1/n_cities), NaN patterns encode geography")

if len(CITIES_TO_PROCESS) >= 2:
    feat_cols_avail = [c for c in FEATURE_COLS if c in df.columns]
    nan_mask = df[feat_cols_avail].isna().astype(int)
    # only keep features with some variance in NaN pattern
    nan_var = nan_mask.var()
    informative = nan_var[nan_var > 0.01].index.tolist()

    if len(informative) >= 3:
        from sklearn.ensemble import RandomForestClassifier
        from sklearn.model_selection import cross_val_score
        from sklearn.preprocessing import LabelEncoder

        le = LabelEncoder()
        y = le.fit_transform(df['city'])
        X = nan_mask[informative].values

        rf = RandomForestClassifier(n_estimators=50, max_depth=5, random_state=42, n_jobs=-1)
        scores = cross_val_score(rf, X, y, cv=5, scoring='accuracy')
        chance = 1.0 / len(CITIES_TO_PROCESS)
        mean_acc = scores.mean()

        print(f"  Features with NaN variance: {len(informative)}/{len(feat_cols_avail)}")
        print(f"  RF accuracy from NaN mask: {mean_acc:.3f} (chance={chance:.3f})")
        if mean_acc > chance * 2:
            print(f"  CONCLUSION: NaN patterns STRONGLY encode city identity -> leakage risk HIGH")
            print(f"  Implication: LightGBM MIA can exploit NaN as city proxy in NB09")
        elif mean_acc > chance * 1.5:
            print(f"  CONCLUSION: NaN patterns moderately encode city identity -> leakage risk MODERATE")
        else:
            print(f"  CONCLUSION: NaN patterns weakly encode city identity -> leakage risk LOW")

        # which feature groups drive NaN-based city prediction?
        rf.fit(X, y)
        importances = pd.Series(rf.feature_importances_, index=informative)
        top10 = importances.nlargest(10)
        print(f"\n  Top NaN-predictive features (city identity from missingness):")
        for feat, imp in top10.items():
            # find which group this feature belongs to
            grp = 'unknown'
            for gname, cols in FEATURE_GROUPS.items():
                if feat in cols:
                    grp = gname
                    break
            print(f"    {feat:50s} imp={imp:.3f} group={grp}")
    else:
        print(f"  Only {len(informative)} informative NaN features, skipping RF diagnostic")
else:
    print(f"  Single city ({CITIES_TO_PROCESS}), NaN-city diagnostic not applicable")

print(f"\n{'='*70}")
print("NaN DIAGNOSTIC COMPLETE")
print(f"{'='*70}")

gc.collect()
print("  Memory freed")


CELL 3B: NaN STRUCTURE DIAGNOSTIC

[1/4] FEATURE-GROUP NaN FREQUENCY PER CITY (product_prepost)
----------------------------------------------------------------------
  City                        N   ms_bands ms_indices
  Avdiivka                 1186         0%         0%
  Bucha                    1662         0%         0%
  Chernihiv                2810         0%         0%
  Chornobaivka              142         0%         0%
  Dmytrivka                3136         0%         0%
  Hostomel                 4497         0%         0%
  Irpin                    2749         0%         0%
  Kharkiv                  1883         0%         0%
  Kherson                   142         0%         0%
  Kramatorsk                180         0%         0%
  Lysychansk               7962         0%         0%
  Makariv                   448         0%         0%
  Mariupol                22615         0%         0%
  Moschun                   962         0%         0%
  Okhtyrka             

# CELL 4: PER-FEATURE ROC/AUC + MANN-WHITNEY U

Univariate discriminability ranking. AUC = P(score_damaged > score_undamaged).
Mann-Whitney U tests null hypothesis of identical distributions.
Both are non-parametric — no normality assumption required.

In [6]:
# @title CELL 4: PER-FEATURE ROC/AUC + MANN-WHITNEY U
USE_BALANCED = False  # compares groups independently, balance irrelevant
_df = df_balanced if (USE_BALANCED and df_balanced is not None and len(df_balanced) > 0) else df

import numpy as np
import pandas as pd
from scipy import stats as scipy_stats
from sklearn.metrics import roc_auc_score

print("=" * 70)
print("CELL 4: PER-FEATURE ROC/AUC + MANN-WHITNEY U")
print("=" * 70)

auc_rows = []

for CITY in CITIES_TO_PROCESS:
    city_df = _df[_df['city'] == CITY]
    if len(city_df) < MIN_SAMPLES:
        continue
    if TARGET_COL not in city_df.columns or city_df[TARGET_COL].nunique() < 2:
        continue

    print(f"\n  {CITY} ({len(city_df)} buildings)")

    feature_cols = [c for c in city_df.columns
                    if c in FEATURE_COLS
                    and city_df[c].notna().sum() > len(city_df) * MIN_FEATURE_COVERAGE]

    for col in feature_cols:
        valid = city_df[[col, TARGET_COL]].dropna()
        if len(valid) < MIN_SAMPLES or valid[TARGET_COL].nunique() < 2:
            continue

        d_vals = valid.loc[valid[TARGET_COL] == 1, col].values
        c_vals = valid.loc[valid[TARGET_COL] == 0, col].values
        if len(d_vals) < 5 or len(c_vals) < 5:
            continue

        try:
            u_stat, p_val = scipy_stats.mannwhitneyu(d_vals, c_vals, alternative='two-sided')
            auc = roc_auc_score(valid[TARGET_COL], valid[col])
            auc_best = max(auc, 1 - auc)
        except Exception:
            continue

        auc_rows.append({
            'city': CITY, 'feature': col,
            'auc_raw': auc, 'auc_best': auc_best,
            'direction': 'higher=damaged' if auc >= 0.5 else 'lower=damaged',
            'mann_whitney_u': u_stat, 'mann_whitney_p': p_val,
            'mean_damaged': np.mean(d_vals), 'mean_control': np.mean(c_vals),
            'n_valid': len(valid),
        })

    city_rows = sorted([r for r in auc_rows if r['city'] == CITY],
                       key=lambda x: x['auc_best'], reverse=True)
    print(f"    Top features by AUC:")
    for r in city_rows[:15]:
        sig = '*' if r['mann_whitney_p'] < 0.05 else ' '
        print(f"      {r['feature']:50s} AUC={r['auc_best']:.3f} p={r['mann_whitney_p']:.1e} {sig}")

if not auc_rows:
    print("\n  WARNING: No features passed filters. Check:")
    print(f"    TARGET_COL={TARGET_COL} exists: {TARGET_COL in df.columns}")
    if TARGET_COL in df.columns:
        print(f"    TARGET_COL values: {df[TARGET_COL].value_counts().to_dict()}")
    print(f"    MIN_SAMPLES={MIN_SAMPLES}, MIN_FEATURE_COVERAGE={MIN_FEATURE_COVERAGE}")
    print(f"    Cities: {CITIES_TO_PROCESS}")
    for c in CITIES_TO_PROCESS[:3]:
        cdf = df[df['city'] == c]
        print(f"    {c}: {len(cdf)} rows, target nunique={cdf[TARGET_COL].nunique() if TARGET_COL in cdf.columns else 'N/A'}")
    auc_df = pd.DataFrame()
else:
    auc_df = pd.DataFrame(auc_rows).sort_values(['city', 'auc_best'], ascending=[True, False])
    save_result(auc_df, 'feature_auc', 'cell04_auc')

    # registry
    _top10 = auc_df.groupby('feature')['auc_best'].mean().nlargest(10)
    registry.log_statistics(
        cell_id='cell04_auc',
        analysis_name='feature_auc_mannwhitney',
        parquet_name='bda_product_prepost',
        tier_selection=_tiers,
        cities=CITIES_TO_PROCESS,
        n_buildings=len(_df),
        n_features_tested=auc_df['feature'].nunique(),
        summary_metrics={
            'mean_best_auc': float(auc_df['auc_best'].mean()),
            'max_best_auc': float(auc_df['auc_best'].max()),
            'n_significant_p05': int((auc_df['mann_whitney_p'] < 0.05).sum()),
        },
        top_features=[{'feature': f, 'mean_auc': float(a)} for f, a in _top10.items()],
        note='Per-feature AUC + Mann-Whitney U (Cell 4)',
        tags=['auc', 'mann_whitney', 'feature_selection'],
    )
    


CELL 4: PER-FEATURE ROC/AUC + MANN-WHITNEY U

  Avdiivka (1186 buildings)
    Top features by AUC:
      s2__composite__ndwi__prebattle_baseline            AUC=0.754 p=4.2e-16 *
      s2__composite__ndwi__winter_baseline               AUC=0.754 p=4.2e-16 *
      s2__composite__ndwi__post_winter_baseline          AUC=0.750 p=1.2e-15 *
      s2__composite__ndvi__prebattle_baseline            AUC=0.746 p=3.2e-15 *
      s2__composite__ndvi__winter_baseline               AUC=0.746 p=3.2e-15 *
      s2__composite__ndvi__post_winter_baseline          AUC=0.745 p=4.6e-15 *
      s2__composite__mndwi__prebattle_baseline           AUC=0.720 p=1.7e-12 *
      s2__composite__mndwi__winter_baseline              AUC=0.720 p=1.7e-12 *
      s2__composite__ndre__prebattle_baseline            AUC=0.708 p=2.7e-11 *
      s2__composite__ndre__winter_baseline               AUC=0.708 p=2.7e-11 *
      s2__composite__b11__prebattle_baseline             AUC=0.707 p=2.9e-11 *
      s2__composite__b11__winter

# CELL 4: PER-FEATURE ROC/AUC + MANN-WHITNEY U

## What the test does

For each feature column in `bda_product_prepost.parquet`, the test asks:
**Can this single feature value separate damaged buildings (label=1) from undamaged buildings (label=0)?**

AUC = P(feature_value_damaged > feature_value_undamaged) for a random pair.
Mann-Whitney U confirms statistical significance (non-parametric, no normality assumption).

This is NOT a pre-vs-post comparison for the same building.
It compares the DISTRIBUTION of one feature across damaged vs undamaged buildings.

## Why prebattle features can rank high

A prebattle feature measured BEFORE damage occurred can still discriminate damaged
from undamaged buildings because it captures **damage susceptibility**, not damage itself:
- Dense urban buildings (high spectral heterogeneity = high std) were disproportionately
  targeted compared to rural/agricultural structures (low std).
- Building location within the city correlates with both spectral signature and damage probability.

This is a legitimate predictive feature for ML but must be distinguished from change detection.

## Three feature roles in BDA

| Role | Example column | What it captures |
|------|---------------|------------------|
| **Context** (pre-battle) | `s2__b11__prebattle_baseline_std` | WHERE damage is likely (building type, urbanity, location) |
| **Change** (delta/dNBR) | `cd__dnbr__prebattle_vs_postbattle_max` | THAT damage occurred (spectral change between pre and post) |
| **Post-state** | `s2__b11__post_winter_baseline_std` | WHAT the area looks like after (damage signal + context confounded) |

All three are valid ML features but serve different roles.
Context features typically have higher univariate AUC because the location signal is stronger
and less noisy than the change signal. In multivariate models (RF, LightGBM), the combination
of context + change outperforms either alone.

## Why CARD/COH features may rank lower or be absent

- COH features can be ALL NaN for cities with incomplete SAR processing (NaN coverage filter removes them).
- CARD baseline features typically rank #27-63 (AUC 0.64-0.68) — below MS composites but significant.
- The NaN dropout issue (NB09e finding) means SAR features lose training data,
  reducing their effective sample size and statistical power in this test.

## Interpretation guidance

- Top-ranked features reveal which modalities carry the most univariate signal PER CITY.
- Delta/change features are the scientifically purest damage indicators but rank lower
  because change signals are noisier than context signals at 10m resolution.
- Cross-city consistency of top features indicates generalizability.
- Features that rank high for ALL cities are the most robust for multi-city ML.

# CELL 5: DESCRIPTIVE + INFERENTIAL STATISTICS

Cohen's d: pooled standard deviation, measures practical effect size.
Rank-biserial: r = 1 - 2U/(n1*n2), non-parametric effect size from Mann-Whitney.
Cross-city ANOVA on Cohen's d: checks if feature effects are consistent across cities.

In [7]:
# @title CELL 5: DESCRIPTIVE + INFERENTIAL STATISTICS
USE_BALANCED = False  # compares groups independently
_df = df_balanced if (USE_BALANCED and df_balanced is not None and len(df_balanced) > 0) else df

import numpy as np
import pandas as pd
from scipy import stats as scipy_stats

print("=" * 70)
print("CELL 5: DESCRIPTIVE + INFERENTIAL STATISTICS")
print("=" * 70)

def cohens_d(group1, group2):
    n1, n2 = len(group1), len(group2)
    if n1 < 2 or n2 < 2:
        return np.nan
    pooled_std = np.sqrt(((n1 - 1) * np.var(group1, ddof=1) + (n2 - 1) * np.var(group2, ddof=1)) / (n1 + n2 - 2))
    if pooled_std == 0:
        return 0.0
    return (np.mean(group1) - np.mean(group2)) / pooled_std

def rank_biserial(u_stat, n1, n2):
    return 1 - (2 * u_stat) / (n1 * n2)

all_stats_rows = []

for CITY in CITIES_TO_PROCESS:
    city_df = _df[_df['city'] == CITY]
    if len(city_df) < MIN_SAMPLES or TARGET_COL not in city_df.columns:
        continue

    print(f"\n  {CITY}")

    feature_cols = [c for c in city_df.columns
                    if c in FEATURE_COLS
                    and city_df[c].notna().sum() > len(city_df) * MIN_FEATURE_COVERAGE]

    damaged = city_df[city_df[TARGET_COL] == 1]
    control = city_df[city_df[TARGET_COL] == 0]

    for col in feature_cols:
        d_vals = damaged[col].dropna().values
        c_vals = control[col].dropna().values
        if len(d_vals) < 5 or len(c_vals) < 5:
            continue

        try:
            u_stat, p_val = scipy_stats.mannwhitneyu(d_vals, c_vals, alternative='two-sided')
            rbc = rank_biserial(u_stat, len(d_vals), len(c_vals))
        except Exception:
            u_stat, p_val, rbc = np.nan, np.nan, np.nan

        d = cohens_d(d_vals, c_vals)

        all_stats_rows.append({
            'city': CITY, 'feature': col,
            'n_damaged': len(d_vals), 'n_control': len(c_vals),
            'mean_damaged': np.mean(d_vals), 'mean_control': np.mean(c_vals),
            'std_damaged': np.std(d_vals, ddof=1), 'std_control': np.std(c_vals, ddof=1),
            'cohens_d': d, 'mann_whitney_u': u_stat,
            'mann_whitney_p': p_val, 'rank_biserial': rbc,
            'significant_005': p_val < 0.05 if not np.isnan(p_val) else False,
            'effect_size_label': 'large' if abs(d) > 0.8 else 'medium' if abs(d) > 0.5 else 'small' if abs(d) > 0.2 else 'negligible',
        })

if not all_stats_rows:
    print("\n  WARNING: No features passed filters. No cities have both damaged and undamaged buildings.")
    stats_df = pd.DataFrame()
else:
    stats_df = pd.DataFrame(all_stats_rows)
    stats_csv = OUT_DIR / "all_cities_inferential_stats.csv"
    stats_df.to_csv(stats_csv, index=False)

print(f"\n{'='*70}")
print("TOP FEATURES BY EFFECT SIZE (|Cohen's d| > 0.3)")
print(f"{'='*70}")

for CITY in CITIES_TO_PROCESS:
    if len(stats_df) == 0 or 'city' not in stats_df.columns:
        break
    city_stats = stats_df[stats_df['city'] == CITY].copy()
    if len(city_stats) == 0:
        continue
    top = city_stats.reindex(city_stats['cohens_d'].abs().sort_values(ascending=False).index).head(15)
    print(f"\n  {CITY}:")
    for _, row in top.iterrows():
        if abs(row['cohens_d']) >= 0.3:
            sig = '*' if row['significant_005'] else ' '
            print(f"    {row['feature']:50s} d={row['cohens_d']:+.3f} p={row['mann_whitney_p']:.1e} {sig} [{row['effect_size_label']}]")

# cross-city ANOVA on Cohen's d
if len(CITIES_TO_PROCESS) > 1 and len(stats_df) > 0 and 'city' in stats_df.columns:
    print(f"\n{'='*70}")
    print("CROSS-CITY CONSISTENCY (Cohen's d sign agreement)")
    print(f"{'='*70}")
    common_features = set(stats_df[stats_df['city'] == CITIES_TO_PROCESS[0]]['feature'])
    for c in CITIES_TO_PROCESS[1:]:
        common_features &= set(stats_df[stats_df['city'] == c]['feature'])

    anova_rows = []
    for feat in sorted(common_features):
        d_values = stats_df[stats_df['feature'] == feat]['cohens_d'].values
        if len(d_values) > 1:
            same_sign = all(d > 0 for d in d_values) or all(d < 0 for d in d_values)
            anova_rows.append({
                'feature': feat, 'mean_d': np.mean(d_values), 'std_d': np.std(d_values),
                'consistent_sign': same_sign, 'n_cities': len(d_values),
            })

    anova_df = pd.DataFrame(anova_rows).sort_values('mean_d', key=abs, ascending=False)
    consistent = anova_df[anova_df['consistent_sign'] == True]
    print(f"  Features consistent across cities: {len(consistent)}/{len(anova_df)}")
    for _, row in consistent.head(15).iterrows():
        print(f"    {row['feature']:50s} mean_d={row['mean_d']:+.3f} (std={row['std_d']:.3f}) [{row['n_cities']} cities]")

if len(stats_df) > 0:
    print(f"\n  Saved: {stats_csv}")


CELL 5: DESCRIPTIVE + INFERENTIAL STATISTICS

  Avdiivka

  Bucha

  Chernihiv

  Chornobaivka

  Dmytrivka

  Hostomel

  Irpin

  Kharkiv

  Kherson

  Kramatorsk

  Lysychansk

  Makariv

  Mariupol

  Moschun

  Okhtyrka

  Rubizhne

  Sievierodonetsk

  Trostianets

  Volnovakha

TOP FEATURES BY EFFECT SIZE (|Cohen's d| > 0.3)

  Avdiivka:
    s2__composite__ndwi__prebattle_baseline            d=+0.995 p=4.2e-16 * [large]
    s2__composite__ndwi__winter_baseline               d=+0.995 p=4.2e-16 * [large]
    s2__composite__ndvi__winter_baseline               d=-0.888 p=3.2e-15 * [large]
    s2__composite__ndvi__prebattle_baseline            d=-0.888 p=3.2e-15 * [large]
    s2__composite__ndre__prebattle_baseline            d=-0.853 p=2.7e-11 * [large]
    s2__composite__ndre__winter_baseline               d=-0.853 p=2.7e-11 * [large]
    s2__composite__ndwi__post_winter_baseline          d=+0.839 p=1.2e-15 * [large]
    s2__composite__b11__prebattle_baseline             d=-0.791 p

# CELL 6: COHERENCE x LANDUSE CROSS-ANALYSIS

Cross-tabulation: coherence/CARD change per landuse class.
Tests if damage signal concentrates in urban pixels (expected for BDA).

In [8]:
# @title CELL 6: COHERENCE x LANDUSE CROSS-ANALYSIS
USE_BALANCED = False
_df = df_balanced if (USE_BALANCED and df_balanced is not None and len(df_balanced) > 0) else df

import numpy as np
import pandas as pd

print("=" * 70)
print("CELL 6: COHERENCE x LANDUSE CROSS-ANALYSIS")
print("=" * 70)

CLASS_NAMES = {1: 'snow', 2: 'water', 3: 'vegetation', 4: 'sparse_veg', 5: 'urban', 6: 'bare_soil'}

# landuse: try catalog, then catalog without parquet filter, then df scan
lu_cols_all = (cat.get_features_by_measurement('landuse', PP_CAT) if cat else [])
if not lu_cols_all:
    lu_cols_all = (cat.get_features_by_measurement('landuse') if cat else [])
if not lu_cols_all:
    lu_cols_all = [c for c in df.columns if 'landuse' in c.lower() and c not in META_COLS]
lu_cols_all = [c for c in lu_cols_all if c in df.columns]
print(f"  Landuse cols: {len(lu_cols_all)} {lu_cols_all[:5]}")

coh_cols_all = (cat.get_features_by_group('coh', PP_CAT) if cat else [])
if not coh_cols_all:
    coh_cols_all = [c for c in df.columns if c.startswith('s1__coh')]
card_cols_all = (cat.get_features_by_group('card', PP_CAT) if cat else [])
if not card_cols_all:
    card_cols_all = [c for c in df.columns if c.startswith('s1__v') and 'coh' not in c]
sar_pool = coh_cols_all + card_cols_all

for CITY in CITIES_TO_PROCESS:
    city_df = _df[_df['city'] == CITY]
    if len(city_df) < MIN_SAMPLES:
        continue

    print(f"\n  {CITY}")

    lu_cols = [c for c in lu_cols_all if c in city_df.columns
               and 'prebattle' in c.lower()
               and city_df[c].notna().sum() > 10]
    if not lu_cols:
        lu_cols = [c for c in lu_cols_all if c in city_df.columns
                   and ('mode' in c or 'landuse' in c.lower())
                   and city_df[c].notna().sum() > 10]
    if not lu_cols:
        print(f"    No landuse columns found (catalog returned {len(lu_cols_all)}), skipping")
        continue
    lu_col = lu_cols[0]
    print(f"    Landuse col: {lu_col}")

    sar_candidates = [c for c in sar_pool
                      if c in city_df.columns
                      and 'mean' in c and 'count' not in c
                      and city_df[c].notna().sum() > len(city_df) * 0.3
                      and city_df[c].dropna().std() > 1e-10]

    if not sar_candidates:
        print(f"    No SAR features found, skipping")
        continue

    for sar_col in sar_candidates[:3]:
        valid = city_df[[lu_col, sar_col, TARGET_COL]].dropna()
        if len(valid) < MIN_SAMPLES:
            continue

        print(f"\n    {sar_col} by prebattle landuse class:")
        print(f"      {'Class':>12s}  {'n':>5s}  {'mean_dmg':>9s}  {'mean_ctrl':>9s}  {'diff':>8s}")

        for cls_id in sorted(valid[lu_col].unique()):
            cls_name = CLASS_NAMES.get(int(cls_id), f'cls_{int(cls_id)}')
            subset = valid[valid[lu_col] == cls_id]
            dmg = subset[subset[TARGET_COL] == 1][sar_col]
            ctrl = subset[subset[TARGET_COL] == 0][sar_col]
            if len(dmg) >= 3 and len(ctrl) >= 3:
                diff = dmg.mean() - ctrl.mean()
                print(f"      {cls_name:>12s}  {len(subset):>5d}  {dmg.mean():>9.4f}  {ctrl.mean():>9.4f}  {diff:>+8.4f}")


CELL 6: COHERENCE x LANDUSE CROSS-ANALYSIS
  Landuse cols: 0 []

  Avdiivka
    No landuse columns found (catalog returned 0), skipping

  Bucha
    No landuse columns found (catalog returned 0), skipping

  Chernihiv
    No landuse columns found (catalog returned 0), skipping

  Chornobaivka
    No landuse columns found (catalog returned 0), skipping

  Dmytrivka
    No landuse columns found (catalog returned 0), skipping

  Hostomel
    No landuse columns found (catalog returned 0), skipping

  Irpin
    No landuse columns found (catalog returned 0), skipping

  Kharkiv
    No landuse columns found (catalog returned 0), skipping

  Kherson
    No landuse columns found (catalog returned 0), skipping

  Kramatorsk
    No landuse columns found (catalog returned 0), skipping

  Lysychansk
    No landuse columns found (catalog returned 0), skipping

  Makariv
    No landuse columns found (catalog returned 0), skipping

  Mariupol
    No landuse columns found (catalog returned 0), skipping

# CELL 7: PER-DAMAGE-CLASS ANALYSIS (Kruskal-Wallis)

Tests if ordinal UNOSAT damage grades can be distinguished.
Kruskal-Wallis H-test: non-parametric one-way ANOVA (rank-based).
Note: most cities only have binary labels — cell skips if no ordinal grades.

In [9]:
# @title CELL 7: PER-DAMAGE-CLASS ANALYSIS (Kruskal-Wallis)
USE_BALANCED = False  # compares groups independently
_df = df_balanced if (USE_BALANCED and df_balanced is not None and len(df_balanced) > 0) else df

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats as scipy_stats

print("=" * 70)
print("CELL 7: PER-DAMAGE-CLASS ANALYSIS")
print("=" * 70)

for CITY in CITIES_TO_PROCESS:
    city_df = _df[_df['city'] == CITY]
    if len(city_df) < MIN_SAMPLES:
        continue

    damage_col = None
    for candidate in ['damage', 'ems98_grade', 'damage_label', 'damage_class']:
        if candidate in city_df.columns and city_df[candidate].nunique() > 2:
            damage_col = candidate
            break

    if damage_col is None:
        print(f"\n  {CITY}: only binary labels available, skipping ordinal analysis")
        print(f"    -> Binary classification justified (no ordinal grades in data)")
        continue

    print(f"\n  {CITY} (using {damage_col})")
    classes = sorted(city_df[damage_col].dropna().unique())
    print(f"    Damage classes: {classes}")

    feature_cols = [c for c in city_df.columns
                    if c in FEATURE_COLS
                    and city_df[c].notna().sum() > len(city_df) * MIN_FEATURE_COVERAGE]

    kw_results = []
    for feat in feature_cols:
        grps = [city_df.loc[city_df[damage_col] == c, feat].dropna().values for c in classes]
        grps = [g for g in grps if len(g) >= 5]
        if len(grps) < 2:
            continue
        try:
            h_stat, p_val = scipy_stats.kruskal(*grps)
            kw_results.append({'feature': feat, 'H_statistic': h_stat,
                               'p_value': p_val, 'significant': p_val < 0.05,
                               'n_groups': len(grps)})
        except Exception:
            pass

    if kw_results:
        kw_df = pd.DataFrame(kw_results).sort_values('H_statistic', ascending=False)
        save_result(kw_df, f'{CITY}_kruskal_wallis', 'cell07_kruskal')

        sig_feats = kw_df[kw_df['significant']]['feature'].values
        print(f"    Kruskal-Wallis significant: {len(sig_feats)}/{len(kw_df)} features")

        top_feats = kw_df.head(6)['feature'].values
        if len(top_feats) > 0:
            fig, axes = plt.subplots(2, 3, figsize=(15, 10))
            axes = axes.flatten()
            for i, feat in enumerate(top_feats):
                data_by_class = [city_df.loc[city_df[damage_col] == c, feat].dropna().values for c in classes]
                axes[i].boxplot(data_by_class, labels=[str(c) for c in classes])
                axes[i].set_title(f"{feat}\n(H={kw_df.loc[kw_df['feature']==feat, 'H_statistic'].values[0]:.1f})", fontsize=9)
                axes[i].set_xlabel(damage_col)
            for j in range(len(top_feats), 6):
                axes[j].set_visible(False)
            fig.suptitle(f"{CITY} - Feature Distribution by Damage Class", fontsize=12, fontweight='bold')
            plt.tight_layout()
            save_fig(fig, f'{CITY}_damage_class_boxplots', 'cell07_kruskal')
            

        n_sig = len([r for r in kw_results if r['significant']])
        n_total = len(kw_results)
        pct_sig = n_sig / n_total * 100 if n_total > 0 else 0
        if pct_sig < 30:
            print(f"    CONCLUSION: Only {pct_sig:.0f}% features distinguish damage grades -> binary justified")
        else:
            print(f"    CONCLUSION: {pct_sig:.0f}% features distinguish grades -> ordinal MAY be feasible")


CELL 7: PER-DAMAGE-CLASS ANALYSIS

  Avdiivka (using damage)
    Damage classes: [np.int8(0), np.int8(1), np.int8(2), np.int8(3), np.int8(4), np.int8(7)]
    saved -> Avdiivka_kruskal_wallis.csv (76 rows)
    Kruskal-Wallis significant: 64/76 features
    CONCLUSION: 84% features distinguish grades -> ordinal MAY be feasible

  Bucha (using damage)
    Damage classes: [np.int8(0), np.int8(1), np.int8(2), np.int8(3), np.int8(4)]
    saved -> Bucha_kruskal_wallis.csv (77 rows)
    Kruskal-Wallis significant: 67/77 features
    CONCLUSION: 87% features distinguish grades -> ordinal MAY be feasible

  Chernihiv (using damage)
    Damage classes: [np.int8(0), np.int8(1), np.int8(2), np.int8(3), np.int8(4), np.int8(7)]
    saved -> Chernihiv_kruskal_wallis.csv (75 rows)
    Kruskal-Wallis significant: 62/75 features
    CONCLUSION: 83% features distinguish grades -> ordinal MAY be feasible

  Chornobaivka (using damage)
    Damage classes: [np.int8(0), np.int8(1), np.int8(2), np.int8(3), np.

# CELL 8: DISTRIBUTION ANALYSIS

Shapiro-Wilk normality test (sampled at n=5000 — for large samples, even minor
deviations from normality cause rejection; interpret with caution).
Skewness, kurtosis per feature. QQ plots for visual check.
Most satellite features are NON-normal — this justifies non-parametric tests
(Mann-Whitney, Kruskal-Wallis) used throughout NB06.

In [10]:
# @title CELL 8: DISTRIBUTION ANALYSIS + QQ PLOTS
USE_BALANCED = False  # characterizes full population distributions
_df = df_balanced if (USE_BALANCED and df_balanced is not None and len(df_balanced) > 0) else df

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats as scipy_stats

print("=" * 70)
print("CELL 8: DISTRIBUTION ANALYSIS")
print("=" * 70)

dist_rows = []

for CITY in CITIES_TO_PROCESS:
    city_df = _df[_df['city'] == CITY]
    if len(city_df) < MIN_SAMPLES:
        continue

    feature_cols = [c for c in city_df.columns
                    if c in FEATURE_COLS
                    and city_df[c].notna().sum() > len(city_df) * MIN_FEATURE_COVERAGE]

    for col in feature_cols:
        vals = city_df[col].dropna().values
        if len(vals) < MIN_SAMPLES:
            continue

        skew = scipy_stats.skew(vals, nan_policy='omit')
        kurt = scipy_stats.kurtosis(vals, nan_policy='omit')

        # Shapiro-Wilk (sample if n > 5000; note: rejects almost always for large n)
        sample = vals[:5000] if len(vals) > 5000 else vals
        try:
            sw_stat, sw_p = scipy_stats.shapiro(sample)
        except Exception:
            sw_stat, sw_p = np.nan, np.nan

        dist_rows.append({
            'city': CITY, 'feature': col, 'n': len(vals),
            'mean': np.mean(vals), 'std': np.std(vals, ddof=1),
            'skewness': skew, 'kurtosis': kurt,
            'shapiro_stat': sw_stat, 'shapiro_p': sw_p,
            'is_normal_005': sw_p > 0.05 if not np.isnan(sw_p) else False,
            'pct_nan': 1 - len(vals) / len(city_df),
        })

dist_df = pd.DataFrame(dist_rows)
save_result(dist_df, 'distributions', 'cell08_distributions')

high_nan = dist_df[dist_df['pct_nan'] > 0.5]
if len(high_nan) > 0:
    print(f"\n  WARNING: {len(high_nan)} feature-city combos with >50% NaN:")
    for _, row in high_nan.head(10).iterrows():
        print(f"    {row['city']}/{row['feature']}: {row['pct_nan']*100:.0f}% NaN")

print(f"\n  Non-normal features (Shapiro p<0.05): {(dist_df['is_normal_005'] == False).sum()}/{len(dist_df)}")
print(f"  NOTE: For large n, Shapiro-Wilk rejects even approximately normal data.")
print(f"        Non-parametric tests used throughout NB06 are robust to non-normality.")


# QQ plots
print(f"\n{'='*70}")
print("QQ-PLOTS (TOP FEATURES)")
print(f"{'='*70}")

for CITY in CITIES_TO_PROCESS:
    city_df = _df[_df['city'] == CITY]
    feature_cols = [c for c in city_df.columns
                    if c in FEATURE_COLS
                    and city_df[c].notna().sum() > len(city_df) * 0.5]
    if len(feature_cols) < 1:
        continue

    top_feats = sorted(feature_cols, key=lambda c: city_df[c].notna().sum(), reverse=True)[:6]
    n_feats = len(top_feats)
    ncols = min(3, n_feats)
    nrows = (n_feats + ncols - 1) // ncols

    fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 4 * nrows))
    if nrows * ncols == 1:
        axes = np.array([axes])
    axes = axes.flatten()

    for i, feat in enumerate(top_feats):
        vals = city_df[feat].dropna().values
        scipy_stats.probplot(vals, dist="norm", plot=axes[i])
        axes[i].set_title(f"{feat}\n(n={len(vals)})", fontsize=9)
        axes[i].get_lines()[0].set_markersize(2)

    for j in range(i + 1, len(axes)):
        axes[j].set_visible(False)

    fig.suptitle(f"{CITY} - QQ Plots", fontsize=12, fontweight='bold')
    plt.tight_layout()
    save_fig(fig, f'{CITY}_qq_plots', 'cell08_distributions')
    


CELL 8: DISTRIBUTION ANALYSIS
    saved -> distributions.csv (1501 rows)

  Non-normal features (Shapiro p<0.05): 1239/1501
  NOTE: For large n, Shapiro-Wilk rejects even approximately normal data.
        Non-parametric tests used throughout NB06 are robust to non-normality.

QQ-PLOTS (TOP FEATURES)


# CELL 9: MULTICOLLINEARITY (VIF + SPEARMAN)

Three scientifically distinct analyses:

**A. Redundancy detection**: prebattle_baseline vs winter_baseline are identical
for short battles (T0/T1) — rho=1.0 is tautological, not an insight.
QA features (visibility, cloud_freq, obs_count) are observation metadata,
not damage indicators — excluded from ML-focused analysis.

**B. Cross-period correlation (PRE vs POST)**: for matched band x stat pairs,
Spearman rho measures temporal stability. Low rho = change across battle
= potential damage signal. High rho = stable = less useful for BDA.
This is the scientifically meaningful test for a change-detection pipeline.

**C. Within-period VIF + Spearman (POST features only)**: multicollinearity
among post-battle features matters for ML feature selection.
Deduplicated (no prebattle/winter dupes, no QA features).
VIF > 10 = severe multicollinearity.
Spearman chosen over Pearson because features are non-normal (Cell 8).


In [11]:
# @title CELL 9: VIF + SPEARMAN CORRELATION
USE_BALANCED = False  # correlation structure of full feature space
_df = df_balanced if (USE_BALANCED and df_balanced is not None and len(df_balanced) > 0) else df

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import re

print("=" * 70)
print("CELL 9: VIF + SPEARMAN CORRELATION")
print("=" * 70)

# ---- FEATURE PERIOD CLASSIFICATION ----

# period tokens in feature names (order matters: longer matches first)
_PRE_TOKENS = ['prebattle_baseline', 'winter_baseline', 'prebattle']
_POST_TOKENS = ['post_winter_baseline', 'post_baseline', 'post_winter', 'post']
_DELTA_TOKENS = ['delta']

# QA / observation-metadata features: tautological correlations, not damage signal
_QA_PATTERNS = [
    r's2__visibility__',
    r's2__qa__',
    r's2__obs_count__',
    r's1__coh__scenes_observed',
]

# COH drop accumulator features are inherently change measurements
_DELTA_FEATURE_PATTERNS = [
    r's1__coh__running_min',
    r's1__coh__drop_count',
    r's1__coh__max_drop',
    r's1__coh__date_first_drop',
    r's1__coh__date_worst_drop',
    r's1__coh__lu_transition',
]

def classify_feature_period(fname):
    """Return ('pre', 'post', 'delta', 'qa', or 'unknown') for a feature name."""
    for pat in _QA_PATTERNS:
        if re.match(pat, fname):
            return 'qa'
    for pat in _DELTA_FEATURE_PATTERNS:
        if re.match(pat, fname):
            return 'delta'
    for tok in _DELTA_TOKENS:
        if tok in fname:
            return 'delta'
    for tok in _POST_TOKENS:
        if tok in fname:
            return 'post'
    for tok in _PRE_TOKENS:
        if tok in fname:
            return 'pre'
    return 'unknown'

def strip_period_token(fname):
    """Remove the period token from a feature name to get the base signature."""
    for tok in _POST_TOKENS + _PRE_TOKENS:
        if tok in fname:
            return fname.replace(tok, '').replace('____', '__').strip('_')
    return fname

def compute_vif(df_in, feature_cols, max_features=50):
    from numpy.linalg import lstsq
    X = df_in[feature_cols].dropna()
    if len(X) < len(feature_cols) + 10:
        return pd.DataFrame()

    if len(feature_cols) > max_features:
        feature_cols = feature_cols[:max_features]
        X = X[feature_cols]

    # standardize before VIF computation
    X = (X - X.mean()) / X.std().replace(0, 1)

    vif_data = []
    for i, col in enumerate(feature_cols):
        y = X[col].values
        others = X.drop(columns=[col]).values
        try:
            beta, _, _, _ = lstsq(others, y, rcond=None)
            y_pred = others @ beta
            ss_res = np.sum((y - y_pred)**2)
            ss_tot = np.sum((y - y.mean())**2)
            r2 = 1 - ss_res / ss_tot if ss_tot > 0 else 0
            vif = 1 / (1 - r2) if r2 < 1 else np.inf
        except Exception:
            vif = np.nan
        vif_data.append({'feature': col, 'VIF': vif})

    return pd.DataFrame(vif_data).sort_values('VIF', ascending=False)


# ================================================================
# PART A: REDUNDANCY DETECTION (prebattle vs winter, QA features)
# ================================================================
print(f"\n{'='*70}")
print("PART A: REDUNDANCY DETECTION")
print(f"{'='*70}")

for CITY in CITIES_TO_PROCESS:
    city_df = _df[_df['city'] == CITY]
    if len(city_df) < MIN_SAMPLES:
        continue

    feature_cols = [c for c in city_df.columns
                    if c in FEATURE_COLS
                    and city_df[c].notna().sum() > len(city_df) * 0.5]

    # classify all features
    period_map = {c: classify_feature_period(c) for c in feature_cols}
    pre_cols = [c for c in feature_cols if period_map[c] == 'pre']
    post_cols = [c for c in feature_cols if period_map[c] == 'post']
    delta_cols = [c for c in feature_cols if period_map[c] == 'delta']
    qa_cols = [c for c in feature_cols if period_map[c] == 'qa']

    # detect prebattle vs winter duplicates
    pre_by_base = {}
    for c in pre_cols:
        base = strip_period_token(c)
        pre_by_base.setdefault(base, []).append(c)

    n_dupes = 0
    dupe_pairs = []
    for base, cols in pre_by_base.items():
        if len(cols) == 2:
            valid = city_df[cols].dropna()
            if len(valid) > 20:
                rho = valid[cols[0]].corr(valid[cols[1]], method='spearman')
                if abs(rho) > 0.99:
                    n_dupes += 1
                    dupe_pairs.append((cols[0], cols[1], rho))

    print(f"\n  {CITY}: {len(feature_cols)} features")
    print(f"    PRE: {len(pre_cols)}  POST: {len(post_cols)}  DELTA: {len(delta_cols)}  QA: {len(qa_cols)}")
    if n_dupes > 0:
        print(f"    Prebattle-Winter duplicates (rho>0.99): {n_dupes}")
        for a, b, rho in dupe_pairs[:5]:
            print(f"      {a} <-> {b}: rho={rho:.4f}")
    if qa_cols:
        print(f"    QA features excluded from ML analysis: {qa_cols[:5]}")


# ================================================================
# PART B: CROSS-PERIOD CORRELATION (PRE vs POST)
# The scientifically important test: did the feature change?
# Low rho = change across battle = potential BDA signal
# ================================================================
print(f"\n{'='*70}")
print("PART B: CROSS-PERIOD CORRELATION (PRE vs POST)")
print(f"{'='*70}")
print("  Low rho = feature changed across battle = useful for BDA")
print("  High rho = stable feature = less discriminative")

cross_period_rows = []

for CITY in CITIES_TO_PROCESS:
    city_df = _df[_df['city'] == CITY]
    if len(city_df) < MIN_SAMPLES:
        continue

    feature_cols = [c for c in city_df.columns
                    if c in FEATURE_COLS
                    and city_df[c].notna().sum() > len(city_df) * 0.5]

    period_map = {c: classify_feature_period(c) for c in feature_cols}
    pre_cols = [c for c in feature_cols if period_map[c] == 'pre']
    post_cols = [c for c in feature_cols if period_map[c] == 'post']

    # match pre->post by base signature
    # for pre: prefer prebattle over winter (avoid double-counting)
    pre_by_base = {}
    for c in pre_cols:
        base = strip_period_token(c)
        if base not in pre_by_base or 'prebattle' in c:
            pre_by_base[base] = c
    post_bases = {strip_period_token(c): c for c in post_cols}

    matched = []
    for base in sorted(set(pre_by_base.keys()) & set(post_bases.keys())):
        matched.append((pre_by_base[base], post_bases[base], base))

    if not matched:
        print(f"\n  {CITY}: no matched pre-post pairs")
        continue

    print(f"\n  {CITY}: {len(matched)} matched pre-post pairs")
    print(f"    {'Base signature':50s} {'rho':>6s}  {'rho_dmg':>8s}  {'rho_ctrl':>8s}  {'delta_rho':>9s}")

    for pre_col, post_col, base in matched:
        valid = city_df[[pre_col, post_col, TARGET_COL]].dropna()
        if len(valid) < 20:
            continue

        rho_all = valid[pre_col].corr(valid[post_col], method='spearman')

        dmg = valid[valid[TARGET_COL] == 1]
        ctrl = valid[valid[TARGET_COL] == 0]
        rho_dmg = dmg[pre_col].corr(dmg[post_col], method='spearman') if len(dmg) > 10 else np.nan
        rho_ctrl = ctrl[pre_col].corr(ctrl[post_col], method='spearman') if len(ctrl) > 10 else np.nan
        delta_rho = rho_ctrl - rho_dmg if not (np.isnan(rho_dmg) or np.isnan(rho_ctrl)) else np.nan

        cross_period_rows.append({
            'city': CITY, 'base': base,
            'pre_col': pre_col, 'post_col': post_col,
            'rho_all': rho_all, 'rho_damaged': rho_dmg,
            'rho_control': rho_ctrl, 'delta_rho': delta_rho,
        })

    # print sorted by delta_rho (biggest difference = most discriminative)
    city_cross = sorted([r for r in cross_period_rows if r['city'] == CITY],
                        key=lambda x: abs(x.get('delta_rho', 0) or 0), reverse=True)
    for r in city_cross[:10]:
        print(f"    {r['base']:50s} {r['rho_all']:+.3f}  {r['rho_damaged']:+8.3f}  {r['rho_control']:+8.3f}  {r['delta_rho']:+9.3f}")

if cross_period_rows:
    cross_df = pd.DataFrame(cross_period_rows)
    save_result(cross_df, 'cross_period_correlation', 'cell09_vif_corr')

    # heatmap: cities x features, colored by delta_rho
    pivot = cross_df.pivot_table(index='base', columns='city', values='delta_rho')
    if len(pivot) > 3 and len(pivot.columns) > 1:
        fig, ax = plt.subplots(figsize=(max(10, len(pivot.columns) * 0.8), max(8, len(pivot) * 0.3)))
        im = ax.imshow(pivot.values, cmap='RdBu_r', vmin=-0.5, vmax=0.5, aspect='auto')
        ax.set_xticks(range(len(pivot.columns)))
        ax.set_yticks(range(len(pivot)))
        ax.set_xticklabels(pivot.columns, rotation=45, ha='right', fontsize=7)
        ax.set_yticklabels(pivot.index, fontsize=6)
        plt.colorbar(im, ax=ax, label='delta_rho (control - damaged)', shrink=0.8)
        ax.set_title('Cross-period rho difference: positive = damaged buildings changed more', fontsize=10)
        plt.tight_layout()
        save_fig(fig, 'cross_period_delta_rho_heatmap', 'cell09_vif_corr')
    else:
        print("  Not enough data for cross-period heatmap")


# ================================================================
# PART C: WITHIN-POST VIF + SPEARMAN (deduplicated, no QA)
# ML-relevant multicollinearity among post-battle features
# ================================================================
print(f"\n{'='*70}")
print("PART C: WITHIN-POST VIF + SPEARMAN (deduplicated, ML-relevant)")
print(f"{'='*70}")

for CITY in CITIES_TO_PROCESS:
    city_df = _df[_df['city'] == CITY]
    if len(city_df) < MIN_SAMPLES:
        continue

    # post features only, exclude QA
    feature_cols = [c for c in city_df.columns
                    if c in FEATURE_COLS
                    and city_df[c].notna().sum() > len(city_df) * 0.5]
    period_map = {c: classify_feature_period(c) for c in feature_cols}
    post_cols = [c for c in feature_cols if period_map[c] == 'post']
    delta_cols = [c for c in feature_cols if period_map[c] == 'delta']

    # for pre: keep only one of prebattle/winter (prefer prebattle)
    pre_cols_raw = [c for c in feature_cols if period_map[c] == 'pre']
    pre_by_base = {}
    for c in pre_cols_raw:
        base = strip_period_token(c)
        pre_by_base.setdefault(base, []).append(c)
    pre_cols_dedup = []
    for base, cols in pre_by_base.items():
        # prefer prebattle over winter
        pb = [c for c in cols if 'prebattle' in c]
        pre_cols_dedup.append(pb[0] if pb else cols[0])

    ml_cols = post_cols + delta_cols + pre_cols_dedup

    # rank by AUC if available
    if len(auc_df) > 0 and 'city' in auc_df.columns:
        city_auc = auc_df[auc_df['city'] == CITY].sort_values('auc_best', ascending=False)
        ranked = city_auc['feature'].tolist()
        ml_cols = [c for c in ranked if c in ml_cols] + [c for c in ml_cols if c not in ranked]

    ml_cols = ml_cols[:50]

    if len(ml_cols) < 3:
        continue

    # VIF
    vif_result = compute_vif(city_df, ml_cols)
    if len(vif_result) == 0:
        continue

    save_result(vif_result, f'{CITY}_vif_dedup', 'cell09_vif_corr')

    print(f"\n  {CITY}: {len(vif_result)} features (deduplicated, no QA)")
    high_vif = vif_result[vif_result['VIF'] > 10]
    print(f"    VIF > 10: {len(high_vif)}")
    for _, row in vif_result.head(10).iterrows():
        tag = classify_feature_period(row['feature'])
        print(f"      [{tag:5s}] {row['feature']:50s} VIF={row['VIF']:.1f}")

    # Spearman heatmap (top 30 by AUC)
    top_cols = ml_cols[:30]
    if len(top_cols) < 3:
        continue

    corr = city_df[top_cols].corr(method='spearman')

    fig, ax = plt.subplots(figsize=(14, 12))
    im = ax.imshow(corr.values, cmap='RdBu_r', vmin=-1, vmax=1, aspect='auto')
    ax.set_xticks(range(len(top_cols)))
    ax.set_yticks(range(len(top_cols)))
    # label with period tag
    labels = [f"[{classify_feature_period(c)[:3]}] {c}" for c in top_cols]
    ax.set_xticklabels(labels, rotation=90, fontsize=5)
    ax.set_yticklabels(labels, fontsize=5)
    plt.colorbar(im, ax=ax, label='Spearman rho', shrink=0.8)
    ax.set_title(f"{CITY} - Spearman (deduplicated, no QA, top {len(top_cols)} by AUC)", fontsize=11)
    plt.tight_layout()
    save_fig(fig, f'{CITY}_correlation_heatmap_dedup', 'cell09_vif_corr')

    high_corr = []
    for i in range(len(top_cols)):
        for j in range(i+1, len(top_cols)):
            r = corr.iloc[i, j]
            if abs(r) > 0.9:
                # skip pre-pre same-base pairs (already flagged in Part A)
                pi = classify_feature_period(top_cols[i])
                pj = classify_feature_period(top_cols[j])
                if pi == 'pre' and pj == 'pre' and strip_period_token(top_cols[i]) == strip_period_token(top_cols[j]):
                    continue
                high_corr.append((top_cols[i], top_cols[j], r))
    print(f"    Non-trivial |rho| > 0.9 pairs: {len(high_corr)}")
    for a, b, r in high_corr[:5]:
        print(f"      {a} <-> {b}: rho={r:.3f}")


CELL 9: VIF + SPEARMAN CORRELATION

PART A: REDUNDANCY DETECTION

  Avdiivka: 79 features
    PRE: 30  POST: 15  DELTA: 15  QA: 19
    Prebattle-Winter duplicates (rho>0.99): 15
      s2__composite__b02__prebattle_baseline <-> s2__composite__b02__winter_baseline: rho=1.0000
      s2__composite__b03__prebattle_baseline <-> s2__composite__b03__winter_baseline: rho=1.0000
      s2__composite__b04__prebattle_baseline <-> s2__composite__b04__winter_baseline: rho=1.0000
      s2__composite__b05__prebattle_baseline <-> s2__composite__b05__winter_baseline: rho=1.0000
      s2__composite__b07__prebattle_baseline <-> s2__composite__b07__winter_baseline: rho=1.0000
    QA features excluded from ML analysis: ['s2__qa__cloud_freq__crossbattle', 's2__visibility__clear__freq__crossbattle', 's2__visibility__cloud__freq__crossbattle', 's2__visibility__fire__freq__crossbattle', 's2__visibility__smoke__freq__crossbattle']

  Bucha: 79 features
    PRE: 30  POST: 15  DELTA: 15  QA: 19
    Prebattle-Winter

# CELL 10: PCA

Limitation: PCA requires complete data — rows with ANY NaN are dropped.
With SAR NaN structure, this can lose significant data. Results should be
interpreted with this caveat. Consider per-modality PCA if data loss > 50%.

In [12]:
# @title CELL 10: PCA (PER-MODALITY)
USE_BALANCED = True
_df = df_balanced if (USE_BALANCED and df_balanced is not None and len(df_balanced) > 0) else df

import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

print("=" * 70)
print("CELL 10: PCA (PER-MODALITY)")
print("=" * 70)

MODALITY_GROUPS = {
    'COH': (cat.get_features_by_group('coh', PP_CAT) if cat else []),
    'SAR_backscatter': (cat.get_features_by_group('card', PP_CAT) if cat else []),
    'MS_bands': (cat.get_features_by_group('ms_bands', PP_CAT) if cat else []),
    'MS_indices': (cat.get_features_by_group('ms_indices', PP_CAT) if cat else []),
    'ALL_features': FEATURE_COLS,
}
MODALITY_GROUPS = {k: [c for c in v if c in df.columns] for k, v in MODALITY_GROUPS.items()}
MODALITY_GROUPS = {k: v for k, v in MODALITY_GROUPS.items() if len(v) >= 3}

for mod_name, mod_cols in MODALITY_GROUPS.items():
    print(f"\n{'='*50}")
    print(f"  MODALITY: {mod_name} ({len(mod_cols)} features)")
    print(f"{'='*50}")

    for CITY in CITIES_TO_PROCESS:
        city_df = _df[_df['city'] == CITY]
        if len(city_df) < MIN_SAMPLES:
            continue

        avail_cols = [c for c in mod_cols if city_df[c].notna().sum() > len(city_df) * 0.5]
        if len(avail_cols) < 3:
            continue

        X = city_df[avail_cols].dropna()
        if len(X) < 20:
            continue

        pct_retained = len(X) / len(city_df) * 100

        scaler = StandardScaler()
        X_scaled = scaler.fit_transform(X)

        n_comp = min(20, len(avail_cols), len(X) - 1)
        pca = PCA(n_components=n_comp)
        pca.fit(X_scaled)

        cum_var = np.cumsum(pca.explained_variance_ratio_)
        n_95 = int(np.searchsorted(cum_var, 0.95)) + 1

        print(f"\n    {CITY}: {len(avail_cols)} features, {len(X)}/{len(city_df)} rows ({pct_retained:.0f}%)")
        print(f"      {n_95} PCs for 95% variance")
        top3 = pca.explained_variance_ratio_[:3] * 100
        print(f"      PC1: {top3[0]:.1f}%  PC2: {top3[1]:.1f}%  PC3: {top3[2]:.1f}%")

        loadings = pd.Series(pca.components_[0], index=avail_cols)
        top_pos = loadings.nlargest(3)
        top_neg = loadings.nsmallest(3)
        print(f"      PC1 top+: {', '.join(f'{k}({v:.2f})' for k, v in top_pos.items())}") 
        print(f"      PC1 top-: {', '.join(f'{k}({v:.2f})' for k, v in top_neg.items())}") 

        if pct_retained < 50:
            print(f"      WARNING: >50% data lost to NaN dropout")


CELL 10: PCA (PER-MODALITY)

  MODALITY: ALL_features (79 features)

    Avdiivka: 79 features, 1186/1186 rows (100%)
      14 PCs for 95% variance
      PC1: 26.6%  PC2: 19.7%  PC3: 15.9%
      PC1 top+: s2__composite__mndwi__post_winter_baseline(0.21), s2__composite__b02__post_winter_baseline(0.21), s2__composite__b04__post_winter_baseline(0.21)
      PC1 top-: s2__composite__ndbi__post_winter_baseline(-0.20), s2__composite__ndbi__delta(-0.20), s2__composite__ndre__post_winter_baseline(-0.17)

    Bucha: 79 features, 1662/1662 rows (100%)
      16 PCs for 95% variance
      PC1: 32.6%  PC2: 14.5%  PC3: 10.7%
      PC1 top+: s2__composite__b04__prebattle_baseline(0.19), s2__composite__b04__winter_baseline(0.19), s2__composite__b03__prebattle_baseline(0.19)
      PC1 top-: s2__composite__b04__delta(-0.18), s2__composite__b03__delta(-0.18), s2__composite__b05__delta(-0.18)

    Chernihiv: 79 features, 2810/2810 rows (100%)
      16 PCs for 95% variance
      PC1: 22.8%  PC2: 19.1%  PC3:

# CELL 11: FEATURE SEPARABILITY (J-M + BHATTACHARYYA + CHI-SQUARE)


In [13]:
# @title CELL 11: FEATURE SEPARABILITY (J-M + BHATTACHARYYA + CHI-SQUARE)
USE_BALANCED = False
_df = df_balanced if (USE_BALANCED and df_balanced is not None and len(df_balanced) > 0) else df

import numpy as np
import pandas as pd
from scipy import stats as scipy_stats

print("=" * 70)
print("CELL 11: FEATURE SEPARABILITY (J-M + BHATTACHARYYA + CHI-SQUARE)")
print("=" * 70)
print("  Bhattacharyya/J-M: continuous features (Gaussian assumption)")
print("  Chi-square + Cramer V: categorical features (landuse mode, fire binary)")

def bhattacharyya_distance(mu1, mu2, var1, var2):
    if var1 <= 0 or var2 <= 0:
        return np.nan
    term1 = 0.25 * np.log(0.25 * (var1 / var2 + var2 / var1 + 2))
    term2 = 0.25 * ((mu1 - mu2)**2) / (var1 + var2)
    return term1 + term2

def jm_distance(bd):
    if np.isnan(bd):
        return np.nan
    return 2 * (1 - np.exp(-bd))

def cramers_v(contingency_table):
    chi2 = scipy_stats.chi2_contingency(contingency_table)[0]
    n = contingency_table.sum().sum()
    k = min(contingency_table.shape) - 1
    if k == 0 or n == 0:
        return 0.0
    return np.sqrt(chi2 / (n * k))

# identify categorical features from catalog
categorical_measurements = {'landuse'}
cat_features = set()
for m in categorical_measurements:
    cat_features.update(cat.get_features_by_measurement(m, PP_CAT) if cat else [c for c in FEATURE_COLS if m in c.lower()])
# also flag any feature with <= 10 unique values as categorical
for c in FEATURE_COLS:
    if c in _df.columns and _df[c].dropna().nunique() <= 10:
        cat_features.add(c)

sep_rows = []

for CITY in CITIES_TO_PROCESS:
    city_df = _df[_df['city'] == CITY]
    if len(city_df) < MIN_SAMPLES or TARGET_COL not in city_df.columns:
        continue

    feature_cols = [c for c in city_df.columns
                    if c in FEATURE_COLS
                    and city_df[c].notna().sum() > len(city_df) * MIN_FEATURE_COVERAGE]

    for col in feature_cols:
        d = city_df.loc[city_df[TARGET_COL] == 1, col].dropna()
        c = city_df.loc[city_df[TARGET_COL] == 0, col].dropna()
        if len(d) < 10 or len(c) < 10:
            continue

        if col in cat_features:
            # categorical: chi-square + Cramer V
            try:
                combined = pd.DataFrame({'val': pd.concat([d, c]),
                                         'dmg': [1]*len(d) + [0]*len(c)})
                ct = pd.crosstab(combined['val'], combined['dmg'])
                if ct.shape[0] < 2 or ct.shape[1] < 2:
                    continue
                chi2, p_val, dof, _ = scipy_stats.chi2_contingency(ct)
                cv = cramers_v(ct)
                sep_rows.append({
                    'city': CITY, 'feature': col,
                    'method': 'chi_square',
                    'bhattacharyya': np.nan, 'jm_distance': np.nan,
                    'chi2': chi2, 'chi2_p': p_val, 'cramers_v': cv,
                    'separability': 'excellent' if cv > 0.5 else 'good' if cv > 0.3 else 'moderate' if cv > 0.15 else 'poor',
                })
            except Exception:
                continue
        else:
            # continuous: Bhattacharyya + J-M
            bd = bhattacharyya_distance(d.mean(), c.mean(), d.var(), c.var())
            jm = jm_distance(bd)
            sep_rows.append({
                'city': CITY, 'feature': col,
                'method': 'bhattacharyya',
                'bhattacharyya': bd, 'jm_distance': jm,
                'chi2': np.nan, 'chi2_p': np.nan, 'cramers_v': np.nan,
                'separability': 'excellent' if jm > 1.9 else 'good' if jm > 1.5 else 'moderate' if jm > 1.0 else 'poor',
            })

sep_df = pd.DataFrame(sep_rows).sort_values(['city', 'separability', 'feature'], ascending=[True, True, True])
save_result(sep_df, 'separability', 'cell11_separability')

# registry
_cont = sep_df[sep_df['method'] == 'bhattacharyya']
_top_jm = _cont.groupby('feature')['jm_distance'].mean().nlargest(10) if len(_cont) > 0 else pd.Series(dtype=float)
registry.log_statistics(
    cell_id='cell11_separability',
    analysis_name='feature_separability_jm_bhattacharyya',
    parquet_name='bda_product_prepost',
    tier_selection=_tiers,
    cities=CITIES_TO_PROCESS,
    n_buildings=len(_df),
    n_features_tested=sep_df['feature'].nunique(),
    summary_metrics={
        'mean_jm': float(_cont['jm_distance'].mean()) if len(_cont) > 0 else None,
        'n_excellent': int((sep_df['separability'] == 'excellent').sum()),
        'n_good': int((sep_df['separability'] == 'good').sum()),
        'n_poor': int((sep_df['separability'] == 'poor').sum()),
    },
    top_features=[{'feature': f, 'mean_jm': float(v)} for f, v in _top_jm.items()],
    note='J-M + Bhattacharyya + Chi-square separability (Cell 11)',
    tags=['separability', 'jm_distance', 'bhattacharyya'],
)

for CITY in CITIES_TO_PROCESS:
    city_sep = sep_df[sep_df['city'] == CITY]
    if len(city_sep) == 0:
        continue

    # continuous features by J-M
    cont = city_sep[city_sep['method'] == 'bhattacharyya'].sort_values('jm_distance', ascending=False)
    if len(cont) > 0:
        print(f"\n  {CITY}: top 10 continuous by J-M distance")
        for _, row in cont.head(10).iterrows():
            print(f"    {row['feature']:50s} JM={row['jm_distance']:.3f} BD={row['bhattacharyya']:.3f} [{row['separability']}]")

    # categorical features by Cramer V
    catg = city_sep[city_sep['method'] == 'chi_square'].sort_values('cramers_v', ascending=False)
    if len(catg) > 0:
        print(f"\n  {CITY}: categorical features by Cramer V")
        for _, row in catg.iterrows():
            sig = '*' if row['chi2_p'] < 0.05 else ' '
            print(f"    {row['feature']:50s} V={row['cramers_v']:.3f} chi2={row['chi2']:.1f} p={row['chi2_p']:.1e} {sig} [{row['separability']}]")




CELL 11: FEATURE SEPARABILITY (J-M + BHATTACHARYYA + CHI-SQUARE)
  Bhattacharyya/J-M: continuous features (Gaussian assumption)
  Chi-square + Cramer V: categorical features (landuse mode, fire binary)
    saved -> separability.csv (1447 rows)

  Avdiivka: top 10 continuous by J-M distance
    s2__qa__cloud_freq__crossbattle                    JM=1.094 BD=0.791 [moderate]
    s2__composite__ndwi__winter_baseline               JM=0.214 BD=0.113 [poor]
    s2__composite__ndwi__prebattle_baseline            JM=0.214 BD=0.113 [poor]
    s2__composite__b8a__winter_baseline                JM=0.199 BD=0.105 [poor]
    s2__composite__b8a__prebattle_baseline             JM=0.199 BD=0.105 [poor]
    s2__composite__ndre__winter_baseline               JM=0.197 BD=0.104 [poor]
    s2__composite__ndre__prebattle_baseline            JM=0.197 BD=0.104 [poor]
    s2__composite__ndwi__post_winter_baseline          JM=0.192 BD=0.101 [poor]
    s2__composite__ndvi__winter_baseline               JM=0.176 B

# CELL 12: LANDUSE TRANSITION MATRIX

Pixel-level analysis (NOT per-building). Reads full rasters from data_stack.
Compares pre-battle vs post-battle landuse classification to detect
urban->bare_soil (strong damage) and urban->vegetation (damage+time).

In [14]:
# @title CELL 12: LANDUSE TRANSITION MATRIX
import numpy as np
import rasterio
from pathlib import Path
import re
from datetime import datetime

print("=" * 70)
print("CELL 12: LANDUSE TRANSITION MATRIX")
print("=" * 70)

CLASS_NAMES = {0: 'nodata', 1: 'snow', 2: 'water', 3: 'vegetation', 4: 'sparse_veg', 5: 'urban', 6: 'bare_soil'}

def find_landuse_prepost_flat(flat_dir, battle_start_str, battle_stop_str=None):
    """Find closest pre-battle and post-battle lulc__class from landuse/flat/.

    Period derivation (runtime, no folder structure):
      date < battle_start                    -> prebattle
      battle_start <= date <= battle_stop     -> crossbattle
      date > battle_stop                      -> postbattle
      battle_stop is None (ongoing)           -> no postbattle exists

    Returns (pre_path, post_path, post_is_crossbattle):
      pre_path:  closest lulc BEFORE battle_start
      post_path: closest lulc AFTER battle_stop (true post-battle)
                 OR closest lulc AFTER battle_start if ongoing (crossbattle fallback)
      post_is_crossbattle: True if post_path is from crossbattle period
    """
    if not flat_dir.is_dir():
        return None, None, False
    bs = datetime.strptime(battle_start_str, "%Y-%m-%d")
    be = None
    if battle_stop_str and str(battle_stop_str).lower() not in ('', 'none', 'ongoing', 'nat'):
        be = datetime.strptime(str(battle_stop_str)[:10], "%Y-%m-%d")

    lulc_files = sorted(flat_dir.glob("lulc__class__*.tif"))
    if not lulc_files:
        lulc_files = sorted(flat_dir.glob("s2__landuse__*.tif"))
    if not lulc_files:
        lulc_files = sorted([f for f in flat_dir.glob("*landuse*__????????.tif")])
    if not lulc_files:
        return None, None, False

    pre_files, cross_files, post_files = [], [], []
    for f in lulc_files:
        m = re.search(r'__(\d{8})\.tif$', f.name)
        if not m:
            continue
        fdate = datetime.strptime(m.group(1), "%Y%m%d")
        if fdate < bs:
            pre_files.append((fdate, f))
        elif be and fdate > be:
            post_files.append((fdate, f))
        else:
            cross_files.append((fdate, f))

    pre_path = max(pre_files, key=lambda x: x[0])[1] if pre_files else None

    # prefer true post-battle; fallback to latest crossbattle for ongoing conflicts
    post_is_cross = False
    if post_files:
        post_path = min(post_files, key=lambda x: x[0])[1]
    elif cross_files:
        post_path = max(cross_files, key=lambda x: x[0])[1]
        post_is_cross = True
    else:
        post_path = None

    return pre_path, post_path, post_is_cross

for CITY in CITIES_TO_PROCESS:
    print(f"\n  {CITY}")
    flat_dir = STACK_ROOT / CITY / "landuse" / "flat"
    if not flat_dir.exists():
        print(f"    No landuse/flat/ directory, skipping")
        continue

    bdate = (cat.get_battle_date(CITY) if cat else BATTLE_DATES.get(CITY, {}).get('battle_start', '')) or ''
    if not bdate:
        print(f"    No battle date in catalog, skipping")
        continue
    bstop = BATTLE_DATES.get(CITY, {}).get('battle_stop', None)

    pre_path, post_path, post_is_cross = find_landuse_prepost_flat(flat_dir, bdate, bstop)

    if pre_path is None or post_path is None:
        print(f"    Missing pre ({pre_path}) or post ({post_path}) landuse, skipping")
        continue

    if post_is_cross:
        print(f"    Pre:  {pre_path.name}")
        print(f"    Post: {post_path.name}  (CROSSBATTLE fallback - ongoing conflict, no true post-battle)")
    else:
        print(f"    Pre:  {pre_path.name}")
        print(f"    Post: {post_path.name}")

    try:
        with rasterio.open(pre_path) as src:
            pre = src.read(1)
        with rasterio.open(post_path) as src:
            post = src.read(1)
    except Exception as e:
        print(f"    ERROR reading landuse rasters: {e}")
        continue

    if pre.shape != post.shape:
        print(f"    Shape mismatch: pre={pre.shape} post={post.shape}, skipping")
        continue

    valid = (pre > 0) & (post > 0) & np.isfinite(pre) & np.isfinite(post)
    pre_v = pre[valid].astype(int)
    post_v = post[valid].astype(int)

    n_classes = 7
    matrix = np.zeros((n_classes, n_classes), dtype=int)
    for p, q in zip(pre_v, post_v):
        if 0 <= p < n_classes and 0 <= q < n_classes:
            matrix[p, q] += 1

    total = matrix[1:, 1:].sum()
    if total == 0:
        print(f"    No valid transition pixels")
        continue

    print(f"    Total valid pixels: {total:,}")
    print(f"    {'':>12s}", end="")
    for j in range(1, n_classes):
        print(f" {CLASS_NAMES[j]:>10s}", end="")
    print()
    for i in range(1, n_classes):
        print(f"    {CLASS_NAMES[i]:>12s}", end="")
        for j in range(1, n_classes):
            pct = matrix[i, j] / total * 100
            print(f" {pct:10.1f}%", end="")
        print()

    urban_pre = pre_v == 5
    urban_to_bare = urban_pre & (post_v == 6)
    if urban_pre.sum() > 0:
        pct_urban_destroyed = urban_to_bare.sum() / urban_pre.sum() * 100
        print(f"\n    Urban -> bare_soil: {pct_urban_destroyed:.1f}% of pre-battle urban pixels")
    veg_pre = pre_v == 3
    veg_to_bare = veg_pre & (post_v == 6)
    if veg_pre.sum() > 0:
        pct_veg_cleared = veg_to_bare.sum() / veg_pre.sum() * 100
        print(f"    Vegetation -> bare_soil: {pct_veg_cleared:.1f}% of pre-battle vegetation pixels")


CELL 12: LANDUSE TRANSITION MATRIX

  Avdiivka
    Pre:  s2__landuse__20220108.tif
    Post: s2__landuse__20240227.tif
    Total valid pixels: 1,048,560
                       snow      water vegetation sparse_veg      urban  bare_soil
            snow        0.0%        0.0%        0.0%        0.2%        1.6%        1.2%
           water        0.0%        0.0%        0.0%        0.0%        0.0%        0.0%
      vegetation        0.0%        0.0%        0.0%        0.0%        0.2%        0.0%
      sparse_veg        0.0%        0.0%        0.0%        2.7%        2.3%        0.0%
           urban        0.0%        0.0%        0.0%        4.2%       46.9%        0.1%
       bare_soil        0.0%        0.0%        0.0%        8.4%       31.5%        0.7%

    Urban -> bare_soil: 0.2% of pre-battle urban pixels
    Vegetation -> bare_soil: 0.2% of pre-battle vegetation pixels

  Bucha
    Pre:  s2__landuse__20220214.tif
    Post: s2__landuse__20221228.tif
    Total valid pixels: 58

# CELL 13: SPATIAL STATISTICS

Moran's I (global): spatial autocorrelation of damage. KNN k=8 (standard).
LISA (local Moran): identifies HH/LL/LH/HL clusters.
Semi-variogram: spatial structure of key features (gstools, spherical model).
RF residual Moran's I: tests if model captures spatial structure.

Note: coordinates are lon/lat (WGS84). For small-area cities this is
acceptable; semi-variogram converts to meters for proper distance.

In [15]:
# @title CELL 13: SPATIAL STATISTICS
USE_BALANCED = False  # Moran I needs real spatial pattern
_df = df_balanced if (USE_BALANCED and df_balanced is not None and len(df_balanced) > 0) else df

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

print("=" * 70)
print("CELL 13: SPATIAL STATISTICS")
print("=" * 70)

try:
    import libpysal
    import esda
except ImportError:
    import subprocess
    subprocess.check_call(['pip', 'install', '--break-system-packages', '-q', 'libpysal', 'esda'])
    import libpysal
    import esda

try:
    import gstools as gs
except ImportError:
    import subprocess
    subprocess.check_call(['pip', 'install', '--break-system-packages', '-q', 'gstools'])
    import gstools as gs

from libpysal.weights import KNN
from esda.moran import Moran, Moran_Local

spatial_rows = []

for CITY in CITIES_TO_PROCESS:
    city_df = _df[_df['city'] == CITY]
    if len(city_df) < 30 or TARGET_COL not in city_df.columns or city_df[TARGET_COL].nunique() < 2:
        continue

    print(f"\n  {CITY} ({len(city_df)} buildings)")

    # need spatial coordinates
    x_col = 'centroid_x' if 'centroid_x' in city_df.columns else 'lon'
    y_col = 'centroid_y' if 'centroid_y' in city_df.columns else 'lat'
    if x_col not in city_df.columns or y_col not in city_df.columns:
        print(f"    No spatial coordinates, skipping")
        continue

    valid_spatial = city_df[[x_col, y_col, TARGET_COL]].dropna()
    if len(valid_spatial) < 30:
        continue
    coords = valid_spatial[[x_col, y_col]].values

    # Global Moran's I on damage
    try:
        w = KNN.from_array(coords, k=8)
        w.transform = 'R'
        y = valid_spatial[TARGET_COL].values.astype(float)
        mi = Moran(y, w)
        print(f"    Moran's I (damage): I={mi.I:.4f}, E[I]={mi.EI:.4f}, p={mi.p_sim:.4f}, z={mi.z_sim:.2f}")
        print(f"    {'Significant spatial clustering' if mi.p_sim < 0.05 else 'No significant spatial autocorrelation'}")
        spatial_rows.append({'city': CITY, 'variable': 'damaged',
                             'morans_I': mi.I, 'expected_I': mi.EI,
                             'p_value': mi.p_sim, 'z_score': mi.z_sim,
                             'significant': mi.p_sim < 0.05})
    except Exception as e:
        print(f"    Moran's I failed: {e}")
        mi = None

    # Moran's I on key features (auto-detect deltas and change detection)
    _coh = (cat.get_features_by_group('coh', PP_CAT) if cat else [])
    _card = (cat.get_features_by_group('card', PP_CAT) if cat else [])
    key_feats = [c for c in _coh + _card
                 if c in city_df.columns
                 and 'mean' in c and 'count' not in c
                 and city_df[c].notna().sum() > 30][:5]
    for feat in key_feats:
        valid = city_df[[feat, x_col, y_col]].dropna()
        if len(valid) < 30:
            continue
        try:
            w_f = KNN.from_array(valid[[x_col, y_col]].values, k=8)
            w_f.transform = 'R'
            mi_f = Moran(valid[feat].values, w_f)
            print(f"    Moran's I ({feat[:40]}): I={mi_f.I:.4f}, p={mi_f.p_sim:.4f}")
            spatial_rows.append({'city': CITY, 'variable': feat,
                                 'morans_I': mi_f.I, 'expected_I': mi_f.EI,
                                 'p_value': mi_f.p_sim, 'z_score': mi_f.z_sim,
                                 'significant': mi_f.p_sim < 0.05})
        except Exception:
            pass

    # LISA
    try:
        lisa = Moran_Local(y, w, permutations=999)
        sig_mask = lisa.p_sim < 0.05
        q_labels = {1: 'HH', 2: 'LH', 3: 'LL', 4: 'HL'}
        lisa_clusters = {}
        for q_val, label in q_labels.items():
            lisa_clusters[label] = int(np.sum((lisa.q == q_val) & sig_mask))
        lisa_clusters['not_significant'] = int(np.sum(~sig_mask))
        print(f"\n    LISA significant: {int(sig_mask.sum())}/{len(y)}")
        for label in ['HH', 'LL', 'LH', 'HL']:
            print(f"      {label}: {lisa_clusters[label]}")

        spatial_rows.append({'city': CITY, 'variable': 'damaged_LISA',
                             'morans_I': float(np.mean(lisa.Is)), 'expected_I': np.nan,
                             'p_value': np.nan, 'z_score': np.nan, 'significant': True,
                             'lisa_HH': lisa_clusters['HH'], 'lisa_LL': lisa_clusters['LL'],
                             'lisa_LH': lisa_clusters['LH'], 'lisa_HL': lisa_clusters['HL']})

        # Moran scatterplot
        y_std = (y - y.mean()) / y.std() if y.std() > 0 else y
        lag = np.array(w.sparse.dot(y_std))
        colors = ['grey'] * len(y_std)
        for idx_pt in range(len(y_std)):
            if sig_mask[idx_pt]:
                q = lisa.q[idx_pt]
                if q == 1: colors[idx_pt] = 'red'
                elif q == 3: colors[idx_pt] = 'blue'
                elif q == 2: colors[idx_pt] = 'lightblue'
                elif q == 4: colors[idx_pt] = 'pink'
        fig, ax = plt.subplots(1, 1, figsize=(6, 6))
        ax.scatter(y_std, lag, c=colors, s=8, alpha=0.6)
        ax.axhline(0, color='black', linewidth=0.5)
        ax.axvline(0, color='black', linewidth=0.5)
        slope = np.polyfit(y_std, lag, 1)[0]
        x_line = np.linspace(y_std.min(), y_std.max(), 100)
        ax.plot(x_line, slope * x_line, 'k--', linewidth=1, label=f"slope={slope:.3f}")
        ax.set_xlabel('Standardized damage')
        ax.set_ylabel('Spatial lag')
        i_val = mi.I if mi is not None else 0
        ax.set_title(f'{CITY} - Moran Scatterplot (damage)\nI={i_val:.4f}')
        ax.legend()
        plt.tight_layout()
        save_fig(fig, f'{CITY}_moran_scatterplot', 'cell13_spatial')
    except Exception as e:
        print(f"    LISA failed: {e}")

    # RF residual spatial analysis
    try:
        from sklearn.ensemble import RandomForestClassifier

        feat_cols = [c for c in city_df.columns if c in FEATURE_COLS
                     and city_df[c].notna().sum() > len(city_df) * 0.5]
        valid_rf = city_df[[TARGET_COL] + feat_cols + [x_col, y_col]].dropna()

        if len(valid_rf) >= 50 and len(feat_cols) >= 3:
            X_rf = valid_rf[feat_cols].values
            y_rf = valid_rf[TARGET_COL].values
            rf = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1)
            rf.fit(X_rf, y_rf)
            y_prob = rf.predict_proba(X_rf)[:, 1]
            residuals = y_rf - y_prob
            w_rf = KNN.from_array(valid_rf[[x_col, y_col]].values, k=8)
            w_rf.transform = 'R'
            mi_resid = Moran(residuals, w_rf)
            print(f"\n    RF residual Moran I: {mi_resid.I:.4f} (p={mi_resid.p_sim:.4f})")
            if mi is not None:
                print(f"    Spatial structure: {mi.I:.4f} -> {mi_resid.I:.4f} (reduction)")
            spatial_rows.append({'city': CITY, 'variable': 'rf_residuals',
                                 'morans_I': mi_resid.I, 'expected_I': mi_resid.EI,
                                 'p_value': mi_resid.p_sim, 'z_score': mi_resid.z_sim,
                                 'significant': mi_resid.p_sim < 0.05})
    except Exception as e:
        print(f"    RF residual analysis failed: {e}")

    # Semi-variogram
    print(f"\n    Semi-variograms:")
    vario_feats = [TARGET_COL] + key_feats[:3]
    for feat in vario_feats:
        if feat not in city_df.columns:
            continue
        valid = city_df[[x_col, y_col, feat]].dropna()
        if len(valid) < 50:
            continue
        if len(valid) > 2000:
            valid = valid.sample(n=2000, random_state=42)
        try:
            if x_col == 'centroid_x':
                x_m = valid[x_col].values - valid[x_col].mean()
                y_m = valid[y_col].values - valid[y_col].mean()
            else:
                x_m = (valid[x_col].values - valid[x_col].mean()) * 111320 * np.cos(np.radians(valid[y_col].mean()))
                y_m = (valid[y_col].values - valid[y_col].mean()) * 111320
            values = valid[feat].values
            bin_center, gamma = gs.vario_estimate((x_m, y_m), values, bin_edges=np.linspace(0, 5000, 30))
            try:
                model = gs.Spherical(dim=2)
                model.fit_variogram(bin_center, gamma, nugget=True)
                sill = model.var + model.nugget
                nugget_ratio = model.nugget / sill if sill > 0 else np.nan
                strength = 'Strong' if nugget_ratio < 0.25 else 'Moderate' if nugget_ratio < 0.75 else 'Weak'
                print(f"      {feat[:40]:40s}: range={model.len_scale:.0f}m, nugget_ratio={nugget_ratio:.2f} [{strength}]")
            except Exception:
                print(f"      {feat[:40]:40s}: empirical variogram computed, model fit failed")
        except Exception as e:
            print(f"      {feat[:40]:40s}: failed ({e})")

spatial_df = pd.DataFrame(spatial_rows)
if len(spatial_df) > 0:
    save_result(spatial_df, 'spatial_stats', 'cell13_spatial')
    


CELL 13: SPATIAL STATISTICS

  Avdiivka (1186 buildings)
    Moran's I (damage): I=0.1276, E[I]=-0.0008, p=0.0010, z=9.33
    Significant spatial clustering

    LISA significant: 192/1186
      HH: 47
      LL: 0
      LH: 145
      HL: 0

    RF residual Moran I: 0.0176 (p=0.0870)
    Spatial structure: 0.1276 -> 0.0176 (reduction)

    Semi-variograms:
      damage_binary                           : range=922m, nugget_ratio=0.63 [Moderate]

  Bucha (1662 buildings)
    Moran's I (damage): I=0.1150, E[I]=-0.0006, p=0.0010, z=10.06
    Significant spatial clustering

    LISA significant: 123/1662
      HH: 34
      LL: 0
      LH: 43
      HL: 46

    RF residual Moran I: 0.0803 (p=0.0010)
    Spatial structure: 0.1150 -> 0.0803 (reduction)

    Semi-variograms:
      damage_binary                           : range=589m, nugget_ratio=0.95 [Weak]

  Chernihiv (2810 buildings)
    Moran's I (damage): I=0.1372, E[I]=-0.0004, p=0.0010, z=15.80
    Significant spatial clustering

    LISA

# CELL 14: KDE DAMAGE HOTSPOT + POINT PATTERN ANALYSIS

KDE: Gaussian kernel density of damage spatial intensity (Scott bandwidth).
NNA: Clark & Evans (1954) nearest neighbor ratio R. R < 1 = clustered.
Ripley's K / L-function: second-order spatial analysis.
Note: no edge correction applied to Ripley's K — L(r) slightly biased at large r.

In [16]:
# @title CELL 14: KDE DAMAGE HOTSPOT (SAR background, pink buildings)
USE_BALANCED = True
_df = df_balanced if (USE_BALANCED and df_balanced is not None and len(df_balanced) > 0) else df

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import rasterio
import re as _re
from scipy.spatial.distance import cdist

print("=" * 70)
print("CELL 14: KDE DAMAGE HOTSPOT (SAR background, pink buildings)")
print("=" * 70)

def load_post_sar_image(city_name):
    """Load first post-battle VH CARD scene as background image."""
    card_dir = STACK_ROOT / city_name / "SAR_CARD" / "flat"
    if not card_dir.exists():
        return None, None

    battle_start = None
    city_bldg = df_points[df_points['city'] == city_name]
    if 'battle_start' in city_bldg.columns and len(city_bldg) > 0:
        battle_start = str(city_bldg.iloc[0]['battle_start'])[:10]

    vh_files = sorted(card_dir.glob("s1__vh__*.tif"))
    post_files = []
    for f in vh_files:
        m = _re.search(r'__(\d{8})\.tif$', f.name)
        if m and battle_start:
            if m.group(1) > battle_start.replace('-', ''):
                post_files.append(f)

    if not post_files:
        post_files = vh_files[:1]
    if not post_files:
        return None, None

    with rasterio.open(post_files[0]) as src:
        sar_data = src.read(1).astype(np.float32)
        sar_bounds = src.bounds
    sar_data[~np.isfinite(sar_data)] = np.nan
    return sar_data, sar_bounds


for CITY in CITIES_TO_PROCESS:
    city_df = _df[_df['city'] == CITY]
    if len(city_df) < MIN_SAMPLES or TARGET_COL not in city_df.columns or city_df[TARGET_COL].nunique() < 2:
        continue

    x_col = 'centroid_x' if 'centroid_x' in city_df.columns else 'lon'
    y_col = 'centroid_y' if 'centroid_y' in city_df.columns else 'lat'
    if x_col not in city_df.columns:
        continue

    damaged_pts = city_df[city_df[TARGET_COL] == 1][[x_col, y_col]].dropna()
    all_pts = city_df[[x_col, y_col, TARGET_COL]].dropna()
    if len(damaged_pts) < 10:
        continue

    print(f"\n  {CITY} ({len(damaged_pts)} damaged / {len(all_pts)} total buildings)")

    x_all = all_pts[x_col].values
    y_all = all_pts[y_col].values
    dmg_all = all_pts[TARGET_COL].values
    x_d = damaged_pts[x_col].values
    y_d = damaged_pts[y_col].values

    # offset to local coords (UTM centroids)
    x_off, y_off = x_all.mean(), y_all.mean()
    x_m, y_m = x_d - x_off, y_d - y_off
    x_a, y_a = x_all - x_off, y_all - y_off

    # ---- KDE with SAR background + pink buildings ----
    try:
        from scipy.stats import gaussian_kde
        kde = gaussian_kde(np.vstack([x_m, y_m]), bw_method='scott')
        pad = 500
        xmin, xmax = x_a.min() - pad, x_a.max() + pad
        ymin, ymax = y_a.min() - pad, y_a.max() + pad
        xx, yy = np.meshgrid(np.linspace(xmin, xmax, 300), np.linspace(ymin, ymax, 300))
        zz = kde(np.vstack([xx.ravel(), yy.ravel()])).reshape(xx.shape)

        fig, ax = plt.subplots(1, 1, figsize=(12, 10))

        # SAR background
        sar_data, sar_bounds = load_post_sar_image(CITY)
        if sar_data is not None:
            sar_extent = [
                sar_bounds.left - x_off, sar_bounds.right - x_off,
                sar_bounds.bottom - y_off, sar_bounds.top - y_off,
            ]
            valid_sar = sar_data[np.isfinite(sar_data)]
            if len(valid_sar) > 100:
                vmin = np.percentile(valid_sar, 2)
                vmax = np.percentile(valid_sar, 98)
                ax.imshow(sar_data, cmap='gray', extent=sar_extent,
                         origin='upper', aspect='equal', alpha=0.7,
                         vmin=vmin, vmax=vmax, interpolation='bilinear')
                print(f"    SAR background: {sar_data.shape}, dB [{vmin:.1f}, {vmax:.1f}]")

        # KDE contours
        im = ax.contourf(xx, yy, zz, levels=15, cmap='YlOrRd', alpha=0.5)
        plt.colorbar(im, ax=ax, label='Damage density', shrink=0.8)

        # All buildings pink
        ax.scatter(x_a[dmg_all == 0], y_a[dmg_all == 0],
                  s=2, c='#FF69B4', alpha=0.15, label='Undamaged', zorder=2)
        # Damaged buildings bright magenta
        ax.scatter(x_a[dmg_all == 1], y_a[dmg_all == 1],
                  s=8, c='#FF1493', alpha=0.7, edgecolors='white', linewidths=0.3,
                  label='Damaged', zorder=3)

        ax.set_title(f'{CITY} - Damage Hotspot (SAR VH background)', fontsize=14)
        ax.set_xlabel('meters (E-W)')
        ax.set_ylabel('meters (N-S)')
        ax.legend(loc='upper right', fontsize=9, framealpha=0.8)
        ax.set_aspect('equal')
        ax.set_xlim(xmin, xmax)
        ax.set_ylim(ymin, ymax)
        plt.tight_layout()
        save_fig(fig, f'{CITY}_kde_sar_hotspot', 'cell14_kde')
        print(f"    KDE + SAR saved")
    except Exception as e:
        print(f"    KDE failed: {e}")

    # NNA
    try:
        coords = np.column_stack([x_m, y_m])
        if len(coords) > 3000:
            coords = coords[np.random.choice(len(coords), 3000, replace=False)]
        dists = cdist(coords, coords)
        np.fill_diagonal(dists, np.inf)
        nn_dists = dists.min(axis=1)
        observed_mean = nn_dists.mean()
        area = (x_m.max() - x_m.min()) * (y_m.max() - y_m.min())
        n = len(coords)
        density = n / area if area > 0 else 0
        if density > 0:
            expected_mean = 0.5 / np.sqrt(density)
            se = 0.26136 / np.sqrt(n * density)
            nna_ratio = observed_mean / expected_mean
            z_nna = (observed_mean - expected_mean) / se
            pattern = 'clustered' if nna_ratio < 0.95 else 'dispersed' if nna_ratio > 1.05 else 'random'
            print(f"    NNA: R={nna_ratio:.3f}, z={z_nna:.2f}, pattern={pattern}")
    except Exception as e:
        print(f"    NNA failed: {e}")

    # Ripley K
    try:
        if area > 0:
            if len(coords) > 1000:
                coords_k = coords[np.random.choice(len(coords), 1000, replace=False)]
            else:
                coords_k = coords
            n_k = len(coords_k)
            max_dist = min(2000, np.sqrt(area) / 4)
            radii = np.linspace(0, max_dist, 30)
            dists_k = cdist(coords_k, coords_k)
            density_k = n_k / area
            K_obs = np.zeros(len(radii))
            for ri, r in enumerate(radii):
                if r == 0: continue
                count = np.sum(dists_k < r) - n_k
                K_obs[ri] = count / (n_k * density_k)
            K_csr = np.pi * radii**2
            L_obs = np.sqrt(K_obs / np.pi) - radii
            fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
            ax1.plot(radii, K_obs, 'b-', label="K observed")
            ax1.plot(radii, K_csr, 'r--', label="K CSR")
            ax1.set_xlabel('Distance (m)'); ax1.set_ylabel("K(r)")
            ax1.set_title(f"{CITY} - Ripley K"); ax1.legend()
            ax2.plot(radii, L_obs, 'b-', label="L(r) - r")
            ax2.axhline(0, color='red', linestyle='--', label='CSR')
            ax2.set_xlabel('Distance (m)'); ax2.set_ylabel('L(r) - r')
            ax2.set_title(f'{CITY} - L-function'); ax2.legend()
            plt.tight_layout()
            save_fig(fig, f'{CITY}_ripley_k', 'cell14_kde')
            max_L = L_obs[1:].max() if len(L_obs) > 1 else 0
            print(f"    Ripley K: max L(r)-r = {max_L:.1f}m")
    except Exception as e:
        print(f"    Ripley K failed: {e}")


CELL 14: KDE DAMAGE HOTSPOT (SAR background, pink buildings)

  Avdiivka (93 damaged / 1186 total buildings)
    SAR background: (1024, 1024), dB [-24.3, -9.9]
    KDE + SAR saved
    NNA: R=0.541, z=-8.46, pattern=clustered
    Ripley K: max L(r)-r = 0.0m

  Bucha (194 damaged / 1662 total buildings)
    SAR background: (768, 768), dB [-26.4, -6.8]
    KDE + SAR saved
    NNA: R=0.653, z=-9.25, pattern=clustered
    Ripley K: max L(r)-r = 0.0m

  Chernihiv (358 damaged / 2810 total buildings)
    SAR background: (1280, 1280), dB [-28.4, -8.9]
    KDE + SAR saved
    NNA: R=0.529, z=-17.04, pattern=clustered
    Ripley K: max L(r)-r = 0.0m

  Chornobaivka (13 damaged / 142 total buildings)
    SAR background: (1280, 1280), dB [-26.4, -7.5]
    KDE + SAR saved
    NNA: R=0.861, z=-0.96, pattern=clustered
    Ripley K: max L(r)-r = 0.0m

  Dmytrivka (361 damaged / 3136 total buildings)
    SAR background: (1280, 2304), dB [-26.3, -6.6]
    KDE + SAR saved
    NNA: R=0.467, z=-19.39, patt

# CELL 15: CROSS-CITY AGGREGATE SUMMARY

In [17]:
# @title CELL 15: CROSS-CITY AGGREGATE SUMMARY
import pandas as pd
import json
from datetime import datetime

print("=" * 70)
print("CELL 15: CROSS-CITY AGGREGATE SUMMARY")
print("=" * 70)

# per-city feature summary
print(f"\n  {'City':<22s} {'Points':>10s} {'Damaged':>8s} {'Features':>9s} {'Non-NaN':>8s}")
print(f"  {'-'*22} {'-'*10} {'-'*8} {'-'*9} {'-'*8}")

for CITY in CITIES_TO_PROCESS:
    city_df = df[df['city'] == CITY]
    n_bldg = len(city_df)
    n_dmg = (city_df[TARGET_COL] == 1).sum() if TARGET_COL in city_df.columns else 0
    feat_cols = [c for c in city_df.columns if c in FEATURE_COLS]
    n_feat = len(feat_cols)
    n_nonnull = sum(1 for c in feat_cols if city_df[c].notna().sum() > 0)
    print(f"  {CITY:<22s} {n_bldg:>10d} {n_dmg:>8d} {n_feat:>9d} {n_nonnull:>8d}")

# summary manifest
summary = {
    'created': datetime.now().isoformat(),
    'notebook': 'NB06V3 v1',
    'source': 'V3 parquets + bda_points via bda.sqlite catalog',
    'cities_processed': CITIES_TO_PROCESS,
    'n_cities': len(CITIES_TO_PROCESS),
    'n_points_total': len(df),
    'n_features_total': len(df.columns),
    'feature_groups': {k: len(v) for k, v in FEATURE_GROUPS.items()},
}

manifest_path = OUT_DIR / "cell15_summary" / f"NB06_summary_{datetime.now().strftime('%Y%m%d_%H%M%S')}.json"
manifest_path.parent.mkdir(parents=True, exist_ok=True)
with open(manifest_path, 'w') as f:
    json.dump(summary, f, indent=2)

# cross-city AUC consistency
if len(auc_df) > 0 and 'feature' in auc_df.columns and len(CITIES_TO_PROCESS) > 1:
    print(f"\n  Cross-city top features (consistent AUC > 0.6):")
    for feat in auc_df['feature'].unique():
        feat_auc = auc_df[auc_df['feature'] == feat]
        if len(feat_auc) >= len(CITIES_TO_PROCESS):
            mean_auc = feat_auc['auc_best'].mean()
            min_auc = feat_auc['auc_best'].min()
            if mean_auc > 0.6 and min_auc > 0.55:
                print(f"    {feat:50s} mean_AUC={mean_auc:.3f} min={min_auc:.3f} [{len(feat_auc)} cities]")

print(f"\n  Summary: {manifest_path}")
print(f"  All outputs: {OUT_DIR}")
print(f"\n{'='*70}")
print("NB06 COMPLETE")
print(f"{'='*70}")


CELL 15: CROSS-CITY AGGREGATE SUMMARY

  City                       Points  Damaged  Features  Non-NaN
  ---------------------- ---------- -------- --------- --------
  Avdiivka                     1186       93        79       79
  Bucha                        1662      194        79       79
  Chernihiv                    2810      358        79       79
  Chornobaivka                  142       13        79       79
  Dmytrivka                    3136      361        79       79
  Hostomel                     4497      618        79       79
  Irpin                        2749      308        79       79
  Kharkiv                      1883      263        79       79
  Kherson                       142       13        79       79
  Kramatorsk                    180       25        79       79
  Lysychansk                   7962     1117        79       79
  Makariv                       448       61        79       79
  Mariupol                    22615     3311        79       79
 

# PER-PARQUET GEOSPATIAL ANALYSISEach cell below loads one parquet, runs AUC + separability + VIF (wide) or temporal trajectory (long), saves results, frees memory.**Wide parquets:** AUC top-15, J-M separability top-10, VIF collinearity check**Long parquets:** Per-period AUC (pre/cross/post), temporal AUC trajectory plot

## WIDE PARQUETS (one row per building)

In [18]:
# @title CELL P1: A15 -- block_stats (AUC + VIF + Separability)
# Parquet: block_stats | SAR CARD + COH block stats (Dietrich features)
import gc
import numpy as np
import pandas as pd
from scipy import stats as scipy_stats
from sklearn.metrics import roc_auc_score

print("=" * 70)
print("CELL P1: A15 -- block_stats")
print("  SAR CARD + COH block stats (Dietrich features)")
print("=" * 70)

_pq_df, _pq_feat, _pq_wide = get_analysis_df('block_stats')
if _pq_df is None or len(_pq_feat) == 0:
    print("  SKIP: parquet not found or no features")
else:
    _n_cities = _pq_df['city'].nunique()
    _n_dam = (_pq_df[TARGET_COL] == 1).sum() if TARGET_COL in _pq_df.columns else 0
    _n_total = len(_pq_df)
    print(f"  Rows: {_n_total}, Cities: {_n_cities}, Damaged: {_n_dam}, Features: {len(_pq_feat)}")

    # ---- 1. AUC per feature (top 15) ----
    print(f"\n  [1/3] PER-FEATURE AUC (top 15 across all cities)")
    _auc_rows = []
    for _city in sorted(_pq_df['city'].unique()):
        _cdf = _pq_df[_pq_df['city'] == _city]
        if TARGET_COL not in _cdf.columns or _cdf[TARGET_COL].nunique() < 2 or len(_cdf) < MIN_SAMPLES:
            continue
        for _col in _pq_feat:
            _valid = _cdf[[_col, TARGET_COL]].dropna()
            if len(_valid) < 20 or _valid[TARGET_COL].nunique() < 2:
                continue
            _dv = _valid.loc[_valid[TARGET_COL] == 1, _col].values
            _cv = _valid.loc[_valid[TARGET_COL] == 0, _col].values
            if len(_dv) < 5 or len(_cv) < 5:
                continue
            try:
                _auc = roc_auc_score(_valid[TARGET_COL], _valid[_col])
                _auc_best = max(_auc, 1 - _auc)
                _u, _p = scipy_stats.mannwhitneyu(_dv, _cv, alternative='two-sided')
                _auc_rows.append({'city': _city, 'feature': _col, 'auc_best': _auc_best,
                                  'mann_whitney_p': _p, 'direction': 'higher=damaged' if _auc >= 0.5 else 'lower=damaged'})
            except Exception:
                continue

    if _auc_rows:
        _auc_df = pd.DataFrame(_auc_rows)
        _top = _auc_df.groupby('feature')['auc_best'].mean().nlargest(15)
        for _f, _a in _top.items():
            _n_sig = (_auc_df[_auc_df['feature'] == _f]['mann_whitney_p'] < 0.05).sum()
            _n_cit = len(_auc_df[_auc_df['feature'] == _f])
            print(f"    {_f:55s} AUC={_a:.3f} ({_n_cit} cities, {_n_sig} sig)")
        save_result(_auc_df, 'block_stats_auc', 'per_parquet')
        log_nb06_result('block_stats', 'P1', 'best_auc', float(_top.iloc[0]), feature=_top.index[0])
    else:
        print("    No features passed AUC filters")

    # ---- 2. SEPARABILITY (J-M distance, top 10) ----
    print(f"\n  [2/3] FEATURE SEPARABILITY (J-M distance, top 10)")
    _sep_rows = []
    for _city in sorted(_pq_df['city'].unique()):
        _cdf = _pq_df[_pq_df['city'] == _city]
        if TARGET_COL not in _cdf.columns or _cdf[TARGET_COL].nunique() < 2:
            continue
        for _col in _pq_feat:
            _d = _cdf.loc[_cdf[TARGET_COL] == 1, _col].dropna()
            _c = _cdf.loc[_cdf[TARGET_COL] == 0, _col].dropna()
            if len(_d) < 10 or len(_c) < 10:
                continue
            _mu1, _mu2 = _d.mean(), _c.mean()
            _v1, _v2 = _d.var(), _c.var()
            if _v1 <= 0 or _v2 <= 0:
                continue
            _bd = 0.25 * np.log(0.25 * (_v1/_v2 + _v2/_v1 + 2)) + 0.25 * ((_mu1-_mu2)**2) / (_v1+_v2)
            _jm = 2 * (1 - np.exp(-_bd))
            _sep_rows.append({'city': _city, 'feature': _col, 'jm_distance': _jm, 'bhattacharyya': _bd})

    if _sep_rows:
        _sep_df = pd.DataFrame(_sep_rows)
        _top_sep = _sep_df.groupby('feature')['jm_distance'].mean().nlargest(10)
        for _f, _jm in _top_sep.items():
            _interp = "excellent" if _jm > 1.9 else "good" if _jm > 1.5 else "moderate" if _jm > 1.0 else "poor"
            print(f"    {_f:55s} JM={_jm:.3f} ({_interp})")
        save_result(_sep_df, 'block_stats_separability', 'per_parquet')
    else:
        print("    No features passed separability filters")

    # ---- 3. VIF (top correlated pairs) ----
    print(f"\n  [3/3] VIF / COLLINEARITY CHECK")
    _vif_feats = [c for c in _pq_feat if _pq_df[c].notna().mean() > 0.5]
    if len(_vif_feats) > 3:
        _sample = _pq_df[_vif_feats].dropna()
        if len(_sample) > 100:
            if len(_vif_feats) > 50:
                # too many for full VIF, use Spearman top pairs
                _corr = _sample[_vif_feats[:50]].corr(method='spearman')
                _high_corr = []
                for _ii in range(len(_corr)):
                    for _jj in range(_ii+1, len(_corr)):
                        _r = abs(_corr.iloc[_ii, _jj])
                        if _r > 0.9:
                            _high_corr.append((_corr.index[_ii], _corr.columns[_jj], _r))
                _high_corr.sort(key=lambda x: -x[2])
                print(f"    Spearman |r| > 0.9: {len(_high_corr)} pairs (of {len(_vif_feats)} features)")
                for _a, _b, _r in _high_corr[:10]:
                    print(f"      {_a:40s} <-> {_b:40s} r={_r:.3f}")
            else:
                from statsmodels.stats.outliers_influence import variance_inflation_factor
                try:
                    _X = _sample[_vif_feats].values
                    _vif_vals = [variance_inflation_factor(_X, _k) for _k in range(len(_vif_feats))]
                    _vif_df = pd.DataFrame({'feature': _vif_feats, 'VIF': _vif_vals}).sort_values('VIF', ascending=False)
                    _high_vif = _vif_df[_vif_df['VIF'] > 10]
                    print(f"    VIF > 10: {len(_high_vif)}/{len(_vif_feats)} features")
                    for _, _row in _high_vif.head(10).iterrows():
                        print(f"      {_row['feature']:55s} VIF={_row['VIF']:.1f}")
                    save_result(_vif_df, 'block_stats_vif', 'per_parquet')
                except Exception as _e:
                    print(f"    VIF failed: {_e}")
        else:
            print(f"    Too few valid rows ({len(_sample)}) for VIF")
    else:
        print(f"    Too few features ({len(_vif_feats)}) for VIF")

    print(f"\n  A15 analysis complete")

del _pq_df
gc.collect()
print("  Memory freed")


CELL P1: A15 -- block_stats
  SAR CARD + COH block stats (Dietrich features)
  Loaded block_stats: 63243 rows, 330 features, 96.6 MB
  Rows: 63243, Cities: 21, Damaged: 8247, Features: 330

  [1/3] PER-FEATURE AUC (top 15 across all cities)
    s1__vh__blk02__max                                      AUC=0.585 (6 cities, 2 sig)
    s1__vv__blk02__median                                   AUC=0.569 (6 cities, 1 sig)
    s1__vv__blk02__mean                                     AUC=0.567 (6 cities, 1 sig)
    s1__vh__blk02__median                                   AUC=0.566 (6 cities, 0 sig)
    s1__vv__blk02__max                                      AUC=0.562 (6 cities, 0 sig)
    s1__vh__blk02__mean                                     AUC=0.562 (6 cities, 0 sig)
    s1__vv__blk01__std                                      AUC=0.559 (21 cities, 12 sig)
    s1__vv__blk01__mean                                     AUC=0.558 (21 cities, 9 sig)
    s1__vv__blk01__median                           

In [19]:
# @title CELL P2: A14 -- coh_drop (AUC + VIF + Separability)
# Parquet: coh_drop | Coherence drop accumulator (7 features)
import gc
import numpy as np
import pandas as pd
from scipy import stats as scipy_stats
from sklearn.metrics import roc_auc_score

print("=" * 70)
print("CELL P2: A14 -- coh_drop")
print("  Coherence drop accumulator (7 features)")
print("=" * 70)

_pq_df, _pq_feat, _pq_wide = get_analysis_df('coh_drop')
if _pq_df is None or len(_pq_feat) == 0:
    print("  SKIP: parquet not found or no features")
else:
    _n_cities = _pq_df['city'].nunique()
    _n_dam = (_pq_df[TARGET_COL] == 1).sum() if TARGET_COL in _pq_df.columns else 0
    _n_total = len(_pq_df)
    print(f"  Rows: {_n_total}, Cities: {_n_cities}, Damaged: {_n_dam}, Features: {len(_pq_feat)}")

    # ---- 1. AUC per feature (top 15) ----
    print(f"\n  [1/3] PER-FEATURE AUC (top 15 across all cities)")
    _auc_rows = []
    for _city in sorted(_pq_df['city'].unique()):
        _cdf = _pq_df[_pq_df['city'] == _city]
        if TARGET_COL not in _cdf.columns or _cdf[TARGET_COL].nunique() < 2 or len(_cdf) < MIN_SAMPLES:
            continue
        for _col in _pq_feat:
            _valid = _cdf[[_col, TARGET_COL]].dropna()
            if len(_valid) < 20 or _valid[TARGET_COL].nunique() < 2:
                continue
            _dv = _valid.loc[_valid[TARGET_COL] == 1, _col].values
            _cv = _valid.loc[_valid[TARGET_COL] == 0, _col].values
            if len(_dv) < 5 or len(_cv) < 5:
                continue
            try:
                _auc = roc_auc_score(_valid[TARGET_COL], _valid[_col])
                _auc_best = max(_auc, 1 - _auc)
                _u, _p = scipy_stats.mannwhitneyu(_dv, _cv, alternative='two-sided')
                _auc_rows.append({'city': _city, 'feature': _col, 'auc_best': _auc_best,
                                  'mann_whitney_p': _p, 'direction': 'higher=damaged' if _auc >= 0.5 else 'lower=damaged'})
            except Exception:
                continue

    if _auc_rows:
        _auc_df = pd.DataFrame(_auc_rows)
        _top = _auc_df.groupby('feature')['auc_best'].mean().nlargest(15)
        for _f, _a in _top.items():
            _n_sig = (_auc_df[_auc_df['feature'] == _f]['mann_whitney_p'] < 0.05).sum()
            _n_cit = len(_auc_df[_auc_df['feature'] == _f])
            print(f"    {_f:55s} AUC={_a:.3f} ({_n_cit} cities, {_n_sig} sig)")
        save_result(_auc_df, 'coh_drop_auc', 'per_parquet')
        log_nb06_result('coh_drop', 'P2', 'best_auc', float(_top.iloc[0]), feature=_top.index[0])
    else:
        print("    No features passed AUC filters")

    # ---- 2. SEPARABILITY (J-M distance, top 10) ----
    print(f"\n  [2/3] FEATURE SEPARABILITY (J-M distance, top 10)")
    _sep_rows = []
    for _city in sorted(_pq_df['city'].unique()):
        _cdf = _pq_df[_pq_df['city'] == _city]
        if TARGET_COL not in _cdf.columns or _cdf[TARGET_COL].nunique() < 2:
            continue
        for _col in _pq_feat:
            _d = _cdf.loc[_cdf[TARGET_COL] == 1, _col].dropna()
            _c = _cdf.loc[_cdf[TARGET_COL] == 0, _col].dropna()
            if len(_d) < 10 or len(_c) < 10:
                continue
            _mu1, _mu2 = _d.mean(), _c.mean()
            _v1, _v2 = _d.var(), _c.var()
            if _v1 <= 0 or _v2 <= 0:
                continue
            _bd = 0.25 * np.log(0.25 * (_v1/_v2 + _v2/_v1 + 2)) + 0.25 * ((_mu1-_mu2)**2) / (_v1+_v2)
            _jm = 2 * (1 - np.exp(-_bd))
            _sep_rows.append({'city': _city, 'feature': _col, 'jm_distance': _jm, 'bhattacharyya': _bd})

    if _sep_rows:
        _sep_df = pd.DataFrame(_sep_rows)
        _top_sep = _sep_df.groupby('feature')['jm_distance'].mean().nlargest(10)
        for _f, _jm in _top_sep.items():
            _interp = "excellent" if _jm > 1.9 else "good" if _jm > 1.5 else "moderate" if _jm > 1.0 else "poor"
            print(f"    {_f:55s} JM={_jm:.3f} ({_interp})")
        save_result(_sep_df, 'coh_drop_separability', 'per_parquet')
    else:
        print("    No features passed separability filters")

    # ---- 3. VIF (top correlated pairs) ----
    print(f"\n  [3/3] VIF / COLLINEARITY CHECK")
    _vif_feats = [c for c in _pq_feat if _pq_df[c].notna().mean() > 0.5]
    if len(_vif_feats) > 3:
        _sample = _pq_df[_vif_feats].dropna()
        if len(_sample) > 100:
            if len(_vif_feats) > 50:
                # too many for full VIF, use Spearman top pairs
                _corr = _sample[_vif_feats[:50]].corr(method='spearman')
                _high_corr = []
                for _ii in range(len(_corr)):
                    for _jj in range(_ii+1, len(_corr)):
                        _r = abs(_corr.iloc[_ii, _jj])
                        if _r > 0.9:
                            _high_corr.append((_corr.index[_ii], _corr.columns[_jj], _r))
                _high_corr.sort(key=lambda x: -x[2])
                print(f"    Spearman |r| > 0.9: {len(_high_corr)} pairs (of {len(_vif_feats)} features)")
                for _a, _b, _r in _high_corr[:10]:
                    print(f"      {_a:40s} <-> {_b:40s} r={_r:.3f}")
            else:
                from statsmodels.stats.outliers_influence import variance_inflation_factor
                try:
                    _X = _sample[_vif_feats].values
                    _vif_vals = [variance_inflation_factor(_X, _k) for _k in range(len(_vif_feats))]
                    _vif_df = pd.DataFrame({'feature': _vif_feats, 'VIF': _vif_vals}).sort_values('VIF', ascending=False)
                    _high_vif = _vif_df[_vif_df['VIF'] > 10]
                    print(f"    VIF > 10: {len(_high_vif)}/{len(_vif_feats)} features")
                    for _, _row in _high_vif.head(10).iterrows():
                        print(f"      {_row['feature']:55s} VIF={_row['VIF']:.1f}")
                    save_result(_vif_df, 'coh_drop_vif', 'per_parquet')
                except Exception as _e:
                    print(f"    VIF failed: {_e}")
        else:
            print(f"    Too few valid rows ({len(_sample)}) for VIF")
    else:
        print(f"    Too few features ({len(_vif_feats)}) for VIF")

    print(f"\n  A14 analysis complete")

del _pq_df
gc.collect()
print("  Memory freed")


CELL P2: A14 -- coh_drop
  Coherence drop accumulator (7 features)
  Loaded coh_drop: 51800 rows, 7 features, 12.3 MB
  Rows: 51800, Cities: 12, Damaged: 6935, Features: 7

  [1/3] PER-FEATURE AUC (top 15 across all cities)
    s1__coh__lu_transition                                  AUC=0.523 (12 cities, 5 sig)
    s1__coh__max_drop                                       AUC=0.521 (12 cities, 4 sig)
    s1__coh__date_first_drop                                AUC=0.521 (12 cities, 4 sig)
    s1__coh__date_worst_drop                                AUC=0.515 (12 cities, 4 sig)
    s1__coh__running_min                                    AUC=0.514 (12 cities, 1 sig)
    s1__coh__drop_count                                     AUC=0.513 (12 cities, 3 sig)
    s1__coh__scenes_observed                                AUC=0.508 (12 cities, 3 sig)
    saved -> coh_drop_auc.csv (84 rows)

  [2/3] FEATURE SEPARABILITY (J-M distance, top 10)
    s1__coh__scenes_observed                                

In [20]:
# @title CELL P2b: A19 -- card_drop (AUC + VIF + Separability)
# Parquet: card_drop | CARD z-score drop accumulator (19 features: z_running_min, drop_count, max_z_drop, dates)
import gc
import numpy as np
import pandas as pd
from scipy import stats as scipy_stats
from sklearn.metrics import roc_auc_score

print("=" * 70)
print("CELL P2b: A19 -- card_drop")
print("  CARD z-score drop accumulator (19 features: z_running_min, drop_count, max_z_drop, dates)")
print("=" * 70)

_pq_df, _pq_feat, _pq_wide = get_analysis_df('card_drop')
if _pq_df is None or len(_pq_feat) == 0:
    print("  SKIP: parquet not found or no features")
else:
    _n_cities = _pq_df['city'].nunique()
    _n_dam = (_pq_df[TARGET_COL] == 1).sum() if TARGET_COL in _pq_df.columns else 0
    _n_total = len(_pq_df)
    print(f"  Rows: {_n_total}, Cities: {_n_cities}, Damaged: {_n_dam}, Features: {len(_pq_feat)}")

    # ---- 1. AUC per feature (top 15) ----
    print(f"\n  [1/3] PER-FEATURE AUC (top 15 across all cities)")
    _auc_rows = []
    for _city in sorted(_pq_df['city'].unique()):
        _cdf = _pq_df[_pq_df['city'] == _city]
        if TARGET_COL not in _cdf.columns or _cdf[TARGET_COL].nunique() < 2 or len(_cdf) < MIN_SAMPLES:
            continue
        for _col in _pq_feat:
            _valid = _cdf[[_col, TARGET_COL]].dropna()
            if len(_valid) < 20 or _valid[TARGET_COL].nunique() < 2:
                continue
            _dv = _valid.loc[_valid[TARGET_COL] == 1, _col].values
            _cv = _valid.loc[_valid[TARGET_COL] == 0, _col].values
            if len(_dv) < 5 or len(_cv) < 5:
                continue
            try:
                _auc = roc_auc_score(_valid[TARGET_COL], _valid[_col])
                _auc_best = max(_auc, 1 - _auc)
                _u, _p = scipy_stats.mannwhitneyu(_dv, _cv, alternative='two-sided')
                _auc_rows.append({'city': _city, 'feature': _col, 'auc_best': _auc_best,
                                  'mann_whitney_p': _p, 'direction': 'higher=damaged' if _auc >= 0.5 else 'lower=damaged'})
            except Exception:
                continue

    if _auc_rows:
        _auc_df = pd.DataFrame(_auc_rows)
        _top = _auc_df.groupby('feature')['auc_best'].mean().nlargest(15)
        for _f, _a in _top.items():
            _n_sig = (_auc_df[_auc_df['feature'] == _f]['mann_whitney_p'] < 0.05).sum()
            _n_cit = len(_auc_df[_auc_df['feature'] == _f])
            print(f"    {_f:55s} AUC={_a:.3f} ({_n_cit} cities, {_n_sig} sig)")
        save_result(_auc_df, 'card_drop_auc', 'per_parquet')
        log_nb06_result('card_drop', 'P2b', 'best_auc', float(_top.iloc[0]), feature=_top.index[0])
    else:
        print("    No features passed AUC filters")

    # ---- 2. SEPARABILITY (J-M distance, top 10) ----
    print(f"\n  [2/3] FEATURE SEPARABILITY (J-M distance, top 10)")
    _sep_rows = []
    for _city in sorted(_pq_df['city'].unique()):
        _cdf = _pq_df[_pq_df['city'] == _city]
        if TARGET_COL not in _cdf.columns or _cdf[TARGET_COL].nunique() < 2:
            continue
        for _col in _pq_feat:
            _d = _cdf.loc[_cdf[TARGET_COL] == 1, _col].dropna()
            _c = _cdf.loc[_cdf[TARGET_COL] == 0, _col].dropna()
            if len(_d) < 10 or len(_c) < 10:
                continue
            _mu1, _mu2 = _d.mean(), _c.mean()
            _v1, _v2 = _d.var(), _c.var()
            if _v1 <= 0 or _v2 <= 0:
                continue
            _bd = 0.25 * np.log(0.25 * (_v1/_v2 + _v2/_v1 + 2)) + 0.25 * ((_mu1-_mu2)**2) / (_v1+_v2)
            _jm = 2 * (1 - np.exp(-_bd))
            _sep_rows.append({'city': _city, 'feature': _col, 'jm_distance': _jm, 'bhattacharyya': _bd})

    if _sep_rows:
        _sep_df = pd.DataFrame(_sep_rows)
        _top_sep = _sep_df.groupby('feature')['jm_distance'].mean().nlargest(10)
        for _f, _jm in _top_sep.items():
            _interp = "excellent" if _jm > 1.9 else "good" if _jm > 1.5 else "moderate" if _jm > 1.0 else "poor"
            print(f"    {_f:55s} JM={_jm:.3f} ({_interp})")
        save_result(_sep_df, 'card_drop_separability', 'per_parquet')
    else:
        print("    No features passed separability filters")

    # ---- 3. VIF (top correlated pairs) ----
    print(f"\n  [3/3] VIF / COLLINEARITY CHECK")
    _vif_feats = [c for c in _pq_feat if _pq_df[c].notna().mean() > 0.5]
    if len(_vif_feats) > 3:
        _sample = _pq_df[_vif_feats].dropna()
        if len(_sample) > 100:
            if len(_vif_feats) > 50:
                # too many for full VIF, use Spearman top pairs
                _corr = _sample[_vif_feats[:50]].corr(method='spearman')
                _high_corr = []
                for _ii in range(len(_corr)):
                    for _jj in range(_ii+1, len(_corr)):
                        _r = abs(_corr.iloc[_ii, _jj])
                        if _r > 0.9:
                            _high_corr.append((_corr.index[_ii], _corr.columns[_jj], _r))
                _high_corr.sort(key=lambda x: -x[2])
                print(f"    Spearman |r| > 0.9: {len(_high_corr)} pairs (of {len(_vif_feats)} features)")
                for _a, _b, _r in _high_corr[:10]:
                    print(f"      {_a:40s} <-> {_b:40s} r={_r:.3f}")
            else:
                from statsmodels.stats.outliers_influence import variance_inflation_factor
                try:
                    _X = _sample[_vif_feats].values
                    _vif_vals = [variance_inflation_factor(_X, _k) for _k in range(len(_vif_feats))]
                    _vif_df = pd.DataFrame({'feature': _vif_feats, 'VIF': _vif_vals}).sort_values('VIF', ascending=False)
                    _high_vif = _vif_df[_vif_df['VIF'] > 10]
                    print(f"    VIF > 10: {len(_high_vif)}/{len(_vif_feats)} features")
                    for _, _row in _high_vif.head(10).iterrows():
                        print(f"      {_row['feature']:55s} VIF={_row['VIF']:.1f}")
                    save_result(_vif_df, 'card_drop_vif', 'per_parquet')
                except Exception as _e:
                    print(f"    VIF failed: {_e}")
        else:
            print(f"    Too few valid rows ({len(_sample)}) for VIF")
    else:
        print(f"    Too few features ({len(_vif_feats)}) for VIF")

    print(f"\n  A19 analysis complete")

del _pq_df
gc.collect()
print("  Memory freed")


CELL P2b: A19 -- card_drop
  CARD z-score drop accumulator (19 features: z_running_min, drop_count, max_z_drop, dates)
  Loaded card_drop: 62043 rows, 7 features, 14.7 MB
  Rows: 62043, Cities: 19, Damaged: 8080, Features: 7

  [1/3] PER-FEATURE AUC (top 15 across all cities)
    s1__vv__date_first_drop                                 AUC=0.525 (19 cities, 4 sig)
    s1__vv__lu_transition                                   AUC=0.524 (19 cities, 3 sig)
    s1__vv__max_z_drop                                      AUC=0.523 (19 cities, 0 sig)
    s1__vv__z_running_min                                   AUC=0.522 (19 cities, 0 sig)
    s1__vv__date_worst_drop                                 AUC=0.521 (19 cities, 4 sig)
    s1__vv__scenes_observed                                 AUC=0.516 (19 cities, 4 sig)
    s1__vv__drop_count                                      AUC=0.506 (19 cities, 1 sig)
    saved -> card_drop_auc.csv (133 rows)

  [2/3] FEATURE SEPARABILITY (J-M distance, top 10)
    s

In [21]:
# @title CELL P2c: A20 -- ms_change (AUC + VIF + Separability)
# Parquet: ms_change | MS change accumulator (SWIR brightness + NBR z) (34 features: swir_z, nbr_z, rise_count, dates + baselines)
import gc
import numpy as np
import pandas as pd
from scipy import stats as scipy_stats
from sklearn.metrics import roc_auc_score

print("=" * 70)
print("CELL P2c: A20 -- ms_change")
print("  MS change accumulator (SWIR brightness + NBR z) (34 features: swir_z, nbr_z, rise_count, dates + baselines)")
print("=" * 70)

_pq_df, _pq_feat, _pq_wide = get_analysis_df('ms_change')
if _pq_df is None or len(_pq_feat) == 0:
    print("  SKIP: parquet not found or no features")
else:
    _n_cities = _pq_df['city'].nunique()
    _n_dam = (_pq_df[TARGET_COL] == 1).sum() if TARGET_COL in _pq_df.columns else 0
    _n_total = len(_pq_df)
    print(f"  Rows: {_n_total}, Cities: {_n_cities}, Damaged: {_n_dam}, Features: {len(_pq_feat)}")

    # ---- 1. AUC per feature (top 15) ----
    print(f"\n  [1/3] PER-FEATURE AUC (top 15 across all cities)")
    _auc_rows = []
    for _city in sorted(_pq_df['city'].unique()):
        _cdf = _pq_df[_pq_df['city'] == _city]
        if TARGET_COL not in _cdf.columns or _cdf[TARGET_COL].nunique() < 2 or len(_cdf) < MIN_SAMPLES:
            continue
        for _col in _pq_feat:
            _valid = _cdf[[_col, TARGET_COL]].dropna()
            if len(_valid) < 20 or _valid[TARGET_COL].nunique() < 2:
                continue
            _dv = _valid.loc[_valid[TARGET_COL] == 1, _col].values
            _cv = _valid.loc[_valid[TARGET_COL] == 0, _col].values
            if len(_dv) < 5 or len(_cv) < 5:
                continue
            try:
                _auc = roc_auc_score(_valid[TARGET_COL], _valid[_col])
                _auc_best = max(_auc, 1 - _auc)
                _u, _p = scipy_stats.mannwhitneyu(_dv, _cv, alternative='two-sided')
                _auc_rows.append({'city': _city, 'feature': _col, 'auc_best': _auc_best,
                                  'mann_whitney_p': _p, 'direction': 'higher=damaged' if _auc >= 0.5 else 'lower=damaged'})
            except Exception:
                continue

    if _auc_rows:
        _auc_df = pd.DataFrame(_auc_rows)
        _top = _auc_df.groupby('feature')['auc_best'].mean().nlargest(15)
        for _f, _a in _top.items():
            _n_sig = (_auc_df[_auc_df['feature'] == _f]['mann_whitney_p'] < 0.05).sum()
            _n_cit = len(_auc_df[_auc_df['feature'] == _f])
            print(f"    {_f:55s} AUC={_a:.3f} ({_n_cit} cities, {_n_sig} sig)")
        save_result(_auc_df, 'ms_change_auc', 'per_parquet')
        log_nb06_result('ms_change', 'P2c', 'best_auc', float(_top.iloc[0]), feature=_top.index[0])
    else:
        print("    No features passed AUC filters")

    # ---- 2. SEPARABILITY (J-M distance, top 10) ----
    print(f"\n  [2/3] FEATURE SEPARABILITY (J-M distance, top 10)")
    _sep_rows = []
    for _city in sorted(_pq_df['city'].unique()):
        _cdf = _pq_df[_pq_df['city'] == _city]
        if TARGET_COL not in _cdf.columns or _cdf[TARGET_COL].nunique() < 2:
            continue
        for _col in _pq_feat:
            _d = _cdf.loc[_cdf[TARGET_COL] == 1, _col].dropna()
            _c = _cdf.loc[_cdf[TARGET_COL] == 0, _col].dropna()
            if len(_d) < 10 or len(_c) < 10:
                continue
            _mu1, _mu2 = _d.mean(), _c.mean()
            _v1, _v2 = _d.var(), _c.var()
            if _v1 <= 0 or _v2 <= 0:
                continue
            _bd = 0.25 * np.log(0.25 * (_v1/_v2 + _v2/_v1 + 2)) + 0.25 * ((_mu1-_mu2)**2) / (_v1+_v2)
            _jm = 2 * (1 - np.exp(-_bd))
            _sep_rows.append({'city': _city, 'feature': _col, 'jm_distance': _jm, 'bhattacharyya': _bd})

    if _sep_rows:
        _sep_df = pd.DataFrame(_sep_rows)
        _top_sep = _sep_df.groupby('feature')['jm_distance'].mean().nlargest(10)
        for _f, _jm in _top_sep.items():
            _interp = "excellent" if _jm > 1.9 else "good" if _jm > 1.5 else "moderate" if _jm > 1.0 else "poor"
            print(f"    {_f:55s} JM={_jm:.3f} ({_interp})")
        save_result(_sep_df, 'ms_change_separability', 'per_parquet')
    else:
        print("    No features passed separability filters")

    # ---- 3. VIF (top correlated pairs) ----
    print(f"\n  [3/3] VIF / COLLINEARITY CHECK")
    _vif_feats = [c for c in _pq_feat if _pq_df[c].notna().mean() > 0.5]
    if len(_vif_feats) > 3:
        _sample = _pq_df[_vif_feats].dropna()
        if len(_sample) > 100:
            if len(_vif_feats) > 50:
                # too many for full VIF, use Spearman top pairs
                _corr = _sample[_vif_feats[:50]].corr(method='spearman')
                _high_corr = []
                for _ii in range(len(_corr)):
                    for _jj in range(_ii+1, len(_corr)):
                        _r = abs(_corr.iloc[_ii, _jj])
                        if _r > 0.9:
                            _high_corr.append((_corr.index[_ii], _corr.columns[_jj], _r))
                _high_corr.sort(key=lambda x: -x[2])
                print(f"    Spearman |r| > 0.9: {len(_high_corr)} pairs (of {len(_vif_feats)} features)")
                for _a, _b, _r in _high_corr[:10]:
                    print(f"      {_a:40s} <-> {_b:40s} r={_r:.3f}")
            else:
                from statsmodels.stats.outliers_influence import variance_inflation_factor
                try:
                    _X = _sample[_vif_feats].values
                    _vif_vals = [variance_inflation_factor(_X, _k) for _k in range(len(_vif_feats))]
                    _vif_df = pd.DataFrame({'feature': _vif_feats, 'VIF': _vif_vals}).sort_values('VIF', ascending=False)
                    _high_vif = _vif_df[_vif_df['VIF'] > 10]
                    print(f"    VIF > 10: {len(_high_vif)}/{len(_vif_feats)} features")
                    for _, _row in _high_vif.head(10).iterrows():
                        print(f"      {_row['feature']:55s} VIF={_row['VIF']:.1f}")
                    save_result(_vif_df, 'ms_change_vif', 'per_parquet')
                except Exception as _e:
                    print(f"    VIF failed: {_e}")
        else:
            print(f"    Too few valid rows ({len(_sample)}) for VIF")
    else:
        print(f"    Too few features ({len(_vif_feats)}) for VIF")

    print(f"\n  A20 analysis complete")

del _pq_df
gc.collect()
print("  Memory freed")


CELL P2c: A20 -- ms_change
  MS change accumulator (SWIR brightness + NBR z) (34 features: swir_z, nbr_z, rise_count, dates + baselines)
  Loaded ms_change: 39428 rows, 9 features, 9.6 MB
  Rows: 39428, Cities: 18, Damaged: 4769, Features: 9

  [1/3] PER-FEATURE AUC (top 15 across all cities)
    s2__date_worst_swir_rise                                AUC=0.500 (5 cities, 0 sig)
    s2__lu_transition                                       AUC=0.500 (5 cities, 0 sig)
    s2__scenes_observed                                     AUC=0.500 (5 cities, 0 sig)
    s2__swir_baseline_mean                                  AUC=0.500 (5 cities, 0 sig)
    s2__swir_baseline_std                                   AUC=0.500 (5 cities, 0 sig)
    s2__swir_z_running_max                                  AUC=0.500 (5 cities, 0 sig)
    saved -> ms_change_auc.csv (30 rows)

  [2/3] FEATURE SEPARABILITY (J-M distance, top 10)
    No features passed separability filters

  [3/3] VIF / COLLINEARITY CHECK
    To

In [22]:
# @title CELL P2d: A21 -- ms_maha (AUC + VIF + Separability)
# Parquet: ms_maha | MS Mahalanobis distance accumulator (16 features: running_max, exceedance_count, dates)
import gc
import numpy as np
import pandas as pd
from scipy import stats as scipy_stats
from sklearn.metrics import roc_auc_score

print("=" * 70)
print("CELL P2d: A21 -- ms_maha")
print("  MS Mahalanobis distance accumulator (16 features: running_max, exceedance_count, dates)")
print("=" * 70)

_pq_df, _pq_feat, _pq_wide = get_analysis_df('ms_maha')
if _pq_df is None or len(_pq_feat) == 0:
    print("  SKIP: parquet not found or no features")
else:
    _n_cities = _pq_df['city'].nunique()
    _n_dam = (_pq_df[TARGET_COL] == 1).sum() if TARGET_COL in _pq_df.columns else 0
    _n_total = len(_pq_df)
    print(f"  Rows: {_n_total}, Cities: {_n_cities}, Damaged: {_n_dam}, Features: {len(_pq_feat)}")

    # ---- 1. AUC per feature (top 15) ----
    print(f"\n  [1/3] PER-FEATURE AUC (top 15 across all cities)")
    _auc_rows = []
    for _city in sorted(_pq_df['city'].unique()):
        _cdf = _pq_df[_pq_df['city'] == _city]
        if TARGET_COL not in _cdf.columns or _cdf[TARGET_COL].nunique() < 2 or len(_cdf) < MIN_SAMPLES:
            continue
        for _col in _pq_feat:
            _valid = _cdf[[_col, TARGET_COL]].dropna()
            if len(_valid) < 20 or _valid[TARGET_COL].nunique() < 2:
                continue
            _dv = _valid.loc[_valid[TARGET_COL] == 1, _col].values
            _cv = _valid.loc[_valid[TARGET_COL] == 0, _col].values
            if len(_dv) < 5 or len(_cv) < 5:
                continue
            try:
                _auc = roc_auc_score(_valid[TARGET_COL], _valid[_col])
                _auc_best = max(_auc, 1 - _auc)
                _u, _p = scipy_stats.mannwhitneyu(_dv, _cv, alternative='two-sided')
                _auc_rows.append({'city': _city, 'feature': _col, 'auc_best': _auc_best,
                                  'mann_whitney_p': _p, 'direction': 'higher=damaged' if _auc >= 0.5 else 'lower=damaged'})
            except Exception:
                continue

    if _auc_rows:
        _auc_df = pd.DataFrame(_auc_rows)
        _top = _auc_df.groupby('feature')['auc_best'].mean().nlargest(15)
        for _f, _a in _top.items():
            _n_sig = (_auc_df[_auc_df['feature'] == _f]['mann_whitney_p'] < 0.05).sum()
            _n_cit = len(_auc_df[_auc_df['feature'] == _f])
            print(f"    {_f:55s} AUC={_a:.3f} ({_n_cit} cities, {_n_sig} sig)")
        save_result(_auc_df, 'ms_maha_auc', 'per_parquet')
        log_nb06_result('ms_maha', 'P2d', 'best_auc', float(_top.iloc[0]), feature=_top.index[0])
    else:
        print("    No features passed AUC filters")

    # ---- 2. SEPARABILITY (J-M distance, top 10) ----
    print(f"\n  [2/3] FEATURE SEPARABILITY (J-M distance, top 10)")
    _sep_rows = []
    for _city in sorted(_pq_df['city'].unique()):
        _cdf = _pq_df[_pq_df['city'] == _city]
        if TARGET_COL not in _cdf.columns or _cdf[TARGET_COL].nunique() < 2:
            continue
        for _col in _pq_feat:
            _d = _cdf.loc[_cdf[TARGET_COL] == 1, _col].dropna()
            _c = _cdf.loc[_cdf[TARGET_COL] == 0, _col].dropna()
            if len(_d) < 10 or len(_c) < 10:
                continue
            _mu1, _mu2 = _d.mean(), _c.mean()
            _v1, _v2 = _d.var(), _c.var()
            if _v1 <= 0 or _v2 <= 0:
                continue
            _bd = 0.25 * np.log(0.25 * (_v1/_v2 + _v2/_v1 + 2)) + 0.25 * ((_mu1-_mu2)**2) / (_v1+_v2)
            _jm = 2 * (1 - np.exp(-_bd))
            _sep_rows.append({'city': _city, 'feature': _col, 'jm_distance': _jm, 'bhattacharyya': _bd})

    if _sep_rows:
        _sep_df = pd.DataFrame(_sep_rows)
        _top_sep = _sep_df.groupby('feature')['jm_distance'].mean().nlargest(10)
        for _f, _jm in _top_sep.items():
            _interp = "excellent" if _jm > 1.9 else "good" if _jm > 1.5 else "moderate" if _jm > 1.0 else "poor"
            print(f"    {_f:55s} JM={_jm:.3f} ({_interp})")
        save_result(_sep_df, 'ms_maha_separability', 'per_parquet')
    else:
        print("    No features passed separability filters")

    # ---- 3. VIF (top correlated pairs) ----
    print(f"\n  [3/3] VIF / COLLINEARITY CHECK")
    _vif_feats = [c for c in _pq_feat if _pq_df[c].notna().mean() > 0.5]
    if len(_vif_feats) > 3:
        _sample = _pq_df[_vif_feats].dropna()
        if len(_sample) > 100:
            if len(_vif_feats) > 50:
                # too many for full VIF, use Spearman top pairs
                _corr = _sample[_vif_feats[:50]].corr(method='spearman')
                _high_corr = []
                for _ii in range(len(_corr)):
                    for _jj in range(_ii+1, len(_corr)):
                        _r = abs(_corr.iloc[_ii, _jj])
                        if _r > 0.9:
                            _high_corr.append((_corr.index[_ii], _corr.columns[_jj], _r))
                _high_corr.sort(key=lambda x: -x[2])
                print(f"    Spearman |r| > 0.9: {len(_high_corr)} pairs (of {len(_vif_feats)} features)")
                for _a, _b, _r in _high_corr[:10]:
                    print(f"      {_a:40s} <-> {_b:40s} r={_r:.3f}")
            else:
                from statsmodels.stats.outliers_influence import variance_inflation_factor
                try:
                    _X = _sample[_vif_feats].values
                    _vif_vals = [variance_inflation_factor(_X, _k) for _k in range(len(_vif_feats))]
                    _vif_df = pd.DataFrame({'feature': _vif_feats, 'VIF': _vif_vals}).sort_values('VIF', ascending=False)
                    _high_vif = _vif_df[_vif_df['VIF'] > 10]
                    print(f"    VIF > 10: {len(_high_vif)}/{len(_vif_feats)} features")
                    for _, _row in _high_vif.head(10).iterrows():
                        print(f"      {_row['feature']:55s} VIF={_row['VIF']:.1f}")
                    save_result(_vif_df, 'ms_maha_vif', 'per_parquet')
                except Exception as _e:
                    print(f"    VIF failed: {_e}")
        else:
            print(f"    Too few valid rows ({len(_sample)}) for VIF")
    else:
        print(f"    Too few features ({len(_vif_feats)}) for VIF")

    print(f"\n  A21 analysis complete")

del _pq_df
gc.collect()
print("  Memory freed")


CELL P2d: A21 -- ms_maha
  MS Mahalanobis distance accumulator (16 features: running_max, exceedance_count, dates)
  Loaded ms_maha: 39428 rows, 6 features, 9.1 MB
  Rows: 39428, Cities: 18, Damaged: 4769, Features: 6

  [1/3] PER-FEATURE AUC (top 15 across all cities)
    s2__mahalanobis_running_max                             AUC=0.599 (18 cities, 10 sig)
    s2__date_worst_exceedance                               AUC=0.529 (18 cities, 7 sig)
    s2__lu_transition                                       AUC=0.520 (18 cities, 1 sig)
    s2__scenes_observed                                     AUC=0.515 (18 cities, 4 sig)
    s2__date_first_exceedance                               AUC=0.511 (18 cities, 4 sig)
    s2__mahalanobis_exceedance_count                        AUC=0.506 (18 cities, 2 sig)
    saved -> ms_maha_auc.csv (108 rows)

  [2/3] FEATURE SEPARABILITY (J-M distance, top 10)
    s2__mahalanobis_exceedance_count                        JM=0.564 (poor)
    s2__date_first_exceeda

In [23]:
# @title CELL P2e: A22 -- lu_change (AUC + VIF + Separability)
# Parquet: lu_change | Landuse change accumulator (urban to other) (20 features: loss_count, loss_fraction, urban_retained, classes)
import gc
import numpy as np
import pandas as pd
from scipy import stats as scipy_stats
from sklearn.metrics import roc_auc_score

print("=" * 70)
print("CELL P2e: A22 -- lu_change")
print("  Landuse change accumulator (urban to other) (20 features: loss_count, loss_fraction, urban_retained, classes)")
print("=" * 70)

_pq_df, _pq_feat, _pq_wide = get_analysis_df('lu_change')
if _pq_df is None or len(_pq_feat) == 0:
    print("  SKIP: parquet not found or no features")
else:
    _n_cities = _pq_df['city'].nunique()
    _n_dam = (_pq_df[TARGET_COL] == 1).sum() if TARGET_COL in _pq_df.columns else 0
    _n_total = len(_pq_df)
    print(f"  Rows: {_n_total}, Cities: {_n_cities}, Damaged: {_n_dam}, Features: {len(_pq_feat)}")

    # ---- 1. AUC per feature (top 15) ----
    print(f"\n  [1/3] PER-FEATURE AUC (top 15 across all cities)")
    _auc_rows = []
    for _city in sorted(_pq_df['city'].unique()):
        _cdf = _pq_df[_pq_df['city'] == _city]
        if TARGET_COL not in _cdf.columns or _cdf[TARGET_COL].nunique() < 2 or len(_cdf) < MIN_SAMPLES:
            continue
        for _col in _pq_feat:
            _valid = _cdf[[_col, TARGET_COL]].dropna()
            if len(_valid) < 20 or _valid[TARGET_COL].nunique() < 2:
                continue
            _dv = _valid.loc[_valid[TARGET_COL] == 1, _col].values
            _cv = _valid.loc[_valid[TARGET_COL] == 0, _col].values
            if len(_dv) < 5 or len(_cv) < 5:
                continue
            try:
                _auc = roc_auc_score(_valid[TARGET_COL], _valid[_col])
                _auc_best = max(_auc, 1 - _auc)
                _u, _p = scipy_stats.mannwhitneyu(_dv, _cv, alternative='two-sided')
                _auc_rows.append({'city': _city, 'feature': _col, 'auc_best': _auc_best,
                                  'mann_whitney_p': _p, 'direction': 'higher=damaged' if _auc >= 0.5 else 'lower=damaged'})
            except Exception:
                continue

    if _auc_rows:
        _auc_df = pd.DataFrame(_auc_rows)
        _top = _auc_df.groupby('feature')['auc_best'].mean().nlargest(15)
        for _f, _a in _top.items():
            _n_sig = (_auc_df[_auc_df['feature'] == _f]['mann_whitney_p'] < 0.05).sum()
            _n_cit = len(_auc_df[_auc_df['feature'] == _f])
            print(f"    {_f:55s} AUC={_a:.3f} ({_n_cit} cities, {_n_sig} sig)")
        save_result(_auc_df, 'lu_change_auc', 'per_parquet')
        log_nb06_result('lu_change', 'P2e', 'best_auc', float(_top.iloc[0]), feature=_top.index[0])
    else:
        print("    No features passed AUC filters")

    # ---- 2. SEPARABILITY (J-M distance, top 10) ----
    print(f"\n  [2/3] FEATURE SEPARABILITY (J-M distance, top 10)")
    _sep_rows = []
    for _city in sorted(_pq_df['city'].unique()):
        _cdf = _pq_df[_pq_df['city'] == _city]
        if TARGET_COL not in _cdf.columns or _cdf[TARGET_COL].nunique() < 2:
            continue
        for _col in _pq_feat:
            _d = _cdf.loc[_cdf[TARGET_COL] == 1, _col].dropna()
            _c = _cdf.loc[_cdf[TARGET_COL] == 0, _col].dropna()
            if len(_d) < 10 or len(_c) < 10:
                continue
            _mu1, _mu2 = _d.mean(), _c.mean()
            _v1, _v2 = _d.var(), _c.var()
            if _v1 <= 0 or _v2 <= 0:
                continue
            _bd = 0.25 * np.log(0.25 * (_v1/_v2 + _v2/_v1 + 2)) + 0.25 * ((_mu1-_mu2)**2) / (_v1+_v2)
            _jm = 2 * (1 - np.exp(-_bd))
            _sep_rows.append({'city': _city, 'feature': _col, 'jm_distance': _jm, 'bhattacharyya': _bd})

    if _sep_rows:
        _sep_df = pd.DataFrame(_sep_rows)
        _top_sep = _sep_df.groupby('feature')['jm_distance'].mean().nlargest(10)
        for _f, _jm in _top_sep.items():
            _interp = "excellent" if _jm > 1.9 else "good" if _jm > 1.5 else "moderate" if _jm > 1.0 else "poor"
            print(f"    {_f:55s} JM={_jm:.3f} ({_interp})")
        save_result(_sep_df, 'lu_change_separability', 'per_parquet')
    else:
        print("    No features passed separability filters")

    # ---- 3. VIF (top correlated pairs) ----
    print(f"\n  [3/3] VIF / COLLINEARITY CHECK")
    _vif_feats = [c for c in _pq_feat if _pq_df[c].notna().mean() > 0.5]
    if len(_vif_feats) > 3:
        _sample = _pq_df[_vif_feats].dropna()
        if len(_sample) > 100:
            if len(_vif_feats) > 50:
                # too many for full VIF, use Spearman top pairs
                _corr = _sample[_vif_feats[:50]].corr(method='spearman')
                _high_corr = []
                for _ii in range(len(_corr)):
                    for _jj in range(_ii+1, len(_corr)):
                        _r = abs(_corr.iloc[_ii, _jj])
                        if _r > 0.9:
                            _high_corr.append((_corr.index[_ii], _corr.columns[_jj], _r))
                _high_corr.sort(key=lambda x: -x[2])
                print(f"    Spearman |r| > 0.9: {len(_high_corr)} pairs (of {len(_vif_feats)} features)")
                for _a, _b, _r in _high_corr[:10]:
                    print(f"      {_a:40s} <-> {_b:40s} r={_r:.3f}")
            else:
                from statsmodels.stats.outliers_influence import variance_inflation_factor
                try:
                    _X = _sample[_vif_feats].values
                    _vif_vals = [variance_inflation_factor(_X, _k) for _k in range(len(_vif_feats))]
                    _vif_df = pd.DataFrame({'feature': _vif_feats, 'VIF': _vif_vals}).sort_values('VIF', ascending=False)
                    _high_vif = _vif_df[_vif_df['VIF'] > 10]
                    print(f"    VIF > 10: {len(_high_vif)}/{len(_vif_feats)} features")
                    for _, _row in _high_vif.head(10).iterrows():
                        print(f"      {_row['feature']:55s} VIF={_row['VIF']:.1f}")
                    save_result(_vif_df, 'lu_change_vif', 'per_parquet')
                except Exception as _e:
                    print(f"    VIF failed: {_e}")
        else:
            print(f"    Too few valid rows ({len(_sample)}) for VIF")
    else:
        print(f"    Too few features ({len(_vif_feats)}) for VIF")

    print(f"\n  A22 analysis complete")

del _pq_df
gc.collect()
print("  Memory freed")


CELL P2e: A22 -- lu_change
  Landuse change accumulator (urban to other) (20 features: loss_count, loss_fraction, urban_retained, classes)
  Loaded lu_change: 62043 rows, 8 features, 14.9 MB
  Rows: 62043, Cities: 19, Damaged: 8080, Features: 8

  [1/3] PER-FEATURE AUC (top 15 across all cities)
    s2__lu__loss_fraction                                   AUC=0.563 (19 cities, 8 sig)
    s2__lu__loss_count                                      AUC=0.541 (19 cities, 10 sig)
    s2__lu__date_persistent_loss                            AUC=0.531 (19 cities, 5 sig)
    s2__lu__modal_post_class                                AUC=0.531 (19 cities, 6 sig)
    s2__lu__date_first_loss                                 AUC=0.530 (19 cities, 6 sig)
    s2__lu__scenes_observed                                 AUC=0.523 (19 cities, 3 sig)
    s2__lu__final_class                                     AUC=0.513 (19 cities, 3 sig)
    s2__lu__urban_retained                                  AUC=0.500 (19 citie

In [24]:
# @title CELL P3: A13 -- prepost_single_card (AUC + VIF + Separability)
# Parquet: prepost_single_card | Single-scene CARD pre/post
import gc
import numpy as np
import pandas as pd
from scipy import stats as scipy_stats
from sklearn.metrics import roc_auc_score

print("=" * 70)
print("CELL P3: A13 -- prepost_single_card")
print("  Single-scene CARD pre/post")
print("=" * 70)

_pq_df, _pq_feat, _pq_wide = get_analysis_df('prepost_single_card')
if _pq_df is None or len(_pq_feat) == 0:
    print("  SKIP: parquet not found or no features")
else:
    _n_cities = _pq_df['city'].nunique()
    _n_dam = (_pq_df[TARGET_COL] == 1).sum() if TARGET_COL in _pq_df.columns else 0
    _n_total = len(_pq_df)
    print(f"  Rows: {_n_total}, Cities: {_n_cities}, Damaged: {_n_dam}, Features: {len(_pq_feat)}")

    # ---- 1. AUC per feature (top 15) ----
    print(f"\n  [1/3] PER-FEATURE AUC (top 15 across all cities)")
    _auc_rows = []
    for _city in sorted(_pq_df['city'].unique()):
        _cdf = _pq_df[_pq_df['city'] == _city]
        if TARGET_COL not in _cdf.columns or _cdf[TARGET_COL].nunique() < 2 or len(_cdf) < MIN_SAMPLES:
            continue
        for _col in _pq_feat:
            _valid = _cdf[[_col, TARGET_COL]].dropna()
            if len(_valid) < 20 or _valid[TARGET_COL].nunique() < 2:
                continue
            _dv = _valid.loc[_valid[TARGET_COL] == 1, _col].values
            _cv = _valid.loc[_valid[TARGET_COL] == 0, _col].values
            if len(_dv) < 5 or len(_cv) < 5:
                continue
            try:
                _auc = roc_auc_score(_valid[TARGET_COL], _valid[_col])
                _auc_best = max(_auc, 1 - _auc)
                _u, _p = scipy_stats.mannwhitneyu(_dv, _cv, alternative='two-sided')
                _auc_rows.append({'city': _city, 'feature': _col, 'auc_best': _auc_best,
                                  'mann_whitney_p': _p, 'direction': 'higher=damaged' if _auc >= 0.5 else 'lower=damaged'})
            except Exception:
                continue

    if _auc_rows:
        _auc_df = pd.DataFrame(_auc_rows)
        _top = _auc_df.groupby('feature')['auc_best'].mean().nlargest(15)
        for _f, _a in _top.items():
            _n_sig = (_auc_df[_auc_df['feature'] == _f]['mann_whitney_p'] < 0.05).sum()
            _n_cit = len(_auc_df[_auc_df['feature'] == _f])
            print(f"    {_f:55s} AUC={_a:.3f} ({_n_cit} cities, {_n_sig} sig)")
        save_result(_auc_df, 'prepost_single_card_auc', 'per_parquet')
        log_nb06_result('prepost_single_card', 'P3', 'best_auc', float(_top.iloc[0]), feature=_top.index[0])
    else:
        print("    No features passed AUC filters")

    # ---- 2. SEPARABILITY (J-M distance, top 10) ----
    print(f"\n  [2/3] FEATURE SEPARABILITY (J-M distance, top 10)")
    _sep_rows = []
    for _city in sorted(_pq_df['city'].unique()):
        _cdf = _pq_df[_pq_df['city'] == _city]
        if TARGET_COL not in _cdf.columns or _cdf[TARGET_COL].nunique() < 2:
            continue
        for _col in _pq_feat:
            _d = _cdf.loc[_cdf[TARGET_COL] == 1, _col].dropna()
            _c = _cdf.loc[_cdf[TARGET_COL] == 0, _col].dropna()
            if len(_d) < 10 or len(_c) < 10:
                continue
            _mu1, _mu2 = _d.mean(), _c.mean()
            _v1, _v2 = _d.var(), _c.var()
            if _v1 <= 0 or _v2 <= 0:
                continue
            _bd = 0.25 * np.log(0.25 * (_v1/_v2 + _v2/_v1 + 2)) + 0.25 * ((_mu1-_mu2)**2) / (_v1+_v2)
            _jm = 2 * (1 - np.exp(-_bd))
            _sep_rows.append({'city': _city, 'feature': _col, 'jm_distance': _jm, 'bhattacharyya': _bd})

    if _sep_rows:
        _sep_df = pd.DataFrame(_sep_rows)
        _top_sep = _sep_df.groupby('feature')['jm_distance'].mean().nlargest(10)
        for _f, _jm in _top_sep.items():
            _interp = "excellent" if _jm > 1.9 else "good" if _jm > 1.5 else "moderate" if _jm > 1.0 else "poor"
            print(f"    {_f:55s} JM={_jm:.3f} ({_interp})")
        save_result(_sep_df, 'prepost_single_card_separability', 'per_parquet')
    else:
        print("    No features passed separability filters")

    # ---- 3. VIF (top correlated pairs) ----
    print(f"\n  [3/3] VIF / COLLINEARITY CHECK")
    _vif_feats = [c for c in _pq_feat if _pq_df[c].notna().mean() > 0.5]
    if len(_vif_feats) > 3:
        _sample = _pq_df[_vif_feats].dropna()
        if len(_sample) > 100:
            if len(_vif_feats) > 50:
                # too many for full VIF, use Spearman top pairs
                _corr = _sample[_vif_feats[:50]].corr(method='spearman')
                _high_corr = []
                for _ii in range(len(_corr)):
                    for _jj in range(_ii+1, len(_corr)):
                        _r = abs(_corr.iloc[_ii, _jj])
                        if _r > 0.9:
                            _high_corr.append((_corr.index[_ii], _corr.columns[_jj], _r))
                _high_corr.sort(key=lambda x: -x[2])
                print(f"    Spearman |r| > 0.9: {len(_high_corr)} pairs (of {len(_vif_feats)} features)")
                for _a, _b, _r in _high_corr[:10]:
                    print(f"      {_a:40s} <-> {_b:40s} r={_r:.3f}")
            else:
                from statsmodels.stats.outliers_influence import variance_inflation_factor
                try:
                    _X = _sample[_vif_feats].values
                    _vif_vals = [variance_inflation_factor(_X, _k) for _k in range(len(_vif_feats))]
                    _vif_df = pd.DataFrame({'feature': _vif_feats, 'VIF': _vif_vals}).sort_values('VIF', ascending=False)
                    _high_vif = _vif_df[_vif_df['VIF'] > 10]
                    print(f"    VIF > 10: {len(_high_vif)}/{len(_vif_feats)} features")
                    for _, _row in _high_vif.head(10).iterrows():
                        print(f"      {_row['feature']:55s} VIF={_row['VIF']:.1f}")
                    save_result(_vif_df, 'prepost_single_card_vif', 'per_parquet')
                except Exception as _e:
                    print(f"    VIF failed: {_e}")
        else:
            print(f"    Too few valid rows ({len(_sample)}) for VIF")
    else:
        print(f"    Too few features ({len(_vif_feats)}) for VIF")

    print(f"\n  A13 analysis complete")

del _pq_df
gc.collect()
print("  Memory freed")


CELL P3: A13 -- prepost_single_card
  Single-scene CARD pre/post
  Loaded prepost_single_card: 63243 rows, 6 features, 16.7 MB
  Rows: 63243, Cities: 21, Damaged: 8247, Features: 6

  [1/3] PER-FEATURE AUC (top 15 across all cities)
    post_vv                                                 AUC=0.554 (21 cities, 10 sig)
    delta_vh                                                AUC=0.548 (21 cities, 5 sig)
    pre_vh                                                  AUC=0.544 (21 cities, 4 sig)
    pre_vv                                                  AUC=0.543 (21 cities, 7 sig)
    delta_vv                                                AUC=0.536 (21 cities, 3 sig)
    post_vh                                                 AUC=0.534 (21 cities, 4 sig)
    saved -> prepost_single_card_auc.csv (126 rows)

  [2/3] FEATURE SEPARABILITY (J-M distance, top 10)
    post_vv                                                 JM=0.076 (poor)
    pre_vv                                         

In [25]:
# @title CELL P4: A10 -- composite_prepost_landuse (AUC + VIF + Separability)
# Parquet: composite_prepost_landuse | Composite landuse pre vs post
import gc
import numpy as np
import pandas as pd
from scipy import stats as scipy_stats
from sklearn.metrics import roc_auc_score

print("=" * 70)
print("CELL P4: A10 -- composite_prepost_landuse")
print("  Composite landuse pre vs post")
print("=" * 70)

_pq_df, _pq_feat, _pq_wide = get_analysis_df('composite_prepost_landuse')
if _pq_df is None or len(_pq_feat) == 0:
    print("  SKIP: parquet not found or no features")
else:
    _n_cities = _pq_df['city'].nunique()
    _n_dam = (_pq_df[TARGET_COL] == 1).sum() if TARGET_COL in _pq_df.columns else 0
    _n_total = len(_pq_df)
    print(f"  Rows: {_n_total}, Cities: {_n_cities}, Damaged: {_n_dam}, Features: {len(_pq_feat)}")

    # ---- 1. AUC per feature (top 15) ----
    print(f"\n  [1/3] PER-FEATURE AUC (top 15 across all cities)")
    _auc_rows = []
    for _city in sorted(_pq_df['city'].unique()):
        _cdf = _pq_df[_pq_df['city'] == _city]
        if TARGET_COL not in _cdf.columns or _cdf[TARGET_COL].nunique() < 2 or len(_cdf) < MIN_SAMPLES:
            continue
        for _col in _pq_feat:
            _valid = _cdf[[_col, TARGET_COL]].dropna()
            if len(_valid) < 20 or _valid[TARGET_COL].nunique() < 2:
                continue
            _dv = _valid.loc[_valid[TARGET_COL] == 1, _col].values
            _cv = _valid.loc[_valid[TARGET_COL] == 0, _col].values
            if len(_dv) < 5 or len(_cv) < 5:
                continue
            try:
                _auc = roc_auc_score(_valid[TARGET_COL], _valid[_col])
                _auc_best = max(_auc, 1 - _auc)
                _u, _p = scipy_stats.mannwhitneyu(_dv, _cv, alternative='two-sided')
                _auc_rows.append({'city': _city, 'feature': _col, 'auc_best': _auc_best,
                                  'mann_whitney_p': _p, 'direction': 'higher=damaged' if _auc >= 0.5 else 'lower=damaged'})
            except Exception:
                continue

    if _auc_rows:
        _auc_df = pd.DataFrame(_auc_rows)
        _top = _auc_df.groupby('feature')['auc_best'].mean().nlargest(15)
        for _f, _a in _top.items():
            _n_sig = (_auc_df[_auc_df['feature'] == _f]['mann_whitney_p'] < 0.05).sum()
            _n_cit = len(_auc_df[_auc_df['feature'] == _f])
            print(f"    {_f:55s} AUC={_a:.3f} ({_n_cit} cities, {_n_sig} sig)")
        save_result(_auc_df, 'composite_prepost_landuse_auc', 'per_parquet')
        log_nb06_result('composite_prepost_landuse', 'P4', 'best_auc', float(_top.iloc[0]), feature=_top.index[0])
    else:
        print("    No features passed AUC filters")

    # ---- 2. SEPARABILITY (J-M distance, top 10) ----
    print(f"\n  [2/3] FEATURE SEPARABILITY (J-M distance, top 10)")
    _sep_rows = []
    for _city in sorted(_pq_df['city'].unique()):
        _cdf = _pq_df[_pq_df['city'] == _city]
        if TARGET_COL not in _cdf.columns or _cdf[TARGET_COL].nunique() < 2:
            continue
        for _col in _pq_feat:
            _d = _cdf.loc[_cdf[TARGET_COL] == 1, _col].dropna()
            _c = _cdf.loc[_cdf[TARGET_COL] == 0, _col].dropna()
            if len(_d) < 10 or len(_c) < 10:
                continue
            _mu1, _mu2 = _d.mean(), _c.mean()
            _v1, _v2 = _d.var(), _c.var()
            if _v1 <= 0 or _v2 <= 0:
                continue
            _bd = 0.25 * np.log(0.25 * (_v1/_v2 + _v2/_v1 + 2)) + 0.25 * ((_mu1-_mu2)**2) / (_v1+_v2)
            _jm = 2 * (1 - np.exp(-_bd))
            _sep_rows.append({'city': _city, 'feature': _col, 'jm_distance': _jm, 'bhattacharyya': _bd})

    if _sep_rows:
        _sep_df = pd.DataFrame(_sep_rows)
        _top_sep = _sep_df.groupby('feature')['jm_distance'].mean().nlargest(10)
        for _f, _jm in _top_sep.items():
            _interp = "excellent" if _jm > 1.9 else "good" if _jm > 1.5 else "moderate" if _jm > 1.0 else "poor"
            print(f"    {_f:55s} JM={_jm:.3f} ({_interp})")
        save_result(_sep_df, 'composite_prepost_landuse_separability', 'per_parquet')
    else:
        print("    No features passed separability filters")

    # ---- 3. VIF (top correlated pairs) ----
    print(f"\n  [3/3] VIF / COLLINEARITY CHECK")
    _vif_feats = [c for c in _pq_feat if _pq_df[c].notna().mean() > 0.5]
    if len(_vif_feats) > 3:
        _sample = _pq_df[_vif_feats].dropna()
        if len(_sample) > 100:
            if len(_vif_feats) > 50:
                # too many for full VIF, use Spearman top pairs
                _corr = _sample[_vif_feats[:50]].corr(method='spearman')
                _high_corr = []
                for _ii in range(len(_corr)):
                    for _jj in range(_ii+1, len(_corr)):
                        _r = abs(_corr.iloc[_ii, _jj])
                        if _r > 0.9:
                            _high_corr.append((_corr.index[_ii], _corr.columns[_jj], _r))
                _high_corr.sort(key=lambda x: -x[2])
                print(f"    Spearman |r| > 0.9: {len(_high_corr)} pairs (of {len(_vif_feats)} features)")
                for _a, _b, _r in _high_corr[:10]:
                    print(f"      {_a:40s} <-> {_b:40s} r={_r:.3f}")
            else:
                from statsmodels.stats.outliers_influence import variance_inflation_factor
                try:
                    _X = _sample[_vif_feats].values
                    _vif_vals = [variance_inflation_factor(_X, _k) for _k in range(len(_vif_feats))]
                    _vif_df = pd.DataFrame({'feature': _vif_feats, 'VIF': _vif_vals}).sort_values('VIF', ascending=False)
                    _high_vif = _vif_df[_vif_df['VIF'] > 10]
                    print(f"    VIF > 10: {len(_high_vif)}/{len(_vif_feats)} features")
                    for _, _row in _high_vif.head(10).iterrows():
                        print(f"      {_row['feature']:55s} VIF={_row['VIF']:.1f}")
                    save_result(_vif_df, 'composite_prepost_landuse_vif', 'per_parquet')
                except Exception as _e:
                    print(f"    VIF failed: {_e}")
        else:
            print(f"    Too few valid rows ({len(_sample)}) for VIF")
    else:
        print(f"    Too few features ({len(_vif_feats)}) for VIF")

    print(f"\n  A10 analysis complete")

del _pq_df
gc.collect()
print("  Memory freed")


CELL P4: A10 -- composite_prepost_landuse
  Composite landuse pre vs post
  Loaded composite_prepost_landuse: 62043 rows, 4 features, 14.2 MB
  Rows: 62043, Cities: 19, Damaged: 8080, Features: 4

  [1/3] PER-FEATURE AUC (top 15 across all cities)
    landuse__prebattle_baseline                             AUC=0.576 (19 cities, 10 sig)
    landuse__winter_baseline                                AUC=0.574 (19 cities, 10 sig)
    landuse__post_winter_baseline                           AUC=0.568 (19 cities, 10 sig)
    landuse_changed                                         AUC=0.521 (19 cities, 4 sig)
    saved -> composite_prepost_landuse_auc.csv (76 rows)

  [2/3] FEATURE SEPARABILITY (J-M distance, top 10)
    landuse__post_winter_baseline                           JM=0.075 (poor)
    landuse__prebattle_baseline                             JM=0.059 (poor)
    landuse__winter_baseline                                JM=0.057 (poor)
    landuse_changed                                    

In [26]:
# @title CELL P5: A16 -- rolling_stats_roll3 (AUC + VIF + Separability)
# Parquet: rolling_stats_roll3 | Rolling assessment stats (window=3)
import gc
import numpy as np
import pandas as pd
from scipy import stats as scipy_stats
from sklearn.metrics import roc_auc_score

print("=" * 70)
print("CELL P5: A16 -- rolling_stats_roll3")
print("  Rolling assessment stats (window=3)")
print("=" * 70)

_pq_df, _pq_feat, _pq_wide = get_analysis_df('rolling_stats_roll3')
if _pq_df is None or len(_pq_feat) == 0:
    print("  SKIP: parquet not found or no features")
else:
    _n_cities = _pq_df['city'].nunique()
    _n_dam = (_pq_df[TARGET_COL] == 1).sum() if TARGET_COL in _pq_df.columns else 0
    _n_total = len(_pq_df)
    print(f"  Rows: {_n_total}, Cities: {_n_cities}, Damaged: {_n_dam}, Features: {len(_pq_feat)}")

    # ---- 1. AUC per feature (top 15) ----
    print(f"\n  [1/3] PER-FEATURE AUC (top 15 across all cities)")
    _auc_rows = []
    for _city in sorted(_pq_df['city'].unique()):
        _cdf = _pq_df[_pq_df['city'] == _city]
        if TARGET_COL not in _cdf.columns or _cdf[TARGET_COL].nunique() < 2 or len(_cdf) < MIN_SAMPLES:
            continue
        for _col in _pq_feat:
            _valid = _cdf[[_col, TARGET_COL]].dropna()
            if len(_valid) < 20 or _valid[TARGET_COL].nunique() < 2:
                continue
            _dv = _valid.loc[_valid[TARGET_COL] == 1, _col].values
            _cv = _valid.loc[_valid[TARGET_COL] == 0, _col].values
            if len(_dv) < 5 or len(_cv) < 5:
                continue
            try:
                _auc = roc_auc_score(_valid[TARGET_COL], _valid[_col])
                _auc_best = max(_auc, 1 - _auc)
                _u, _p = scipy_stats.mannwhitneyu(_dv, _cv, alternative='two-sided')
                _auc_rows.append({'city': _city, 'feature': _col, 'auc_best': _auc_best,
                                  'mann_whitney_p': _p, 'direction': 'higher=damaged' if _auc >= 0.5 else 'lower=damaged'})
            except Exception:
                continue

    if _auc_rows:
        _auc_df = pd.DataFrame(_auc_rows)
        _top = _auc_df.groupby('feature')['auc_best'].mean().nlargest(15)
        for _f, _a in _top.items():
            _n_sig = (_auc_df[_auc_df['feature'] == _f]['mann_whitney_p'] < 0.05).sum()
            _n_cit = len(_auc_df[_auc_df['feature'] == _f])
            print(f"    {_f:55s} AUC={_a:.3f} ({_n_cit} cities, {_n_sig} sig)")
        save_result(_auc_df, 'rolling_stats_roll3_auc', 'per_parquet')
        log_nb06_result('rolling_stats_roll3', 'P5', 'best_auc', float(_top.iloc[0]), feature=_top.index[0])
    else:
        print("    No features passed AUC filters")

    # ---- 2. SEPARABILITY (J-M distance, top 10) ----
    print(f"\n  [2/3] FEATURE SEPARABILITY (J-M distance, top 10)")
    _sep_rows = []
    for _city in sorted(_pq_df['city'].unique()):
        _cdf = _pq_df[_pq_df['city'] == _city]
        if TARGET_COL not in _cdf.columns or _cdf[TARGET_COL].nunique() < 2:
            continue
        for _col in _pq_feat:
            _d = _cdf.loc[_cdf[TARGET_COL] == 1, _col].dropna()
            _c = _cdf.loc[_cdf[TARGET_COL] == 0, _col].dropna()
            if len(_d) < 10 or len(_c) < 10:
                continue
            _mu1, _mu2 = _d.mean(), _c.mean()
            _v1, _v2 = _d.var(), _c.var()
            if _v1 <= 0 or _v2 <= 0:
                continue
            _bd = 0.25 * np.log(0.25 * (_v1/_v2 + _v2/_v1 + 2)) + 0.25 * ((_mu1-_mu2)**2) / (_v1+_v2)
            _jm = 2 * (1 - np.exp(-_bd))
            _sep_rows.append({'city': _city, 'feature': _col, 'jm_distance': _jm, 'bhattacharyya': _bd})

    if _sep_rows:
        _sep_df = pd.DataFrame(_sep_rows)
        _top_sep = _sep_df.groupby('feature')['jm_distance'].mean().nlargest(10)
        for _f, _jm in _top_sep.items():
            _interp = "excellent" if _jm > 1.9 else "good" if _jm > 1.5 else "moderate" if _jm > 1.0 else "poor"
            print(f"    {_f:55s} JM={_jm:.3f} ({_interp})")
        save_result(_sep_df, 'rolling_stats_roll3_separability', 'per_parquet')
    else:
        print("    No features passed separability filters")

    # ---- 3. VIF (top correlated pairs) ----
    print(f"\n  [3/3] VIF / COLLINEARITY CHECK")
    _vif_feats = [c for c in _pq_feat if _pq_df[c].notna().mean() > 0.5]
    if len(_vif_feats) > 3:
        _sample = _pq_df[_vif_feats].dropna()
        if len(_sample) > 100:
            if len(_vif_feats) > 50:
                # too many for full VIF, use Spearman top pairs
                _corr = _sample[_vif_feats[:50]].corr(method='spearman')
                _high_corr = []
                for _ii in range(len(_corr)):
                    for _jj in range(_ii+1, len(_corr)):
                        _r = abs(_corr.iloc[_ii, _jj])
                        if _r > 0.9:
                            _high_corr.append((_corr.index[_ii], _corr.columns[_jj], _r))
                _high_corr.sort(key=lambda x: -x[2])
                print(f"    Spearman |r| > 0.9: {len(_high_corr)} pairs (of {len(_vif_feats)} features)")
                for _a, _b, _r in _high_corr[:10]:
                    print(f"      {_a:40s} <-> {_b:40s} r={_r:.3f}")
            else:
                from statsmodels.stats.outliers_influence import variance_inflation_factor
                try:
                    _X = _sample[_vif_feats].values
                    _vif_vals = [variance_inflation_factor(_X, _k) for _k in range(len(_vif_feats))]
                    _vif_df = pd.DataFrame({'feature': _vif_feats, 'VIF': _vif_vals}).sort_values('VIF', ascending=False)
                    _high_vif = _vif_df[_vif_df['VIF'] > 10]
                    print(f"    VIF > 10: {len(_high_vif)}/{len(_vif_feats)} features")
                    for _, _row in _high_vif.head(10).iterrows():
                        print(f"      {_row['feature']:55s} VIF={_row['VIF']:.1f}")
                    save_result(_vif_df, 'rolling_stats_roll3_vif', 'per_parquet')
                except Exception as _e:
                    print(f"    VIF failed: {_e}")
        else:
            print(f"    Too few valid rows ({len(_sample)}) for VIF")
    else:
        print(f"    Too few features ({len(_vif_feats)}) for VIF")

    print(f"\n  A16 analysis complete")

del _pq_df
gc.collect()
print("  Memory freed")


CELL P5: A16 -- rolling_stats_roll3
  Rolling assessment stats (window=3)
  Loaded rolling_stats_roll3: 63243 rows, 38 features, 22.8 MB
  Rows: 63243, Cities: 21, Damaged: 8247, Features: 38

  [1/3] PER-FEATURE AUC (top 15 across all cities)
    s1__vv__baseline__mean                                  AUC=0.557 (21 cities, 10 sig)
    s1__vv__baseline__median                                AUC=0.555 (21 cities, 10 sig)
    s1__vv__baseline__min                                   AUC=0.553 (21 cities, 8 sig)
    s1__vv__baseline__max                                   AUC=0.549 (21 cities, 7 sig)
    s1__vh__baseline__min                                   AUC=0.547 (21 cities, 5 sig)
    s1__vh__baseline__mean                                  AUC=0.547 (21 cities, 5 sig)
    s1__vh__baseline__max                                   AUC=0.547 (21 cities, 6 sig)
    s1__vh__baseline__median                                AUC=0.545 (21 cities, 5 sig)
    s1__vv__baseline__skewness            

In [27]:
# @title CELL P6: A17 -- rolling_stats_roll7 (AUC + VIF + Separability)
# Parquet: rolling_stats_roll7 | Rolling assessment stats (window=7)
import gc
import numpy as np
import pandas as pd
from scipy import stats as scipy_stats
from sklearn.metrics import roc_auc_score

print("=" * 70)
print("CELL P6: A17 -- rolling_stats_roll7")
print("  Rolling assessment stats (window=7)")
print("=" * 70)

_pq_df, _pq_feat, _pq_wide = get_analysis_df('rolling_stats_roll7')
if _pq_df is None or len(_pq_feat) == 0:
    print("  SKIP: parquet not found or no features")
else:
    _n_cities = _pq_df['city'].nunique()
    _n_dam = (_pq_df[TARGET_COL] == 1).sum() if TARGET_COL in _pq_df.columns else 0
    _n_total = len(_pq_df)
    print(f"  Rows: {_n_total}, Cities: {_n_cities}, Damaged: {_n_dam}, Features: {len(_pq_feat)}")

    # ---- 1. AUC per feature (top 15) ----
    print(f"\n  [1/3] PER-FEATURE AUC (top 15 across all cities)")
    _auc_rows = []
    for _city in sorted(_pq_df['city'].unique()):
        _cdf = _pq_df[_pq_df['city'] == _city]
        if TARGET_COL not in _cdf.columns or _cdf[TARGET_COL].nunique() < 2 or len(_cdf) < MIN_SAMPLES:
            continue
        for _col in _pq_feat:
            _valid = _cdf[[_col, TARGET_COL]].dropna()
            if len(_valid) < 20 or _valid[TARGET_COL].nunique() < 2:
                continue
            _dv = _valid.loc[_valid[TARGET_COL] == 1, _col].values
            _cv = _valid.loc[_valid[TARGET_COL] == 0, _col].values
            if len(_dv) < 5 or len(_cv) < 5:
                continue
            try:
                _auc = roc_auc_score(_valid[TARGET_COL], _valid[_col])
                _auc_best = max(_auc, 1 - _auc)
                _u, _p = scipy_stats.mannwhitneyu(_dv, _cv, alternative='two-sided')
                _auc_rows.append({'city': _city, 'feature': _col, 'auc_best': _auc_best,
                                  'mann_whitney_p': _p, 'direction': 'higher=damaged' if _auc >= 0.5 else 'lower=damaged'})
            except Exception:
                continue

    if _auc_rows:
        _auc_df = pd.DataFrame(_auc_rows)
        _top = _auc_df.groupby('feature')['auc_best'].mean().nlargest(15)
        for _f, _a in _top.items():
            _n_sig = (_auc_df[_auc_df['feature'] == _f]['mann_whitney_p'] < 0.05).sum()
            _n_cit = len(_auc_df[_auc_df['feature'] == _f])
            print(f"    {_f:55s} AUC={_a:.3f} ({_n_cit} cities, {_n_sig} sig)")
        save_result(_auc_df, 'rolling_stats_roll7_auc', 'per_parquet')
        log_nb06_result('rolling_stats_roll7', 'P6', 'best_auc', float(_top.iloc[0]), feature=_top.index[0])
    else:
        print("    No features passed AUC filters")

    # ---- 2. SEPARABILITY (J-M distance, top 10) ----
    print(f"\n  [2/3] FEATURE SEPARABILITY (J-M distance, top 10)")
    _sep_rows = []
    for _city in sorted(_pq_df['city'].unique()):
        _cdf = _pq_df[_pq_df['city'] == _city]
        if TARGET_COL not in _cdf.columns or _cdf[TARGET_COL].nunique() < 2:
            continue
        for _col in _pq_feat:
            _d = _cdf.loc[_cdf[TARGET_COL] == 1, _col].dropna()
            _c = _cdf.loc[_cdf[TARGET_COL] == 0, _col].dropna()
            if len(_d) < 10 or len(_c) < 10:
                continue
            _mu1, _mu2 = _d.mean(), _c.mean()
            _v1, _v2 = _d.var(), _c.var()
            if _v1 <= 0 or _v2 <= 0:
                continue
            _bd = 0.25 * np.log(0.25 * (_v1/_v2 + _v2/_v1 + 2)) + 0.25 * ((_mu1-_mu2)**2) / (_v1+_v2)
            _jm = 2 * (1 - np.exp(-_bd))
            _sep_rows.append({'city': _city, 'feature': _col, 'jm_distance': _jm, 'bhattacharyya': _bd})

    if _sep_rows:
        _sep_df = pd.DataFrame(_sep_rows)
        _top_sep = _sep_df.groupby('feature')['jm_distance'].mean().nlargest(10)
        for _f, _jm in _top_sep.items():
            _interp = "excellent" if _jm > 1.9 else "good" if _jm > 1.5 else "moderate" if _jm > 1.0 else "poor"
            print(f"    {_f:55s} JM={_jm:.3f} ({_interp})")
        save_result(_sep_df, 'rolling_stats_roll7_separability', 'per_parquet')
    else:
        print("    No features passed separability filters")

    # ---- 3. VIF (top correlated pairs) ----
    print(f"\n  [3/3] VIF / COLLINEARITY CHECK")
    _vif_feats = [c for c in _pq_feat if _pq_df[c].notna().mean() > 0.5]
    if len(_vif_feats) > 3:
        _sample = _pq_df[_vif_feats].dropna()
        if len(_sample) > 100:
            if len(_vif_feats) > 50:
                # too many for full VIF, use Spearman top pairs
                _corr = _sample[_vif_feats[:50]].corr(method='spearman')
                _high_corr = []
                for _ii in range(len(_corr)):
                    for _jj in range(_ii+1, len(_corr)):
                        _r = abs(_corr.iloc[_ii, _jj])
                        if _r > 0.9:
                            _high_corr.append((_corr.index[_ii], _corr.columns[_jj], _r))
                _high_corr.sort(key=lambda x: -x[2])
                print(f"    Spearman |r| > 0.9: {len(_high_corr)} pairs (of {len(_vif_feats)} features)")
                for _a, _b, _r in _high_corr[:10]:
                    print(f"      {_a:40s} <-> {_b:40s} r={_r:.3f}")
            else:
                from statsmodels.stats.outliers_influence import variance_inflation_factor
                try:
                    _X = _sample[_vif_feats].values
                    _vif_vals = [variance_inflation_factor(_X, _k) for _k in range(len(_vif_feats))]
                    _vif_df = pd.DataFrame({'feature': _vif_feats, 'VIF': _vif_vals}).sort_values('VIF', ascending=False)
                    _high_vif = _vif_df[_vif_df['VIF'] > 10]
                    print(f"    VIF > 10: {len(_high_vif)}/{len(_vif_feats)} features")
                    for _, _row in _high_vif.head(10).iterrows():
                        print(f"      {_row['feature']:55s} VIF={_row['VIF']:.1f}")
                    save_result(_vif_df, 'rolling_stats_roll7_vif', 'per_parquet')
                except Exception as _e:
                    print(f"    VIF failed: {_e}")
        else:
            print(f"    Too few valid rows ({len(_sample)}) for VIF")
    else:
        print(f"    Too few features ({len(_vif_feats)}) for VIF")

    print(f"\n  A17 analysis complete")

del _pq_df
gc.collect()
print("  Memory freed")


CELL P6: A17 -- rolling_stats_roll7
  Rolling assessment stats (window=7)
  Loaded rolling_stats_roll7: 63243 rows, 38 features, 22.8 MB
  Rows: 63243, Cities: 21, Damaged: 8247, Features: 38

  [1/3] PER-FEATURE AUC (top 15 across all cities)
    s1__vv__baseline__mean                                  AUC=0.557 (21 cities, 10 sig)
    s1__vv__baseline__median                                AUC=0.555 (21 cities, 10 sig)
    s1__vv__baseline__min                                   AUC=0.553 (21 cities, 8 sig)
    s1__vv__baseline__max                                   AUC=0.549 (21 cities, 7 sig)
    s1__vh__baseline__min                                   AUC=0.547 (21 cities, 5 sig)
    s1__vh__baseline__mean                                  AUC=0.547 (21 cities, 5 sig)
    s1__vh__baseline__max                                   AUC=0.547 (21 cities, 6 sig)
    s1__vh__baseline__median                                AUC=0.545 (21 cities, 5 sig)
    s1__vv__baseline__skewness            

In [28]:
# @title CELL P7: A18 -- rolling_stats_roll13 (AUC + VIF + Separability)
# Parquet: rolling_stats_roll13 | Rolling assessment stats (window=13)
import gc
import numpy as np
import pandas as pd
from scipy import stats as scipy_stats
from sklearn.metrics import roc_auc_score

print("=" * 70)
print("CELL P7: A18 -- rolling_stats_roll13")
print("  Rolling assessment stats (window=13)")
print("=" * 70)

_pq_df, _pq_feat, _pq_wide = get_analysis_df('rolling_stats_roll13')
if _pq_df is None or len(_pq_feat) == 0:
    print("  SKIP: parquet not found or no features")
else:
    _n_cities = _pq_df['city'].nunique()
    _n_dam = (_pq_df[TARGET_COL] == 1).sum() if TARGET_COL in _pq_df.columns else 0
    _n_total = len(_pq_df)
    print(f"  Rows: {_n_total}, Cities: {_n_cities}, Damaged: {_n_dam}, Features: {len(_pq_feat)}")

    # ---- 1. AUC per feature (top 15) ----
    print(f"\n  [1/3] PER-FEATURE AUC (top 15 across all cities)")
    _auc_rows = []
    for _city in sorted(_pq_df['city'].unique()):
        _cdf = _pq_df[_pq_df['city'] == _city]
        if TARGET_COL not in _cdf.columns or _cdf[TARGET_COL].nunique() < 2 or len(_cdf) < MIN_SAMPLES:
            continue
        for _col in _pq_feat:
            _valid = _cdf[[_col, TARGET_COL]].dropna()
            if len(_valid) < 20 or _valid[TARGET_COL].nunique() < 2:
                continue
            _dv = _valid.loc[_valid[TARGET_COL] == 1, _col].values
            _cv = _valid.loc[_valid[TARGET_COL] == 0, _col].values
            if len(_dv) < 5 or len(_cv) < 5:
                continue
            try:
                _auc = roc_auc_score(_valid[TARGET_COL], _valid[_col])
                _auc_best = max(_auc, 1 - _auc)
                _u, _p = scipy_stats.mannwhitneyu(_dv, _cv, alternative='two-sided')
                _auc_rows.append({'city': _city, 'feature': _col, 'auc_best': _auc_best,
                                  'mann_whitney_p': _p, 'direction': 'higher=damaged' if _auc >= 0.5 else 'lower=damaged'})
            except Exception:
                continue

    if _auc_rows:
        _auc_df = pd.DataFrame(_auc_rows)
        _top = _auc_df.groupby('feature')['auc_best'].mean().nlargest(15)
        for _f, _a in _top.items():
            _n_sig = (_auc_df[_auc_df['feature'] == _f]['mann_whitney_p'] < 0.05).sum()
            _n_cit = len(_auc_df[_auc_df['feature'] == _f])
            print(f"    {_f:55s} AUC={_a:.3f} ({_n_cit} cities, {_n_sig} sig)")
        save_result(_auc_df, 'rolling_stats_roll13_auc', 'per_parquet')
        log_nb06_result('rolling_stats_roll13', 'P7', 'best_auc', float(_top.iloc[0]), feature=_top.index[0])
    else:
        print("    No features passed AUC filters")

    # ---- 2. SEPARABILITY (J-M distance, top 10) ----
    print(f"\n  [2/3] FEATURE SEPARABILITY (J-M distance, top 10)")
    _sep_rows = []
    for _city in sorted(_pq_df['city'].unique()):
        _cdf = _pq_df[_pq_df['city'] == _city]
        if TARGET_COL not in _cdf.columns or _cdf[TARGET_COL].nunique() < 2:
            continue
        for _col in _pq_feat:
            _d = _cdf.loc[_cdf[TARGET_COL] == 1, _col].dropna()
            _c = _cdf.loc[_cdf[TARGET_COL] == 0, _col].dropna()
            if len(_d) < 10 or len(_c) < 10:
                continue
            _mu1, _mu2 = _d.mean(), _c.mean()
            _v1, _v2 = _d.var(), _c.var()
            if _v1 <= 0 or _v2 <= 0:
                continue
            _bd = 0.25 * np.log(0.25 * (_v1/_v2 + _v2/_v1 + 2)) + 0.25 * ((_mu1-_mu2)**2) / (_v1+_v2)
            _jm = 2 * (1 - np.exp(-_bd))
            _sep_rows.append({'city': _city, 'feature': _col, 'jm_distance': _jm, 'bhattacharyya': _bd})

    if _sep_rows:
        _sep_df = pd.DataFrame(_sep_rows)
        _top_sep = _sep_df.groupby('feature')['jm_distance'].mean().nlargest(10)
        for _f, _jm in _top_sep.items():
            _interp = "excellent" if _jm > 1.9 else "good" if _jm > 1.5 else "moderate" if _jm > 1.0 else "poor"
            print(f"    {_f:55s} JM={_jm:.3f} ({_interp})")
        save_result(_sep_df, 'rolling_stats_roll13_separability', 'per_parquet')
    else:
        print("    No features passed separability filters")

    # ---- 3. VIF (top correlated pairs) ----
    print(f"\n  [3/3] VIF / COLLINEARITY CHECK")
    _vif_feats = [c for c in _pq_feat if _pq_df[c].notna().mean() > 0.5]
    if len(_vif_feats) > 3:
        _sample = _pq_df[_vif_feats].dropna()
        if len(_sample) > 100:
            if len(_vif_feats) > 50:
                # too many for full VIF, use Spearman top pairs
                _corr = _sample[_vif_feats[:50]].corr(method='spearman')
                _high_corr = []
                for _ii in range(len(_corr)):
                    for _jj in range(_ii+1, len(_corr)):
                        _r = abs(_corr.iloc[_ii, _jj])
                        if _r > 0.9:
                            _high_corr.append((_corr.index[_ii], _corr.columns[_jj], _r))
                _high_corr.sort(key=lambda x: -x[2])
                print(f"    Spearman |r| > 0.9: {len(_high_corr)} pairs (of {len(_vif_feats)} features)")
                for _a, _b, _r in _high_corr[:10]:
                    print(f"      {_a:40s} <-> {_b:40s} r={_r:.3f}")
            else:
                from statsmodels.stats.outliers_influence import variance_inflation_factor
                try:
                    _X = _sample[_vif_feats].values
                    _vif_vals = [variance_inflation_factor(_X, _k) for _k in range(len(_vif_feats))]
                    _vif_df = pd.DataFrame({'feature': _vif_feats, 'VIF': _vif_vals}).sort_values('VIF', ascending=False)
                    _high_vif = _vif_df[_vif_df['VIF'] > 10]
                    print(f"    VIF > 10: {len(_high_vif)}/{len(_vif_feats)} features")
                    for _, _row in _high_vif.head(10).iterrows():
                        print(f"      {_row['feature']:55s} VIF={_row['VIF']:.1f}")
                    save_result(_vif_df, 'rolling_stats_roll13_vif', 'per_parquet')
                except Exception as _e:
                    print(f"    VIF failed: {_e}")
        else:
            print(f"    Too few valid rows ({len(_sample)}) for VIF")
    else:
        print(f"    Too few features ({len(_vif_feats)}) for VIF")

    print(f"\n  A18 analysis complete")

del _pq_df
gc.collect()
print("  Memory freed")


CELL P7: A18 -- rolling_stats_roll13
  Rolling assessment stats (window=13)
  Loaded rolling_stats_roll13: 63243 rows, 22 features, 18.7 MB
  Rows: 63243, Cities: 21, Damaged: 8247, Features: 22

  [1/3] PER-FEATURE AUC (top 15 across all cities)
    s1__vv__baseline__mean                                  AUC=0.557 (21 cities, 10 sig)
    s1__vv__baseline__median                                AUC=0.555 (21 cities, 10 sig)
    s1__vv__baseline__min                                   AUC=0.553 (21 cities, 8 sig)
    s1__vv__baseline__max                                   AUC=0.549 (21 cities, 7 sig)
    s1__vh__baseline__min                                   AUC=0.547 (21 cities, 5 sig)
    s1__vh__baseline__mean                                  AUC=0.547 (21 cities, 5 sig)
    s1__vh__baseline__max                                   AUC=0.547 (21 cities, 6 sig)
    s1__vh__baseline__median                                AUC=0.545 (21 cities, 5 sig)
    s1__vv__baseline__skewness         

In [29]:
# @title CELL P8: F7 -- fusion_composite_cohdrop (AUC + VIF + Separability)
# Parquet: fusion_composite_cohdrop | MS composites + COH drop (optical + SAR)
import gc
import numpy as np
import pandas as pd
from scipy import stats as scipy_stats
from sklearn.metrics import roc_auc_score

print("=" * 70)
print("CELL P8: F7 -- fusion_composite_cohdrop")
print("  MS composites + COH drop (optical + SAR)")
print("=" * 70)

_pq_df, _pq_feat, _pq_wide = get_analysis_df('fusion_composite_cohdrop')
if _pq_df is None or len(_pq_feat) == 0:
    print("  SKIP: parquet not found or no features")
else:
    _n_cities = _pq_df['city'].nunique()
    _n_dam = (_pq_df[TARGET_COL] == 1).sum() if TARGET_COL in _pq_df.columns else 0
    _n_total = len(_pq_df)
    print(f"  Rows: {_n_total}, Cities: {_n_cities}, Damaged: {_n_dam}, Features: {len(_pq_feat)}")

    # ---- 1. AUC per feature (top 15) ----
    print(f"\n  [1/3] PER-FEATURE AUC (top 15 across all cities)")
    _auc_rows = []
    for _city in sorted(_pq_df['city'].unique()):
        _cdf = _pq_df[_pq_df['city'] == _city]
        if TARGET_COL not in _cdf.columns or _cdf[TARGET_COL].nunique() < 2 or len(_cdf) < MIN_SAMPLES:
            continue
        for _col in _pq_feat:
            _valid = _cdf[[_col, TARGET_COL]].dropna()
            if len(_valid) < 20 or _valid[TARGET_COL].nunique() < 2:
                continue
            _dv = _valid.loc[_valid[TARGET_COL] == 1, _col].values
            _cv = _valid.loc[_valid[TARGET_COL] == 0, _col].values
            if len(_dv) < 5 or len(_cv) < 5:
                continue
            try:
                _auc = roc_auc_score(_valid[TARGET_COL], _valid[_col])
                _auc_best = max(_auc, 1 - _auc)
                _u, _p = scipy_stats.mannwhitneyu(_dv, _cv, alternative='two-sided')
                _auc_rows.append({'city': _city, 'feature': _col, 'auc_best': _auc_best,
                                  'mann_whitney_p': _p, 'direction': 'higher=damaged' if _auc >= 0.5 else 'lower=damaged'})
            except Exception:
                continue

    if _auc_rows:
        _auc_df = pd.DataFrame(_auc_rows)
        _top = _auc_df.groupby('feature')['auc_best'].mean().nlargest(15)
        for _f, _a in _top.items():
            _n_sig = (_auc_df[_auc_df['feature'] == _f]['mann_whitney_p'] < 0.05).sum()
            _n_cit = len(_auc_df[_auc_df['feature'] == _f])
            print(f"    {_f:55s} AUC={_a:.3f} ({_n_cit} cities, {_n_sig} sig)")
        save_result(_auc_df, 'fusion_composite_cohdrop_auc', 'per_parquet')
        log_nb06_result('fusion_composite_cohdrop', 'P8', 'best_auc', float(_top.iloc[0]), feature=_top.index[0])
    else:
        print("    No features passed AUC filters")

    # ---- 2. SEPARABILITY (J-M distance, top 10) ----
    print(f"\n  [2/3] FEATURE SEPARABILITY (J-M distance, top 10)")
    _sep_rows = []
    for _city in sorted(_pq_df['city'].unique()):
        _cdf = _pq_df[_pq_df['city'] == _city]
        if TARGET_COL not in _cdf.columns or _cdf[TARGET_COL].nunique() < 2:
            continue
        for _col in _pq_feat:
            _d = _cdf.loc[_cdf[TARGET_COL] == 1, _col].dropna()
            _c = _cdf.loc[_cdf[TARGET_COL] == 0, _col].dropna()
            if len(_d) < 10 or len(_c) < 10:
                continue
            _mu1, _mu2 = _d.mean(), _c.mean()
            _v1, _v2 = _d.var(), _c.var()
            if _v1 <= 0 or _v2 <= 0:
                continue
            _bd = 0.25 * np.log(0.25 * (_v1/_v2 + _v2/_v1 + 2)) + 0.25 * ((_mu1-_mu2)**2) / (_v1+_v2)
            _jm = 2 * (1 - np.exp(-_bd))
            _sep_rows.append({'city': _city, 'feature': _col, 'jm_distance': _jm, 'bhattacharyya': _bd})

    if _sep_rows:
        _sep_df = pd.DataFrame(_sep_rows)
        _top_sep = _sep_df.groupby('feature')['jm_distance'].mean().nlargest(10)
        for _f, _jm in _top_sep.items():
            _interp = "excellent" if _jm > 1.9 else "good" if _jm > 1.5 else "moderate" if _jm > 1.0 else "poor"
            print(f"    {_f:55s} JM={_jm:.3f} ({_interp})")
        save_result(_sep_df, 'fusion_composite_cohdrop_separability', 'per_parquet')
    else:
        print("    No features passed separability filters")

    # ---- 3. VIF (top correlated pairs) ----
    print(f"\n  [3/3] VIF / COLLINEARITY CHECK")
    _vif_feats = [c for c in _pq_feat if _pq_df[c].notna().mean() > 0.5]
    if len(_vif_feats) > 3:
        _sample = _pq_df[_vif_feats].dropna()
        if len(_sample) > 100:
            if len(_vif_feats) > 50:
                # too many for full VIF, use Spearman top pairs
                _corr = _sample[_vif_feats[:50]].corr(method='spearman')
                _high_corr = []
                for _ii in range(len(_corr)):
                    for _jj in range(_ii+1, len(_corr)):
                        _r = abs(_corr.iloc[_ii, _jj])
                        if _r > 0.9:
                            _high_corr.append((_corr.index[_ii], _corr.columns[_jj], _r))
                _high_corr.sort(key=lambda x: -x[2])
                print(f"    Spearman |r| > 0.9: {len(_high_corr)} pairs (of {len(_vif_feats)} features)")
                for _a, _b, _r in _high_corr[:10]:
                    print(f"      {_a:40s} <-> {_b:40s} r={_r:.3f}")
            else:
                from statsmodels.stats.outliers_influence import variance_inflation_factor
                try:
                    _X = _sample[_vif_feats].values
                    _vif_vals = [variance_inflation_factor(_X, _k) for _k in range(len(_vif_feats))]
                    _vif_df = pd.DataFrame({'feature': _vif_feats, 'VIF': _vif_vals}).sort_values('VIF', ascending=False)
                    _high_vif = _vif_df[_vif_df['VIF'] > 10]
                    print(f"    VIF > 10: {len(_high_vif)}/{len(_vif_feats)} features")
                    for _, _row in _high_vif.head(10).iterrows():
                        print(f"      {_row['feature']:55s} VIF={_row['VIF']:.1f}")
                    save_result(_vif_df, 'fusion_composite_cohdrop_vif', 'per_parquet')
                except Exception as _e:
                    print(f"    VIF failed: {_e}")
        else:
            print(f"    Too few valid rows ({len(_sample)}) for VIF")
    else:
        print(f"    Too few features ({len(_vif_feats)}) for VIF")

    print(f"\n  F7 analysis complete")

del _pq_df
gc.collect()
print("  Memory freed")


CELL P8: F7 -- fusion_composite_cohdrop
  MS composites + COH drop (optical + SAR)
  Loaded fusion_composite_cohdrop: 51293 rows, 86 features, 29.2 MB
  Rows: 51293, Cities: 11, Damaged: 6867, Features: 86

  [1/3] PER-FEATURE AUC (top 15 across all cities)
    s2__composite__ndvi__prebattle_baseline                 AUC=0.628 (11 cities, 9 sig)
    s2__composite__ndvi__winter_baseline                    AUC=0.626 (11 cities, 9 sig)
    s2__composite__ndvi__post_winter_baseline               AUC=0.622 (11 cities, 7 sig)
    s2__composite__ndre__prebattle_baseline                 AUC=0.618 (11 cities, 10 sig)
    s2__composite__b02__post_winter_baseline                AUC=0.616 (11 cities, 7 sig)
    s2__composite__ndwi__prebattle_baseline                 AUC=0.616 (11 cities, 8 sig)
    s2__composite__ndwi__post_winter_baseline               AUC=0.615 (11 cities, 7 sig)
    s2__composite__ndre__post_winter_baseline               AUC=0.615 (11 cities, 8 sig)
    s2__composite__ndwi__wint

In [30]:
# @title CELL P9: F8 -- fusion_composite_blockstats (AUC + VIF + Separability)
# Parquet: fusion_composite_blockstats | MS composites + SAR block stats (full Dietrich)
import gc
import numpy as np
import pandas as pd
from scipy import stats as scipy_stats
from sklearn.metrics import roc_auc_score

print("=" * 70)
print("CELL P9: F8 -- fusion_composite_blockstats")
print("  MS composites + SAR block stats (full Dietrich)")
print("=" * 70)

_pq_df, _pq_feat, _pq_wide = get_analysis_df('fusion_composite_blockstats')
if _pq_df is None or len(_pq_feat) == 0:
    print("  SKIP: parquet not found or no features")
else:
    _n_cities = _pq_df['city'].nunique()
    _n_dam = (_pq_df[TARGET_COL] == 1).sum() if TARGET_COL in _pq_df.columns else 0
    _n_total = len(_pq_df)
    print(f"  Rows: {_n_total}, Cities: {_n_cities}, Damaged: {_n_dam}, Features: {len(_pq_feat)}")

    # ---- 1. AUC per feature (top 15) ----
    print(f"\n  [1/3] PER-FEATURE AUC (top 15 across all cities)")
    _auc_rows = []
    for _city in sorted(_pq_df['city'].unique()):
        _cdf = _pq_df[_pq_df['city'] == _city]
        if TARGET_COL not in _cdf.columns or _cdf[TARGET_COL].nunique() < 2 or len(_cdf) < MIN_SAMPLES:
            continue
        for _col in _pq_feat:
            _valid = _cdf[[_col, TARGET_COL]].dropna()
            if len(_valid) < 20 or _valid[TARGET_COL].nunique() < 2:
                continue
            _dv = _valid.loc[_valid[TARGET_COL] == 1, _col].values
            _cv = _valid.loc[_valid[TARGET_COL] == 0, _col].values
            if len(_dv) < 5 or len(_cv) < 5:
                continue
            try:
                _auc = roc_auc_score(_valid[TARGET_COL], _valid[_col])
                _auc_best = max(_auc, 1 - _auc)
                _u, _p = scipy_stats.mannwhitneyu(_dv, _cv, alternative='two-sided')
                _auc_rows.append({'city': _city, 'feature': _col, 'auc_best': _auc_best,
                                  'mann_whitney_p': _p, 'direction': 'higher=damaged' if _auc >= 0.5 else 'lower=damaged'})
            except Exception:
                continue

    if _auc_rows:
        _auc_df = pd.DataFrame(_auc_rows)
        _top = _auc_df.groupby('feature')['auc_best'].mean().nlargest(15)
        for _f, _a in _top.items():
            _n_sig = (_auc_df[_auc_df['feature'] == _f]['mann_whitney_p'] < 0.05).sum()
            _n_cit = len(_auc_df[_auc_df['feature'] == _f])
            print(f"    {_f:55s} AUC={_a:.3f} ({_n_cit} cities, {_n_sig} sig)")
        save_result(_auc_df, 'fusion_composite_blockstats_auc', 'per_parquet')
        log_nb06_result('fusion_composite_blockstats', 'P9', 'best_auc', float(_top.iloc[0]), feature=_top.index[0])
    else:
        print("    No features passed AUC filters")

    # ---- 2. SEPARABILITY (J-M distance, top 10) ----
    print(f"\n  [2/3] FEATURE SEPARABILITY (J-M distance, top 10)")
    _sep_rows = []
    for _city in sorted(_pq_df['city'].unique()):
        _cdf = _pq_df[_pq_df['city'] == _city]
        if TARGET_COL not in _cdf.columns or _cdf[TARGET_COL].nunique() < 2:
            continue
        for _col in _pq_feat:
            _d = _cdf.loc[_cdf[TARGET_COL] == 1, _col].dropna()
            _c = _cdf.loc[_cdf[TARGET_COL] == 0, _col].dropna()
            if len(_d) < 10 or len(_c) < 10:
                continue
            _mu1, _mu2 = _d.mean(), _c.mean()
            _v1, _v2 = _d.var(), _c.var()
            if _v1 <= 0 or _v2 <= 0:
                continue
            _bd = 0.25 * np.log(0.25 * (_v1/_v2 + _v2/_v1 + 2)) + 0.25 * ((_mu1-_mu2)**2) / (_v1+_v2)
            _jm = 2 * (1 - np.exp(-_bd))
            _sep_rows.append({'city': _city, 'feature': _col, 'jm_distance': _jm, 'bhattacharyya': _bd})

    if _sep_rows:
        _sep_df = pd.DataFrame(_sep_rows)
        _top_sep = _sep_df.groupby('feature')['jm_distance'].mean().nlargest(10)
        for _f, _jm in _top_sep.items():
            _interp = "excellent" if _jm > 1.9 else "good" if _jm > 1.5 else "moderate" if _jm > 1.0 else "poor"
            print(f"    {_f:55s} JM={_jm:.3f} ({_interp})")
        save_result(_sep_df, 'fusion_composite_blockstats_separability', 'per_parquet')
    else:
        print("    No features passed separability filters")

    # ---- 3. VIF (top correlated pairs) ----
    print(f"\n  [3/3] VIF / COLLINEARITY CHECK")
    _vif_feats = [c for c in _pq_feat if _pq_df[c].notna().mean() > 0.5]
    if len(_vif_feats) > 3:
        _sample = _pq_df[_vif_feats].dropna()
        if len(_sample) > 100:
            if len(_vif_feats) > 50:
                # too many for full VIF, use Spearman top pairs
                _corr = _sample[_vif_feats[:50]].corr(method='spearman')
                _high_corr = []
                for _ii in range(len(_corr)):
                    for _jj in range(_ii+1, len(_corr)):
                        _r = abs(_corr.iloc[_ii, _jj])
                        if _r > 0.9:
                            _high_corr.append((_corr.index[_ii], _corr.columns[_jj], _r))
                _high_corr.sort(key=lambda x: -x[2])
                print(f"    Spearman |r| > 0.9: {len(_high_corr)} pairs (of {len(_vif_feats)} features)")
                for _a, _b, _r in _high_corr[:10]:
                    print(f"      {_a:40s} <-> {_b:40s} r={_r:.3f}")
            else:
                from statsmodels.stats.outliers_influence import variance_inflation_factor
                try:
                    _X = _sample[_vif_feats].values
                    _vif_vals = [variance_inflation_factor(_X, _k) for _k in range(len(_vif_feats))]
                    _vif_df = pd.DataFrame({'feature': _vif_feats, 'VIF': _vif_vals}).sort_values('VIF', ascending=False)
                    _high_vif = _vif_df[_vif_df['VIF'] > 10]
                    print(f"    VIF > 10: {len(_high_vif)}/{len(_vif_feats)} features")
                    for _, _row in _high_vif.head(10).iterrows():
                        print(f"      {_row['feature']:55s} VIF={_row['VIF']:.1f}")
                    save_result(_vif_df, 'fusion_composite_blockstats_vif', 'per_parquet')
                except Exception as _e:
                    print(f"    VIF failed: {_e}")
        else:
            print(f"    Too few valid rows ({len(_sample)}) for VIF")
    else:
        print(f"    Too few features ({len(_vif_feats)}) for VIF")

    print(f"\n  F8 analysis complete")

del _pq_df
gc.collect()
print("  Memory freed")


CELL P9: F8 -- fusion_composite_blockstats
  MS composites + SAR block stats (full Dietrich)
  Loaded fusion_composite_blockstats: 62043 rows, 409 features, 115.4 MB
  Rows: 62043, Cities: 19, Damaged: 8080, Features: 409

  [1/3] PER-FEATURE AUC (top 15 across all cities)
    s2__composite__ndvi__post_winter_baseline               AUC=0.622 (19 cities, 11 sig)
    s2__composite__ndvi__prebattle_baseline                 AUC=0.617 (19 cities, 13 sig)
    s2__composite__ndvi__winter_baseline                    AUC=0.616 (19 cities, 13 sig)
    s2__composite__ndwi__post_winter_baseline               AUC=0.613 (19 cities, 11 sig)
    s2__composite__ndwi__prebattle_baseline                 AUC=0.611 (19 cities, 12 sig)
    s2__composite__ndwi__winter_baseline                    AUC=0.610 (19 cities, 12 sig)
    s2__composite__b02__post_winter_baseline                AUC=0.606 (19 cities, 10 sig)
    s2__composite__ndre__post_winter_baseline               AUC=0.605 (19 cities, 12 sig)
    s1

## LONG PARQUETS (one row per building x date)

In [31]:
# @title CELL P10: A1 -- scene_ms (temporal AUC trajectory)
# Parquet: scene_ms | Per-scene optical bands (9 bands + aux)
import gc
import numpy as np
import pandas as pd
from scipy import stats as scipy_stats
from sklearn.metrics import roc_auc_score

print("=" * 70)
print("CELL P10: A1 -- scene_ms")
print("  Per-scene optical bands (9 bands + aux)")
print("=" * 70)

_pq_df, _pq_feat, _pq_wide = get_analysis_df('scene_ms')
if _pq_df is None or len(_pq_feat) == 0:
    print("  SKIP: parquet not found or no features")
else:
    _n_cities = _pq_df['city'].nunique()
    _n_dam = (_pq_df[TARGET_COL] == 1).sum() if TARGET_COL in _pq_df.columns else 0
    _n_obs = len(_pq_df)
    _n_dates = _pq_df['date'].nunique() if 'date' in _pq_df.columns else 0
    print(f"  Observations: {_n_obs}, Cities: {_n_cities}, Dates: {_n_dates}, Damaged obs: {_n_dam}, Features: {len(_pq_feat)}")

    # ---- 1. PER-PERIOD AUC (pre vs cross vs post) ----
    print(f"\n  [1/2] PER-PERIOD AUC (best feature per period)")
    if 'period_label' in _pq_df.columns:
        for _period in ['prebattle', 'crossbattle', 'postbattle']:
            _pdf = _pq_df[_pq_df['period_label'] == _period]
            if TARGET_COL not in _pdf.columns or _pdf[TARGET_COL].nunique() < 2 or len(_pdf) < 50:
                print(f"    {_period:15s}: insufficient data ({len(_pdf)} obs)")
                continue
            _best_auc = 0
            _best_feat = ''
            for _col in _pq_feat:
                _valid = _pdf[[_col, TARGET_COL]].dropna()
                if len(_valid) < 50 or _valid[TARGET_COL].nunique() < 2:
                    continue
                try:
                    _auc = max(roc_auc_score(_valid[TARGET_COL], _valid[_col]),
                              1 - roc_auc_score(_valid[TARGET_COL], _valid[_col]))
                    if _auc > _best_auc:
                        _best_auc = _auc
                        _best_feat = _col
                except Exception:
                    continue
            print(f"    {_period:15s}: best AUC={_best_auc:.3f} ({_best_feat}), {len(_pdf)} obs")
            log_nb06_result('scene_ms', 'P10', f'best_auc_{_period}', _best_auc, feature=_best_feat)
    else:
        print("    No period_label column")

    # ---- 2. TEMPORAL AUC TRAJECTORY (per timestep) ----
    print(f"\n  [2/2] TEMPORAL AUC TRAJECTORY (per timestep, best feature)")
    if 'timestep' in _pq_df.columns and TARGET_COL in _pq_df.columns:
        _ts_range = sorted(_pq_df['timestep'].unique())
        _ts_auc = []
        for _ts in _ts_range:
            _tdf = _pq_df[_pq_df['timestep'] == _ts]
            if _tdf[TARGET_COL].nunique() < 2 or len(_tdf) < 50:
                continue
            _best = 0
            _bf = ''
            for _col in _pq_feat[:10]:  # top 10 features only for speed
                _valid = _tdf[[_col, TARGET_COL]].dropna()
                if len(_valid) < 50 or _valid[TARGET_COL].nunique() < 2:
                    continue
                try:
                    _a = max(roc_auc_score(_valid[TARGET_COL], _valid[_col]),
                             1 - roc_auc_score(_valid[TARGET_COL], _valid[_col]))
                    if _a > _best:
                        _best = _a
                        _bf = _col
                except Exception:
                    continue
            if _best > 0:
                _ts_auc.append({'timestep': _ts, 'best_auc': _best, 'best_feature': _bf, 'n_obs': len(_tdf)})

        if _ts_auc:
            _ts_df = pd.DataFrame(_ts_auc)
            print(f"    Timestep range: [{int(_ts_df['timestep'].min()):+d}..{int(_ts_df['timestep'].max()):+d}]")
            print(f"    Pre-battle AUC (t<0):  {_ts_df[_ts_df['timestep']<0]['best_auc'].mean():.3f}" if len(_ts_df[_ts_df['timestep']<0]) > 0 else "    Pre-battle: no data")
            print(f"    Post-battle AUC (t>=0): {_ts_df[_ts_df['timestep']>=0]['best_auc'].mean():.3f}" if len(_ts_df[_ts_df['timestep']>=0]) > 0 else "    Post-battle: no data")

            # plot
            import matplotlib.pyplot as plt
            fig, ax = plt.subplots(figsize=(12, 5))
            ax.plot(_ts_df['timestep'], _ts_df['best_auc'], 'o-', color='#2196F3', markersize=4)
            ax.axhline(0.5, color='red', linestyle='--', alpha=0.5, label='Random')
            ax.axvline(0, color='black', linestyle='-', alpha=0.3, label='Battle start')
            ax.set_xlabel('Timestep (t=0 = battle start)')
            ax.set_ylabel('Best single-feature AUC')
            ax.set_title('A1 scene_ms: Temporal AUC trajectory')
            ax.legend()
            plt.tight_layout()
            save_fig(fig, 'scene_ms_temporal_auc', 'per_parquet')
            save_result(_ts_df, 'scene_ms_temporal_auc', 'per_parquet')
            print(f"    Saved temporal AUC plot + CSV")
        else:
            print("    No timesteps with sufficient data")
    else:
        print("    No timestep column")

    print(f"\n  A1 analysis complete")

del _pq_df
gc.collect()
print("  Memory freed")


CELL P10: A1 -- scene_ms
  Per-scene optical bands (9 bands + aux)
  Loaded scene_ms: 695385 rows, 12 features, 202.0 MB
  Observations: 695385, Cities: 19, Dates: 147, Damaged obs: 90670, Features: 12

  [1/2] PER-PERIOD AUC (best feature per period)
    prebattle      : best AUC=0.568 (s2__b02), 286002 obs
    crossbattle    : best AUC=0.646 (s2__b02), 152157 obs
    postbattle     : best AUC=0.643 (s2__b02), 257226 obs

  [2/2] TEMPORAL AUC TRAJECTORY (per timestep, best feature)
    Timestep range: [-7..+73]
    Pre-battle AUC (t<0):  0.579
    Post-battle AUC (t>=0): 0.719
    saved -> scene_ms_temporal_auc.csv (81 rows)
    Saved temporal AUC plot + CSV

  A1 analysis complete
  Memory freed


In [32]:
# @title CELL P11: A2 -- scene_card (temporal AUC trajectory)
# Parquet: scene_card | Per-scene CARD backscatter (VV + VH)
import gc
import numpy as np
import pandas as pd
from scipy import stats as scipy_stats
from sklearn.metrics import roc_auc_score

print("=" * 70)
print("CELL P11: A2 -- scene_card")
print("  Per-scene CARD backscatter (VV + VH)")
print("=" * 70)

_pq_df, _pq_feat, _pq_wide = get_analysis_df('scene_card')
if _pq_df is None or len(_pq_feat) == 0:
    print("  SKIP: parquet not found or no features")
else:
    _n_cities = _pq_df['city'].nunique()
    _n_dam = (_pq_df[TARGET_COL] == 1).sum() if TARGET_COL in _pq_df.columns else 0
    _n_obs = len(_pq_df)
    _n_dates = _pq_df['date'].nunique() if 'date' in _pq_df.columns else 0
    print(f"  Observations: {_n_obs}, Cities: {_n_cities}, Dates: {_n_dates}, Damaged obs: {_n_dam}, Features: {len(_pq_feat)}")

    # ---- 1. PER-PERIOD AUC (pre vs cross vs post) ----
    print(f"\n  [1/2] PER-PERIOD AUC (best feature per period)")
    if 'period_label' in _pq_df.columns:
        for _period in ['prebattle', 'crossbattle', 'postbattle']:
            _pdf = _pq_df[_pq_df['period_label'] == _period]
            if TARGET_COL not in _pdf.columns or _pdf[TARGET_COL].nunique() < 2 or len(_pdf) < 50:
                print(f"    {_period:15s}: insufficient data ({len(_pdf)} obs)")
                continue
            _best_auc = 0
            _best_feat = ''
            for _col in _pq_feat:
                _valid = _pdf[[_col, TARGET_COL]].dropna()
                if len(_valid) < 50 or _valid[TARGET_COL].nunique() < 2:
                    continue
                try:
                    _auc = max(roc_auc_score(_valid[TARGET_COL], _valid[_col]),
                              1 - roc_auc_score(_valid[TARGET_COL], _valid[_col]))
                    if _auc > _best_auc:
                        _best_auc = _auc
                        _best_feat = _col
                except Exception:
                    continue
            print(f"    {_period:15s}: best AUC={_best_auc:.3f} ({_best_feat}), {len(_pdf)} obs")
            log_nb06_result('scene_card', 'P11', f'best_auc_{_period}', _best_auc, feature=_best_feat)
    else:
        print("    No period_label column")

    # ---- 2. TEMPORAL AUC TRAJECTORY (per timestep) ----
    print(f"\n  [2/2] TEMPORAL AUC TRAJECTORY (per timestep, best feature)")
    if 'timestep' in _pq_df.columns and TARGET_COL in _pq_df.columns:
        _ts_range = sorted(_pq_df['timestep'].unique())
        _ts_auc = []
        for _ts in _ts_range:
            _tdf = _pq_df[_pq_df['timestep'] == _ts]
            if _tdf[TARGET_COL].nunique() < 2 or len(_tdf) < 50:
                continue
            _best = 0
            _bf = ''
            for _col in _pq_feat[:10]:  # top 10 features only for speed
                _valid = _tdf[[_col, TARGET_COL]].dropna()
                if len(_valid) < 50 or _valid[TARGET_COL].nunique() < 2:
                    continue
                try:
                    _a = max(roc_auc_score(_valid[TARGET_COL], _valid[_col]),
                             1 - roc_auc_score(_valid[TARGET_COL], _valid[_col]))
                    if _a > _best:
                        _best = _a
                        _bf = _col
                except Exception:
                    continue
            if _best > 0:
                _ts_auc.append({'timestep': _ts, 'best_auc': _best, 'best_feature': _bf, 'n_obs': len(_tdf)})

        if _ts_auc:
            _ts_df = pd.DataFrame(_ts_auc)
            print(f"    Timestep range: [{int(_ts_df['timestep'].min()):+d}..{int(_ts_df['timestep'].max()):+d}]")
            print(f"    Pre-battle AUC (t<0):  {_ts_df[_ts_df['timestep']<0]['best_auc'].mean():.3f}" if len(_ts_df[_ts_df['timestep']<0]) > 0 else "    Pre-battle: no data")
            print(f"    Post-battle AUC (t>=0): {_ts_df[_ts_df['timestep']>=0]['best_auc'].mean():.3f}" if len(_ts_df[_ts_df['timestep']>=0]) > 0 else "    Post-battle: no data")

            # plot
            import matplotlib.pyplot as plt
            fig, ax = plt.subplots(figsize=(12, 5))
            ax.plot(_ts_df['timestep'], _ts_df['best_auc'], 'o-', color='#2196F3', markersize=4)
            ax.axhline(0.5, color='red', linestyle='--', alpha=0.5, label='Random')
            ax.axvline(0, color='black', linestyle='-', alpha=0.3, label='Battle start')
            ax.set_xlabel('Timestep (t=0 = battle start)')
            ax.set_ylabel('Best single-feature AUC')
            ax.set_title('A2 scene_card: Temporal AUC trajectory')
            ax.legend()
            plt.tight_layout()
            save_fig(fig, 'scene_card_temporal_auc', 'per_parquet')
            save_result(_ts_df, 'scene_card_temporal_auc', 'per_parquet')
            print(f"    Saved temporal AUC plot + CSV")
        else:
            print("    No timesteps with sufficient data")
    else:
        print("    No timestep column")

    print(f"\n  A2 analysis complete")

del _pq_df
gc.collect()
print("  Memory freed")


CELL P11: A2 -- scene_card
  Per-scene CARD backscatter (VV + VH)
  Loaded scene_card: 834134 rows, 2 features, 208.6 MB
  Observations: 834134, Cities: 21, Dates: 213, Damaged obs: 106458, Features: 2

  [1/2] PER-PERIOD AUC (best feature per period)
    prebattle      : best AUC=0.540 (s1__vh), 312608 obs
    crossbattle    : best AUC=0.548 (s1__vh), 395400 obs
    postbattle     : best AUC=0.532 (s1__vh), 126126 obs

  [2/2] TEMPORAL AUC TRAJECTORY (per timestep, best feature)
    Timestep range: [-5..+119]
    Pre-battle AUC (t<0):  0.540
    Post-battle AUC (t>=0): 0.588
    saved -> scene_card_temporal_auc.csv (125 rows)
    Saved temporal AUC plot + CSV

  A2 analysis complete
  Memory freed


In [33]:
# @title CELL P12: A3 -- scene_coh (temporal AUC trajectory)
# Parquet: scene_coh | Per-scene InSAR coherence
import gc
import numpy as np
import pandas as pd
from scipy import stats as scipy_stats
from sklearn.metrics import roc_auc_score

print("=" * 70)
print("CELL P12: A3 -- scene_coh")
print("  Per-scene InSAR coherence")
print("=" * 70)

_pq_df, _pq_feat, _pq_wide = get_analysis_df('scene_coh')
if _pq_df is None or len(_pq_feat) == 0:
    print("  SKIP: parquet not found or no features")
else:
    _n_cities = _pq_df['city'].nunique()
    _n_dam = (_pq_df[TARGET_COL] == 1).sum() if TARGET_COL in _pq_df.columns else 0
    _n_obs = len(_pq_df)
    _n_dates = _pq_df['date'].nunique() if 'date' in _pq_df.columns else 0
    print(f"  Observations: {_n_obs}, Cities: {_n_cities}, Dates: {_n_dates}, Damaged obs: {_n_dam}, Features: {len(_pq_feat)}")

    # ---- 1. PER-PERIOD AUC (pre vs cross vs post) ----
    print(f"\n  [1/2] PER-PERIOD AUC (best feature per period)")
    if 'period_label' in _pq_df.columns:
        for _period in ['prebattle', 'crossbattle', 'postbattle']:
            _pdf = _pq_df[_pq_df['period_label'] == _period]
            if TARGET_COL not in _pdf.columns or _pdf[TARGET_COL].nunique() < 2 or len(_pdf) < 50:
                print(f"    {_period:15s}: insufficient data ({len(_pdf)} obs)")
                continue
            _best_auc = 0
            _best_feat = ''
            for _col in _pq_feat:
                _valid = _pdf[[_col, TARGET_COL]].dropna()
                if len(_valid) < 50 or _valid[TARGET_COL].nunique() < 2:
                    continue
                try:
                    _auc = max(roc_auc_score(_valid[TARGET_COL], _valid[_col]),
                              1 - roc_auc_score(_valid[TARGET_COL], _valid[_col]))
                    if _auc > _best_auc:
                        _best_auc = _auc
                        _best_feat = _col
                except Exception:
                    continue
            print(f"    {_period:15s}: best AUC={_best_auc:.3f} ({_best_feat}), {len(_pdf)} obs")
            log_nb06_result('scene_coh', 'P12', f'best_auc_{_period}', _best_auc, feature=_best_feat)
    else:
        print("    No period_label column")

    # ---- 2. TEMPORAL AUC TRAJECTORY (per timestep) ----
    print(f"\n  [2/2] TEMPORAL AUC TRAJECTORY (per timestep, best feature)")
    if 'timestep' in _pq_df.columns and TARGET_COL in _pq_df.columns:
        _ts_range = sorted(_pq_df['timestep'].unique())
        _ts_auc = []
        for _ts in _ts_range:
            _tdf = _pq_df[_pq_df['timestep'] == _ts]
            if _tdf[TARGET_COL].nunique() < 2 or len(_tdf) < 50:
                continue
            _best = 0
            _bf = ''
            for _col in _pq_feat[:10]:  # top 10 features only for speed
                _valid = _tdf[[_col, TARGET_COL]].dropna()
                if len(_valid) < 50 or _valid[TARGET_COL].nunique() < 2:
                    continue
                try:
                    _a = max(roc_auc_score(_valid[TARGET_COL], _valid[_col]),
                             1 - roc_auc_score(_valid[TARGET_COL], _valid[_col]))
                    if _a > _best:
                        _best = _a
                        _bf = _col
                except Exception:
                    continue
            if _best > 0:
                _ts_auc.append({'timestep': _ts, 'best_auc': _best, 'best_feature': _bf, 'n_obs': len(_tdf)})

        if _ts_auc:
            _ts_df = pd.DataFrame(_ts_auc)
            print(f"    Timestep range: [{int(_ts_df['timestep'].min()):+d}..{int(_ts_df['timestep'].max()):+d}]")
            print(f"    Pre-battle AUC (t<0):  {_ts_df[_ts_df['timestep']<0]['best_auc'].mean():.3f}" if len(_ts_df[_ts_df['timestep']<0]) > 0 else "    Pre-battle: no data")
            print(f"    Post-battle AUC (t>=0): {_ts_df[_ts_df['timestep']>=0]['best_auc'].mean():.3f}" if len(_ts_df[_ts_df['timestep']>=0]) > 0 else "    Post-battle: no data")

            # plot
            import matplotlib.pyplot as plt
            fig, ax = plt.subplots(figsize=(12, 5))
            ax.plot(_ts_df['timestep'], _ts_df['best_auc'], 'o-', color='#2196F3', markersize=4)
            ax.axhline(0.5, color='red', linestyle='--', alpha=0.5, label='Random')
            ax.axvline(0, color='black', linestyle='-', alpha=0.3, label='Battle start')
            ax.set_xlabel('Timestep (t=0 = battle start)')
            ax.set_ylabel('Best single-feature AUC')
            ax.set_title('A3 scene_coh: Temporal AUC trajectory')
            ax.legend()
            plt.tight_layout()
            save_fig(fig, 'scene_coh_temporal_auc', 'per_parquet')
            save_result(_ts_df, 'scene_coh_temporal_auc', 'per_parquet')
            print(f"    Saved temporal AUC plot + CSV")
        else:
            print("    No timesteps with sufficient data")
    else:
        print("    No timestep column")

    print(f"\n  A3 analysis complete")

del _pq_df
gc.collect()
print("  Memory freed")


CELL P12: A3 -- scene_coh
  Per-scene InSAR coherence
  Loaded scene_coh: 468995 rows, 3 features, 133.8 MB
  Observations: 468995, Cities: 18, Dates: 116, Damaged obs: 57711, Features: 3

  [1/2] PER-PERIOD AUC (best feature per period)
    prebattle      : best AUC=0.523 (s1__coh_vh), 202048 obs
    crossbattle    : best AUC=0.521 (s1__coh_vh), 186669 obs
    postbattle     : best AUC=0.540 (s1__coh_vh), 80278 obs

  [2/2] TEMPORAL AUC TRAJECTORY (per timestep, best feature)
    Timestep range: [-4..+58]
    Pre-battle AUC (t<0):  0.524
    Post-battle AUC (t>=0): 0.546
    saved -> scene_coh_temporal_auc.csv (63 rows)
    Saved temporal AUC plot + CSV

  A3 analysis complete
  Memory freed


In [34]:
# @title CELL P13: A5 -- scene_indices (temporal AUC trajectory)
# Parquet: scene_indices | Per-scene spectral indices (NDVI, BSI, etc.)
import gc
import numpy as np
import pandas as pd
from scipy import stats as scipy_stats
from sklearn.metrics import roc_auc_score

print("=" * 70)
print("CELL P13: A5 -- scene_indices")
print("  Per-scene spectral indices (NDVI, BSI, etc.)")
print("=" * 70)

_pq_df, _pq_feat, _pq_wide = get_analysis_df('scene_indices')
if _pq_df is None or len(_pq_feat) == 0:
    print("  SKIP: parquet not found or no features")
else:
    _n_cities = _pq_df['city'].nunique()
    _n_dam = (_pq_df[TARGET_COL] == 1).sum() if TARGET_COL in _pq_df.columns else 0
    _n_obs = len(_pq_df)
    _n_dates = _pq_df['date'].nunique() if 'date' in _pq_df.columns else 0
    print(f"  Observations: {_n_obs}, Cities: {_n_cities}, Dates: {_n_dates}, Damaged obs: {_n_dam}, Features: {len(_pq_feat)}")

    # ---- 1. PER-PERIOD AUC (pre vs cross vs post) ----
    print(f"\n  [1/2] PER-PERIOD AUC (best feature per period)")
    if 'period_label' in _pq_df.columns:
        for _period in ['prebattle', 'crossbattle', 'postbattle']:
            _pdf = _pq_df[_pq_df['period_label'] == _period]
            if TARGET_COL not in _pdf.columns or _pdf[TARGET_COL].nunique() < 2 or len(_pdf) < 50:
                print(f"    {_period:15s}: insufficient data ({len(_pdf)} obs)")
                continue
            _best_auc = 0
            _best_feat = ''
            for _col in _pq_feat:
                _valid = _pdf[[_col, TARGET_COL]].dropna()
                if len(_valid) < 50 or _valid[TARGET_COL].nunique() < 2:
                    continue
                try:
                    _auc = max(roc_auc_score(_valid[TARGET_COL], _valid[_col]),
                              1 - roc_auc_score(_valid[TARGET_COL], _valid[_col]))
                    if _auc > _best_auc:
                        _best_auc = _auc
                        _best_feat = _col
                except Exception:
                    continue
            print(f"    {_period:15s}: best AUC={_best_auc:.3f} ({_best_feat}), {len(_pdf)} obs")
            log_nb06_result('scene_indices', 'P13', f'best_auc_{_period}', _best_auc, feature=_best_feat)
    else:
        print("    No period_label column")

    # ---- 2. TEMPORAL AUC TRAJECTORY (per timestep) ----
    print(f"\n  [2/2] TEMPORAL AUC TRAJECTORY (per timestep, best feature)")
    if 'timestep' in _pq_df.columns and TARGET_COL in _pq_df.columns:
        _ts_range = sorted(_pq_df['timestep'].unique())
        _ts_auc = []
        for _ts in _ts_range:
            _tdf = _pq_df[_pq_df['timestep'] == _ts]
            if _tdf[TARGET_COL].nunique() < 2 or len(_tdf) < 50:
                continue
            _best = 0
            _bf = ''
            for _col in _pq_feat[:10]:  # top 10 features only for speed
                _valid = _tdf[[_col, TARGET_COL]].dropna()
                if len(_valid) < 50 or _valid[TARGET_COL].nunique() < 2:
                    continue
                try:
                    _a = max(roc_auc_score(_valid[TARGET_COL], _valid[_col]),
                             1 - roc_auc_score(_valid[TARGET_COL], _valid[_col]))
                    if _a > _best:
                        _best = _a
                        _bf = _col
                except Exception:
                    continue
            if _best > 0:
                _ts_auc.append({'timestep': _ts, 'best_auc': _best, 'best_feature': _bf, 'n_obs': len(_tdf)})

        if _ts_auc:
            _ts_df = pd.DataFrame(_ts_auc)
            print(f"    Timestep range: [{int(_ts_df['timestep'].min()):+d}..{int(_ts_df['timestep'].max()):+d}]")
            print(f"    Pre-battle AUC (t<0):  {_ts_df[_ts_df['timestep']<0]['best_auc'].mean():.3f}" if len(_ts_df[_ts_df['timestep']<0]) > 0 else "    Pre-battle: no data")
            print(f"    Post-battle AUC (t>=0): {_ts_df[_ts_df['timestep']>=0]['best_auc'].mean():.3f}" if len(_ts_df[_ts_df['timestep']>=0]) > 0 else "    Post-battle: no data")

            # plot
            import matplotlib.pyplot as plt
            fig, ax = plt.subplots(figsize=(12, 5))
            ax.plot(_ts_df['timestep'], _ts_df['best_auc'], 'o-', color='#2196F3', markersize=4)
            ax.axhline(0.5, color='red', linestyle='--', alpha=0.5, label='Random')
            ax.axvline(0, color='black', linestyle='-', alpha=0.3, label='Battle start')
            ax.set_xlabel('Timestep (t=0 = battle start)')
            ax.set_ylabel('Best single-feature AUC')
            ax.set_title('A5 scene_indices: Temporal AUC trajectory')
            ax.legend()
            plt.tight_layout()
            save_fig(fig, 'scene_indices_temporal_auc', 'per_parquet')
            save_result(_ts_df, 'scene_indices_temporal_auc', 'per_parquet')
            print(f"    Saved temporal AUC plot + CSV")
        else:
            print("    No timesteps with sufficient data")
    else:
        print("    No timestep column")

    print(f"\n  A5 analysis complete")

del _pq_df
gc.collect()
print("  Memory freed")


CELL P13: A5 -- scene_indices
  Per-scene spectral indices (NDVI, BSI, etc.)
  Loaded scene_indices: 794205 rows, 9 features, 221.2 MB
  Observations: 794205, Cities: 19, Dates: 161, Damaged obs: 102688, Features: 9

  [1/2] PER-PERIOD AUC (best feature per period)
    prebattle      : best AUC=0.603 (s2__ndvi), 286002 obs
    crossbattle    : best AUC=0.657 (s2__ndvi), 152157 obs
    postbattle     : best AUC=0.641 (s2__mndwi), 356046 obs

  [2/2] TEMPORAL AUC TRAJECTORY (per timestep, best feature)
    Timestep range: [-7..+73]
    Pre-battle AUC (t<0):  0.609
    Post-battle AUC (t>=0): 0.769
    saved -> scene_indices_temporal_auc.csv (81 rows)
    Saved temporal AUC plot + CSV

  A5 analysis complete
  Memory freed


In [35]:
# @title CELL P15: A7 -- rolling_coh (temporal AUC trajectory)
# Parquet: rolling_coh | Rolling coherence (all windows)
import gc
import numpy as np
import pandas as pd
from scipy import stats as scipy_stats
from sklearn.metrics import roc_auc_score

print("=" * 70)
print("CELL P15: A7 -- rolling_coh")
print("  Rolling coherence (all windows)")
print("=" * 70)

_pq_df, _pq_feat, _pq_wide = get_analysis_df('rolling_coh')
if _pq_df is None or len(_pq_feat) == 0:
    print("  SKIP: parquet not found or no features")
else:
    _n_cities = _pq_df['city'].nunique()
    _n_dam = (_pq_df[TARGET_COL] == 1).sum() if TARGET_COL in _pq_df.columns else 0
    _n_obs = len(_pq_df)
    _n_dates = _pq_df['date'].nunique() if 'date' in _pq_df.columns else 0
    print(f"  Observations: {_n_obs}, Cities: {_n_cities}, Dates: {_n_dates}, Damaged obs: {_n_dam}, Features: {len(_pq_feat)}")

    # ---- 1. PER-PERIOD AUC (pre vs cross vs post) ----
    print(f"\n  [1/2] PER-PERIOD AUC (best feature per period)")
    if 'period_label' in _pq_df.columns:
        for _period in ['prebattle', 'crossbattle', 'postbattle']:
            _pdf = _pq_df[_pq_df['period_label'] == _period]
            if TARGET_COL not in _pdf.columns or _pdf[TARGET_COL].nunique() < 2 or len(_pdf) < 50:
                print(f"    {_period:15s}: insufficient data ({len(_pdf)} obs)")
                continue
            _best_auc = 0
            _best_feat = ''
            for _col in _pq_feat:
                _valid = _pdf[[_col, TARGET_COL]].dropna()
                if len(_valid) < 50 or _valid[TARGET_COL].nunique() < 2:
                    continue
                try:
                    _auc = max(roc_auc_score(_valid[TARGET_COL], _valid[_col]),
                              1 - roc_auc_score(_valid[TARGET_COL], _valid[_col]))
                    if _auc > _best_auc:
                        _best_auc = _auc
                        _best_feat = _col
                except Exception:
                    continue
            print(f"    {_period:15s}: best AUC={_best_auc:.3f} ({_best_feat}), {len(_pdf)} obs")
            log_nb06_result('rolling_coh', 'P15', f'best_auc_{_period}', _best_auc, feature=_best_feat)
    else:
        print("    No period_label column")

    # ---- 2. TEMPORAL AUC TRAJECTORY (per timestep) ----
    print(f"\n  [2/2] TEMPORAL AUC TRAJECTORY (per timestep, best feature)")
    if 'timestep' in _pq_df.columns and TARGET_COL in _pq_df.columns:
        _ts_range = sorted(_pq_df['timestep'].unique())
        _ts_auc = []
        for _ts in _ts_range:
            _tdf = _pq_df[_pq_df['timestep'] == _ts]
            if _tdf[TARGET_COL].nunique() < 2 or len(_tdf) < 50:
                continue
            _best = 0
            _bf = ''
            for _col in _pq_feat[:10]:  # top 10 features only for speed
                _valid = _tdf[[_col, TARGET_COL]].dropna()
                if len(_valid) < 50 or _valid[TARGET_COL].nunique() < 2:
                    continue
                try:
                    _a = max(roc_auc_score(_valid[TARGET_COL], _valid[_col]),
                             1 - roc_auc_score(_valid[TARGET_COL], _valid[_col]))
                    if _a > _best:
                        _best = _a
                        _bf = _col
                except Exception:
                    continue
            if _best > 0:
                _ts_auc.append({'timestep': _ts, 'best_auc': _best, 'best_feature': _bf, 'n_obs': len(_tdf)})

        if _ts_auc:
            _ts_df = pd.DataFrame(_ts_auc)
            print(f"    Timestep range: [{int(_ts_df['timestep'].min()):+d}..{int(_ts_df['timestep'].max()):+d}]")
            print(f"    Pre-battle AUC (t<0):  {_ts_df[_ts_df['timestep']<0]['best_auc'].mean():.3f}" if len(_ts_df[_ts_df['timestep']<0]) > 0 else "    Pre-battle: no data")
            print(f"    Post-battle AUC (t>=0): {_ts_df[_ts_df['timestep']>=0]['best_auc'].mean():.3f}" if len(_ts_df[_ts_df['timestep']>=0]) > 0 else "    Post-battle: no data")

            # plot
            import matplotlib.pyplot as plt
            fig, ax = plt.subplots(figsize=(12, 5))
            ax.plot(_ts_df['timestep'], _ts_df['best_auc'], 'o-', color='#2196F3', markersize=4)
            ax.axhline(0.5, color='red', linestyle='--', alpha=0.5, label='Random')
            ax.axvline(0, color='black', linestyle='-', alpha=0.3, label='Battle start')
            ax.set_xlabel('Timestep (t=0 = battle start)')
            ax.set_ylabel('Best single-feature AUC')
            ax.set_title('A7 rolling_coh: Temporal AUC trajectory')
            ax.legend()
            plt.tight_layout()
            save_fig(fig, 'rolling_coh_temporal_auc', 'per_parquet')
            save_result(_ts_df, 'rolling_coh_temporal_auc', 'per_parquet')
            print(f"    Saved temporal AUC plot + CSV")
        else:
            print("    No timesteps with sufficient data")
    else:
        print("    No timestep column")

    print(f"\n  A7 analysis complete")

del _pq_df
gc.collect()
print("  Memory freed")


CELL P15: A7 -- rolling_coh
  Rolling coherence (all windows)
  Loaded rolling_coh: 350960 rows, 3 features, 88.7 MB
  Observations: 350960, Cities: 16, Dates: 100, Damaged obs: 42356, Features: 3

  [1/2] PER-PERIOD AUC (best feature per period)
    prebattle      : best AUC=0.512 (s1__coh_vv__roll3), 191276 obs
    crossbattle    : best AUC=0.538 (s1__coh_vv__roll13), 159684 obs
    postbattle     : insufficient data (0 obs)

  [2/2] TEMPORAL AUC TRAJECTORY (per timestep, best feature)
    Timestep range: [-4..+56]
    Pre-battle AUC (t<0):  0.512
    Post-battle AUC (t>=0): 0.541
    saved -> rolling_coh_temporal_auc.csv (61 rows)
    Saved temporal AUC plot + CSV

  A7 analysis complete
  Memory freed


In [36]:
# @title CELL P16: A8 -- rolling_card (temporal AUC trajectory)
# Parquet: rolling_card | Rolling CARD (all windows)
import gc
import numpy as np
import pandas as pd
from scipy import stats as scipy_stats
from sklearn.metrics import roc_auc_score

print("=" * 70)
print("CELL P16: A8 -- rolling_card")
print("  Rolling CARD (all windows)")
print("=" * 70)

_pq_df, _pq_feat, _pq_wide = get_analysis_df('rolling_card')
if _pq_df is None or len(_pq_feat) == 0:
    print("  SKIP: parquet not found or no features")
else:
    _n_cities = _pq_df['city'].nunique()
    _n_dam = (_pq_df[TARGET_COL] == 1).sum() if TARGET_COL in _pq_df.columns else 0
    _n_obs = len(_pq_df)
    _n_dates = _pq_df['date'].nunique() if 'date' in _pq_df.columns else 0
    print(f"  Observations: {_n_obs}, Cities: {_n_cities}, Dates: {_n_dates}, Damaged obs: {_n_dam}, Features: {len(_pq_feat)}")

    # ---- 1. PER-PERIOD AUC (pre vs cross vs post) ----
    print(f"\n  [1/2] PER-PERIOD AUC (best feature per period)")
    if 'period_label' in _pq_df.columns:
        for _period in ['prebattle', 'crossbattle', 'postbattle']:
            _pdf = _pq_df[_pq_df['period_label'] == _period]
            if TARGET_COL not in _pdf.columns or _pdf[TARGET_COL].nunique() < 2 or len(_pdf) < 50:
                print(f"    {_period:15s}: insufficient data ({len(_pdf)} obs)")
                continue
            _best_auc = 0
            _best_feat = ''
            for _col in _pq_feat:
                _valid = _pdf[[_col, TARGET_COL]].dropna()
                if len(_valid) < 50 or _valid[TARGET_COL].nunique() < 2:
                    continue
                try:
                    _auc = max(roc_auc_score(_valid[TARGET_COL], _valid[_col]),
                              1 - roc_auc_score(_valid[TARGET_COL], _valid[_col]))
                    if _auc > _best_auc:
                        _best_auc = _auc
                        _best_feat = _col
                except Exception:
                    continue
            print(f"    {_period:15s}: best AUC={_best_auc:.3f} ({_best_feat}), {len(_pdf)} obs")
            log_nb06_result('rolling_card', 'P16', f'best_auc_{_period}', _best_auc, feature=_best_feat)
    else:
        print("    No period_label column")

    # ---- 2. TEMPORAL AUC TRAJECTORY (per timestep) ----
    print(f"\n  [2/2] TEMPORAL AUC TRAJECTORY (per timestep, best feature)")
    if 'timestep' in _pq_df.columns and TARGET_COL in _pq_df.columns:
        _ts_range = sorted(_pq_df['timestep'].unique())
        _ts_auc = []
        for _ts in _ts_range:
            _tdf = _pq_df[_pq_df['timestep'] == _ts]
            if _tdf[TARGET_COL].nunique() < 2 or len(_tdf) < 50:
                continue
            _best = 0
            _bf = ''
            for _col in _pq_feat[:10]:  # top 10 features only for speed
                _valid = _tdf[[_col, TARGET_COL]].dropna()
                if len(_valid) < 50 or _valid[TARGET_COL].nunique() < 2:
                    continue
                try:
                    _a = max(roc_auc_score(_valid[TARGET_COL], _valid[_col]),
                             1 - roc_auc_score(_valid[TARGET_COL], _valid[_col]))
                    if _a > _best:
                        _best = _a
                        _bf = _col
                except Exception:
                    continue
            if _best > 0:
                _ts_auc.append({'timestep': _ts, 'best_auc': _best, 'best_feature': _bf, 'n_obs': len(_tdf)})

        if _ts_auc:
            _ts_df = pd.DataFrame(_ts_auc)
            print(f"    Timestep range: [{int(_ts_df['timestep'].min()):+d}..{int(_ts_df['timestep'].max()):+d}]")
            print(f"    Pre-battle AUC (t<0):  {_ts_df[_ts_df['timestep']<0]['best_auc'].mean():.3f}" if len(_ts_df[_ts_df['timestep']<0]) > 0 else "    Pre-battle: no data")
            print(f"    Post-battle AUC (t>=0): {_ts_df[_ts_df['timestep']>=0]['best_auc'].mean():.3f}" if len(_ts_df[_ts_df['timestep']>=0]) > 0 else "    Post-battle: no data")

            # plot
            import matplotlib.pyplot as plt
            fig, ax = plt.subplots(figsize=(12, 5))
            ax.plot(_ts_df['timestep'], _ts_df['best_auc'], 'o-', color='#2196F3', markersize=4)
            ax.axhline(0.5, color='red', linestyle='--', alpha=0.5, label='Random')
            ax.axvline(0, color='black', linestyle='-', alpha=0.3, label='Battle start')
            ax.set_xlabel('Timestep (t=0 = battle start)')
            ax.set_ylabel('Best single-feature AUC')
            ax.set_title('A8 rolling_card: Temporal AUC trajectory')
            ax.legend()
            plt.tight_layout()
            save_fig(fig, 'rolling_card_temporal_auc', 'per_parquet')
            save_result(_ts_df, 'rolling_card_temporal_auc', 'per_parquet')
            print(f"    Saved temporal AUC plot + CSV")
        else:
            print("    No timesteps with sufficient data")
    else:
        print("    No timestep column")

    print(f"\n  A8 analysis complete")

del _pq_df
gc.collect()
print("  Memory freed")


CELL P16: A8 -- rolling_card
  Rolling CARD (all windows)
  Loaded rolling_card: 711255 rows, 6 features, 189.3 MB
  Observations: 711255, Cities: 21, Dates: 198, Damaged obs: 90472, Features: 6

  [1/2] PER-PERIOD AUC (best feature per period)
    prebattle      : best AUC=0.542 (s1__vh__roll3), 250200 obs
    crossbattle    : best AUC=0.566 (s1__vh__roll7), 397992 obs
    postbattle     : best AUC=0.536 (s1__vh__roll3), 63063 obs

  [2/2] TEMPORAL AUC TRAJECTORY (per timestep, best feature)
    Timestep range: [-4..+118]
    Pre-battle AUC (t<0):  0.542
    Post-battle AUC (t>=0): 0.554
    saved -> rolling_card_temporal_auc.csv (123 rows)
    Saved temporal AUC plot + CSV

  A8 analysis complete
  Memory freed


In [37]:
# @title CELL P18: F1 -- fusion_ms_card (temporal AUC trajectory)
# Parquet: scene_ms | MS optical + CARD backscatter (per-scene)
import gc
import numpy as np
import pandas as pd
from scipy import stats as scipy_stats
from sklearn.metrics import roc_auc_score

print("=" * 70)
print("CELL P18: F1 -- fusion_ms_card")
print("  MS optical + CARD backscatter (per-scene)")
print("=" * 70)

_pq_df, _pq_feat, _pq_wide = get_analysis_df('fusion_ms_card')
if _pq_df is None or len(_pq_feat) == 0:
    print("  SKIP: parquet not found or no features")
else:
    _n_cities = _pq_df['city'].nunique()
    _n_dam = (_pq_df[TARGET_COL] == 1).sum() if TARGET_COL in _pq_df.columns else 0
    _n_obs = len(_pq_df)
    _n_dates = _pq_df['date'].nunique() if 'date' in _pq_df.columns else 0
    print(f"  Observations: {_n_obs}, Cities: {_n_cities}, Dates: {_n_dates}, Damaged obs: {_n_dam}, Features: {len(_pq_feat)}")

    # ---- 1. PER-PERIOD AUC (pre vs cross vs post) ----
    print(f"\n  [1/2] PER-PERIOD AUC (best feature per period)")
    if 'period_label' in _pq_df.columns:
        for _period in ['prebattle', 'crossbattle', 'postbattle']:
            _pdf = _pq_df[_pq_df['period_label'] == _period]
            if TARGET_COL not in _pdf.columns or _pdf[TARGET_COL].nunique() < 2 or len(_pdf) < 50:
                print(f"    {_period:15s}: insufficient data ({len(_pdf)} obs)")
                continue
            _best_auc = 0
            _best_feat = ''
            for _col in _pq_feat:
                _valid = _pdf[[_col, TARGET_COL]].dropna()
                if len(_valid) < 50 or _valid[TARGET_COL].nunique() < 2:
                    continue
                try:
                    _auc = max(roc_auc_score(_valid[TARGET_COL], _valid[_col]),
                              1 - roc_auc_score(_valid[TARGET_COL], _valid[_col]))
                    if _auc > _best_auc:
                        _best_auc = _auc
                        _best_feat = _col
                except Exception:
                    continue
            print(f"    {_period:15s}: best AUC={_best_auc:.3f} ({_best_feat}), {len(_pdf)} obs")
            log_nb06_result('fusion_ms_card', 'P10', f'best_auc_{_period}', _best_auc, feature=_best_feat)
    else:
        print("    No period_label column")

    # ---- 2. TEMPORAL AUC TRAJECTORY (per timestep) ----
    print(f"\n  [2/2] TEMPORAL AUC TRAJECTORY (per timestep, best feature)")
    if 'timestep' in _pq_df.columns and TARGET_COL in _pq_df.columns:
        _ts_range = sorted(_pq_df['timestep'].unique())
        _ts_auc = []
        for _ts in _ts_range:
            _tdf = _pq_df[_pq_df['timestep'] == _ts]
            if _tdf[TARGET_COL].nunique() < 2 or len(_tdf) < 50:
                continue
            _best = 0
            _bf = ''
            for _col in _pq_feat[:10]:  # top 10 features only for speed
                _valid = _tdf[[_col, TARGET_COL]].dropna()
                if len(_valid) < 50 or _valid[TARGET_COL].nunique() < 2:
                    continue
                try:
                    _a = max(roc_auc_score(_valid[TARGET_COL], _valid[_col]),
                             1 - roc_auc_score(_valid[TARGET_COL], _valid[_col]))
                    if _a > _best:
                        _best = _a
                        _bf = _col
                except Exception:
                    continue
            if _best > 0:
                _ts_auc.append({'timestep': _ts, 'best_auc': _best, 'best_feature': _bf, 'n_obs': len(_tdf)})

        if _ts_auc:
            _ts_df = pd.DataFrame(_ts_auc)
            print(f"    Timestep range: [{int(_ts_df['timestep'].min()):+d}..{int(_ts_df['timestep'].max()):+d}]")
            print(f"    Pre-battle AUC (t<0):  {_ts_df[_ts_df['timestep']<0]['best_auc'].mean():.3f}" if len(_ts_df[_ts_df['timestep']<0]) > 0 else "    Pre-battle: no data")
            print(f"    Post-battle AUC (t>=0): {_ts_df[_ts_df['timestep']>=0]['best_auc'].mean():.3f}" if len(_ts_df[_ts_df['timestep']>=0]) > 0 else "    Post-battle: no data")

            # plot
            import matplotlib.pyplot as plt
            fig, ax = plt.subplots(figsize=(12, 5))
            ax.plot(_ts_df['timestep'], _ts_df['best_auc'], 'o-', color='#2196F3', markersize=4)
            ax.axhline(0.5, color='red', linestyle='--', alpha=0.5, label='Random')
            ax.axvline(0, color='black', linestyle='-', alpha=0.3, label='Battle start')
            ax.set_xlabel('Timestep (t=0 = battle start)')
            ax.set_ylabel('Best single-feature AUC')
            ax.set_title('F1 fusion_ms_card: Temporal AUC trajectory')
            ax.legend()
            plt.tight_layout()
            save_fig(fig, 'fusion_ms_card_temporal_auc', 'per_parquet')
            save_result(_ts_df, 'fusion_ms_card_temporal_auc', 'per_parquet')
            print(f"    Saved temporal AUC plot + CSV")
        else:
            print("    No timesteps with sufficient data")
    else:
        print("    No timestep column")

    print(f"\n  F1 analysis complete")

del _pq_df
gc.collect()
print("  Memory freed")


CELL P18: F1 -- fusion_ms_card
  MS optical + CARD backscatter (per-scene)
  Loaded fusion_ms_card: 1493681 rows, 14 features, 461.6 MB
  Observations: 1493681, Cities: 19, Dates: 338, Damaged obs: 192523, Features: 14

  [1/2] PER-PERIOD AUC (best feature per period)
    prebattle      : best AUC=0.568 (s2__b02), 286002 obs
    crossbattle    : best AUC=0.646 (s2__b02), 152157 obs
    postbattle     : best AUC=0.643 (s2__b02), 257226 obs

  [2/2] TEMPORAL AUC TRAJECTORY (per timestep, best feature)
    Timestep range: [-7..+73]
    Pre-battle AUC (t<0):  0.579
    Post-battle AUC (t>=0): 0.719
    saved -> fusion_ms_card_temporal_auc.csv (81 rows)
    Saved temporal AUC plot + CSV

  F1 analysis complete
  Memory freed


In [38]:
# @title CELL P19: F2 -- fusion_ms_card_cohdrop (temporal AUC trajectory)
# Parquet: scene_ms | MS + CARD + COH drop (per-scene)
import gc
import numpy as np
import pandas as pd
from scipy import stats as scipy_stats
from sklearn.metrics import roc_auc_score

print("=" * 70)
print("CELL P19: F2 -- fusion_ms_card_cohdrop")
print("  MS + CARD + COH drop (per-scene)")
print("=" * 70)

_pq_df, _pq_feat, _pq_wide = get_analysis_df('fusion_ms_card_cohdrop')
if _pq_df is None or len(_pq_feat) == 0:
    print("  SKIP: parquet not found or no features")
else:
    _n_cities = _pq_df['city'].nunique()
    _n_dam = (_pq_df[TARGET_COL] == 1).sum() if TARGET_COL in _pq_df.columns else 0
    _n_obs = len(_pq_df)
    _n_dates = _pq_df['date'].nunique() if 'date' in _pq_df.columns else 0
    print(f"  Observations: {_n_obs}, Cities: {_n_cities}, Dates: {_n_dates}, Damaged obs: {_n_dam}, Features: {len(_pq_feat)}")

    # ---- 1. PER-PERIOD AUC (pre vs cross vs post) ----
    print(f"\n  [1/2] PER-PERIOD AUC (best feature per period)")
    if 'period_label' in _pq_df.columns:
        for _period in ['prebattle', 'crossbattle', 'postbattle']:
            _pdf = _pq_df[_pq_df['period_label'] == _period]
            if TARGET_COL not in _pdf.columns or _pdf[TARGET_COL].nunique() < 2 or len(_pdf) < 50:
                print(f"    {_period:15s}: insufficient data ({len(_pdf)} obs)")
                continue
            _best_auc = 0
            _best_feat = ''
            for _col in _pq_feat:
                _valid = _pdf[[_col, TARGET_COL]].dropna()
                if len(_valid) < 50 or _valid[TARGET_COL].nunique() < 2:
                    continue
                try:
                    _auc = max(roc_auc_score(_valid[TARGET_COL], _valid[_col]),
                              1 - roc_auc_score(_valid[TARGET_COL], _valid[_col]))
                    if _auc > _best_auc:
                        _best_auc = _auc
                        _best_feat = _col
                except Exception:
                    continue
            print(f"    {_period:15s}: best AUC={_best_auc:.3f} ({_best_feat}), {len(_pdf)} obs")
            log_nb06_result('fusion_ms_card_cohdrop', 'P10', f'best_auc_{_period}', _best_auc, feature=_best_feat)
    else:
        print("    No period_label column")

    # ---- 2. TEMPORAL AUC TRAJECTORY (per timestep) ----
    print(f"\n  [2/2] TEMPORAL AUC TRAJECTORY (per timestep, best feature)")
    if 'timestep' in _pq_df.columns and TARGET_COL in _pq_df.columns:
        _ts_range = sorted(_pq_df['timestep'].unique())
        _ts_auc = []
        for _ts in _ts_range:
            _tdf = _pq_df[_pq_df['timestep'] == _ts]
            if _tdf[TARGET_COL].nunique() < 2 or len(_tdf) < 50:
                continue
            _best = 0
            _bf = ''
            for _col in _pq_feat[:10]:  # top 10 features only for speed
                _valid = _tdf[[_col, TARGET_COL]].dropna()
                if len(_valid) < 50 or _valid[TARGET_COL].nunique() < 2:
                    continue
                try:
                    _a = max(roc_auc_score(_valid[TARGET_COL], _valid[_col]),
                             1 - roc_auc_score(_valid[TARGET_COL], _valid[_col]))
                    if _a > _best:
                        _best = _a
                        _bf = _col
                except Exception:
                    continue
            if _best > 0:
                _ts_auc.append({'timestep': _ts, 'best_auc': _best, 'best_feature': _bf, 'n_obs': len(_tdf)})

        if _ts_auc:
            _ts_df = pd.DataFrame(_ts_auc)
            print(f"    Timestep range: [{int(_ts_df['timestep'].min()):+d}..{int(_ts_df['timestep'].max()):+d}]")
            print(f"    Pre-battle AUC (t<0):  {_ts_df[_ts_df['timestep']<0]['best_auc'].mean():.3f}" if len(_ts_df[_ts_df['timestep']<0]) > 0 else "    Pre-battle: no data")
            print(f"    Post-battle AUC (t>=0): {_ts_df[_ts_df['timestep']>=0]['best_auc'].mean():.3f}" if len(_ts_df[_ts_df['timestep']>=0]) > 0 else "    Post-battle: no data")

            # plot
            import matplotlib.pyplot as plt
            fig, ax = plt.subplots(figsize=(12, 5))
            ax.plot(_ts_df['timestep'], _ts_df['best_auc'], 'o-', color='#2196F3', markersize=4)
            ax.axhline(0.5, color='red', linestyle='--', alpha=0.5, label='Random')
            ax.axvline(0, color='black', linestyle='-', alpha=0.3, label='Battle start')
            ax.set_xlabel('Timestep (t=0 = battle start)')
            ax.set_ylabel('Best single-feature AUC')
            ax.set_title('F2 fusion_ms_card_cohdrop: Temporal AUC trajectory')
            ax.legend()
            plt.tight_layout()
            save_fig(fig, 'fusion_ms_card_cohdrop_temporal_auc', 'per_parquet')
            save_result(_ts_df, 'fusion_ms_card_cohdrop_temporal_auc', 'per_parquet')
            print(f"    Saved temporal AUC plot + CSV")
        else:
            print("    No timesteps with sufficient data")
    else:
        print("    No timestep column")

    print(f"\n  F2 analysis complete")

del _pq_df
gc.collect()
print("  Memory freed")


CELL P19: F2 -- fusion_ms_card_cohdrop
  MS + CARD + COH drop (per-scene)
  Loaded fusion_ms_card_cohdrop: 1195249 rows, 21 features, 413.4 MB
  Observations: 1195249, Cities: 11, Dates: 170, Damaged obs: 157597, Features: 21

  [1/2] PER-PERIOD AUC (best feature per period)
    prebattle      : best AUC=0.571 (s2__b02), 232230 obs
    crossbattle    : best AUC=0.667 (s2__b02), 111076 obs
    postbattle     : best AUC=0.657 (s2__visibility), 226410 obs

  [2/2] TEMPORAL AUC TRAJECTORY (per timestep, best feature)
    Timestep range: [-7..+17]
    Pre-battle AUC (t<0):  0.584
    Post-battle AUC (t>=0): 0.724
    saved -> fusion_ms_card_cohdrop_temporal_auc.csv (25 rows)
    Saved temporal AUC plot + CSV

  F2 analysis complete
  Memory freed


In [39]:
# @title CELL P20: F3 -- fusion_card_cohdrop (temporal AUC trajectory)
# Parquet: scene_ms | CARD + COH drop (SAR-only fusion, per-scene)
import gc
import numpy as np
import pandas as pd
from scipy import stats as scipy_stats
from sklearn.metrics import roc_auc_score

print("=" * 70)
print("CELL P20: F3 -- fusion_card_cohdrop")
print("  CARD + COH drop (SAR-only fusion, per-scene)")
print("=" * 70)

_pq_df, _pq_feat, _pq_wide = get_analysis_df('fusion_card_cohdrop')
if _pq_df is None or len(_pq_feat) == 0:
    print("  SKIP: parquet not found or no features")
else:
    _n_cities = _pq_df['city'].nunique()
    _n_dam = (_pq_df[TARGET_COL] == 1).sum() if TARGET_COL in _pq_df.columns else 0
    _n_obs = len(_pq_df)
    _n_dates = _pq_df['date'].nunique() if 'date' in _pq_df.columns else 0
    print(f"  Observations: {_n_obs}, Cities: {_n_cities}, Dates: {_n_dates}, Damaged obs: {_n_dam}, Features: {len(_pq_feat)}")

    # ---- 1. PER-PERIOD AUC (pre vs cross vs post) ----
    print(f"\n  [1/2] PER-PERIOD AUC (best feature per period)")
    if 'period_label' in _pq_df.columns:
        for _period in ['prebattle', 'crossbattle', 'postbattle']:
            _pdf = _pq_df[_pq_df['period_label'] == _period]
            if TARGET_COL not in _pdf.columns or _pdf[TARGET_COL].nunique() < 2 or len(_pdf) < 50:
                print(f"    {_period:15s}: insufficient data ({len(_pdf)} obs)")
                continue
            _best_auc = 0
            _best_feat = ''
            for _col in _pq_feat:
                _valid = _pdf[[_col, TARGET_COL]].dropna()
                if len(_valid) < 50 or _valid[TARGET_COL].nunique() < 2:
                    continue
                try:
                    _auc = max(roc_auc_score(_valid[TARGET_COL], _valid[_col]),
                              1 - roc_auc_score(_valid[TARGET_COL], _valid[_col]))
                    if _auc > _best_auc:
                        _best_auc = _auc
                        _best_feat = _col
                except Exception:
                    continue
            print(f"    {_period:15s}: best AUC={_best_auc:.3f} ({_best_feat}), {len(_pdf)} obs")
            log_nb06_result('fusion_card_cohdrop', 'P10', f'best_auc_{_period}', _best_auc, feature=_best_feat)
    else:
        print("    No period_label column")

    # ---- 2. TEMPORAL AUC TRAJECTORY (per timestep) ----
    print(f"\n  [2/2] TEMPORAL AUC TRAJECTORY (per timestep, best feature)")
    if 'timestep' in _pq_df.columns and TARGET_COL in _pq_df.columns:
        _ts_range = sorted(_pq_df['timestep'].unique())
        _ts_auc = []
        for _ts in _ts_range:
            _tdf = _pq_df[_pq_df['timestep'] == _ts]
            if _tdf[TARGET_COL].nunique() < 2 or len(_tdf) < 50:
                continue
            _best = 0
            _bf = ''
            for _col in _pq_feat[:10]:  # top 10 features only for speed
                _valid = _tdf[[_col, TARGET_COL]].dropna()
                if len(_valid) < 50 or _valid[TARGET_COL].nunique() < 2:
                    continue
                try:
                    _a = max(roc_auc_score(_valid[TARGET_COL], _valid[_col]),
                             1 - roc_auc_score(_valid[TARGET_COL], _valid[_col]))
                    if _a > _best:
                        _best = _a
                        _bf = _col
                except Exception:
                    continue
            if _best > 0:
                _ts_auc.append({'timestep': _ts, 'best_auc': _best, 'best_feature': _bf, 'n_obs': len(_tdf)})

        if _ts_auc:
            _ts_df = pd.DataFrame(_ts_auc)
            print(f"    Timestep range: [{int(_ts_df['timestep'].min()):+d}..{int(_ts_df['timestep'].max()):+d}]")
            print(f"    Pre-battle AUC (t<0):  {_ts_df[_ts_df['timestep']<0]['best_auc'].mean():.3f}" if len(_ts_df[_ts_df['timestep']<0]) > 0 else "    Pre-battle: no data")
            print(f"    Post-battle AUC (t>=0): {_ts_df[_ts_df['timestep']>=0]['best_auc'].mean():.3f}" if len(_ts_df[_ts_df['timestep']>=0]) > 0 else "    Post-battle: no data")

            # plot
            import matplotlib.pyplot as plt
            fig, ax = plt.subplots(figsize=(12, 5))
            ax.plot(_ts_df['timestep'], _ts_df['best_auc'], 'o-', color='#2196F3', markersize=4)
            ax.axhline(0.5, color='red', linestyle='--', alpha=0.5, label='Random')
            ax.axvline(0, color='black', linestyle='-', alpha=0.3, label='Battle start')
            ax.set_xlabel('Timestep (t=0 = battle start)')
            ax.set_ylabel('Best single-feature AUC')
            ax.set_title('F3 fusion_card_cohdrop: Temporal AUC trajectory')
            ax.legend()
            plt.tight_layout()
            save_fig(fig, 'fusion_card_cohdrop_temporal_auc', 'per_parquet')
            save_result(_ts_df, 'fusion_card_cohdrop_temporal_auc', 'per_parquet')
            print(f"    Saved temporal AUC plot + CSV")
        else:
            print("    No timesteps with sufficient data")
    else:
        print("    No timestep column")

    print(f"\n  F3 analysis complete")

del _pq_df
gc.collect()
print("  Memory freed")


CELL P20: F3 -- fusion_card_cohdrop
  CARD + COH drop (SAR-only fusion, per-scene)
  Loaded fusion_card_cohdrop: 658108 rows, 9 features, 188.7 MB
  Observations: 658108, Cities: 12, Dates: 120, Damaged obs: 85708, Features: 9

  [1/2] PER-PERIOD AUC (best feature per period)
    prebattle      : best AUC=0.545 (s1__vh), 259000 obs
    crossbattle    : best AUC=0.560 (s1__coh__scenes_observed), 295508 obs
    postbattle     : best AUC=0.540 (s1__vh), 103600 obs

  [2/2] TEMPORAL AUC TRAJECTORY (per timestep, best feature)
    Timestep range: [-5..+59]
    Pre-battle AUC (t<0):  0.546
    Post-battle AUC (t>=0): 0.609
    saved -> fusion_card_cohdrop_temporal_auc.csv (65 rows)
    Saved temporal AUC plot + CSV

  F3 analysis complete
  Memory freed


In [40]:
# @title CELL P21: F4 -- fusion_ms_cohdrop (temporal AUC trajectory)
# Parquet: scene_ms | MS + COH drop (per-scene)
import gc
import numpy as np
import pandas as pd
from scipy import stats as scipy_stats
from sklearn.metrics import roc_auc_score

print("=" * 70)
print("CELL P21: F4 -- fusion_ms_cohdrop")
print("  MS + COH drop (per-scene)")
print("=" * 70)

_pq_df, _pq_feat, _pq_wide = get_analysis_df('fusion_ms_cohdrop')
if _pq_df is None or len(_pq_feat) == 0:
    print("  SKIP: parquet not found or no features")
else:
    _n_cities = _pq_df['city'].nunique()
    _n_dam = (_pq_df[TARGET_COL] == 1).sum() if TARGET_COL in _pq_df.columns else 0
    _n_obs = len(_pq_df)
    _n_dates = _pq_df['date'].nunique() if 'date' in _pq_df.columns else 0
    print(f"  Observations: {_n_obs}, Cities: {_n_cities}, Dates: {_n_dates}, Damaged obs: {_n_dam}, Features: {len(_pq_feat)}")

    # ---- 1. PER-PERIOD AUC (pre vs cross vs post) ----
    print(f"\n  [1/2] PER-PERIOD AUC (best feature per period)")
    if 'period_label' in _pq_df.columns:
        for _period in ['prebattle', 'crossbattle', 'postbattle']:
            _pdf = _pq_df[_pq_df['period_label'] == _period]
            if TARGET_COL not in _pdf.columns or _pdf[TARGET_COL].nunique() < 2 or len(_pdf) < 50:
                print(f"    {_period:15s}: insufficient data ({len(_pdf)} obs)")
                continue
            _best_auc = 0
            _best_feat = ''
            for _col in _pq_feat:
                _valid = _pdf[[_col, TARGET_COL]].dropna()
                if len(_valid) < 50 or _valid[TARGET_COL].nunique() < 2:
                    continue
                try:
                    _auc = max(roc_auc_score(_valid[TARGET_COL], _valid[_col]),
                              1 - roc_auc_score(_valid[TARGET_COL], _valid[_col]))
                    if _auc > _best_auc:
                        _best_auc = _auc
                        _best_feat = _col
                except Exception:
                    continue
            print(f"    {_period:15s}: best AUC={_best_auc:.3f} ({_best_feat}), {len(_pdf)} obs")
            log_nb06_result('fusion_ms_cohdrop', 'P10', f'best_auc_{_period}', _best_auc, feature=_best_feat)
    else:
        print("    No period_label column")

    # ---- 2. TEMPORAL AUC TRAJECTORY (per timestep) ----
    print(f"\n  [2/2] TEMPORAL AUC TRAJECTORY (per timestep, best feature)")
    if 'timestep' in _pq_df.columns and TARGET_COL in _pq_df.columns:
        _ts_range = sorted(_pq_df['timestep'].unique())
        _ts_auc = []
        for _ts in _ts_range:
            _tdf = _pq_df[_pq_df['timestep'] == _ts]
            if _tdf[TARGET_COL].nunique() < 2 or len(_tdf) < 50:
                continue
            _best = 0
            _bf = ''
            for _col in _pq_feat[:10]:  # top 10 features only for speed
                _valid = _tdf[[_col, TARGET_COL]].dropna()
                if len(_valid) < 50 or _valid[TARGET_COL].nunique() < 2:
                    continue
                try:
                    _a = max(roc_auc_score(_valid[TARGET_COL], _valid[_col]),
                             1 - roc_auc_score(_valid[TARGET_COL], _valid[_col]))
                    if _a > _best:
                        _best = _a
                        _bf = _col
                except Exception:
                    continue
            if _best > 0:
                _ts_auc.append({'timestep': _ts, 'best_auc': _best, 'best_feature': _bf, 'n_obs': len(_tdf)})

        if _ts_auc:
            _ts_df = pd.DataFrame(_ts_auc)
            print(f"    Timestep range: [{int(_ts_df['timestep'].min()):+d}..{int(_ts_df['timestep'].max()):+d}]")
            print(f"    Pre-battle AUC (t<0):  {_ts_df[_ts_df['timestep']<0]['best_auc'].mean():.3f}" if len(_ts_df[_ts_df['timestep']<0]) > 0 else "    Pre-battle: no data")
            print(f"    Post-battle AUC (t>=0): {_ts_df[_ts_df['timestep']>=0]['best_auc'].mean():.3f}" if len(_ts_df[_ts_df['timestep']>=0]) > 0 else "    Post-battle: no data")

            # plot
            import matplotlib.pyplot as plt
            fig, ax = plt.subplots(figsize=(12, 5))
            ax.plot(_ts_df['timestep'], _ts_df['best_auc'], 'o-', color='#2196F3', markersize=4)
            ax.axhline(0.5, color='red', linestyle='--', alpha=0.5, label='Random')
            ax.axvline(0, color='black', linestyle='-', alpha=0.3, label='Battle start')
            ax.set_xlabel('Timestep (t=0 = battle start)')
            ax.set_ylabel('Best single-feature AUC')
            ax.set_title('F4 fusion_ms_cohdrop: Temporal AUC trajectory')
            ax.legend()
            plt.tight_layout()
            save_fig(fig, 'fusion_ms_cohdrop_temporal_auc', 'per_parquet')
            save_result(_ts_df, 'fusion_ms_cohdrop_temporal_auc', 'per_parquet')
            print(f"    Saved temporal AUC plot + CSV")
        else:
            print("    No timesteps with sufficient data")
    else:
        print("    No timestep column")

    print(f"\n  F4 analysis complete")

del _pq_df
gc.collect()
print("  Memory freed")


CELL P21: F4 -- fusion_ms_cohdrop
  MS + COH drop (per-scene)
  Loaded fusion_ms_cohdrop: 569716 rows, 19 features, 186.4 MB
  Observations: 569716, Cities: 11, Dates: 63, Damaged obs: 76046, Features: 19

  [1/2] PER-PERIOD AUC (best feature per period)
    prebattle      : best AUC=0.571 (s2__b02), 232230 obs
    crossbattle    : best AUC=0.667 (s2__b02), 111076 obs
    postbattle     : best AUC=0.657 (s2__visibility), 226410 obs

  [2/2] TEMPORAL AUC TRAJECTORY (per timestep, best feature)
    Timestep range: [-7..+17]
    Pre-battle AUC (t<0):  0.584
    Post-battle AUC (t>=0): 0.724
    saved -> fusion_ms_cohdrop_temporal_auc.csv (25 rows)
    Saved temporal AUC plot + CSV

  F4 analysis complete
  Memory freed


In [41]:
# @title CELL P22: F5 -- fusion_indices_card (temporal AUC trajectory)
# Parquet: scene_ms | Spectral indices + CARD (per-scene)
import gc
import numpy as np
import pandas as pd
from scipy import stats as scipy_stats
from sklearn.metrics import roc_auc_score

print("=" * 70)
print("CELL P22: F5 -- fusion_indices_card")
print("  Spectral indices + CARD (per-scene)")
print("=" * 70)

_pq_df, _pq_feat, _pq_wide = get_analysis_df('fusion_indices_card')
if _pq_df is None or len(_pq_feat) == 0:
    print("  SKIP: parquet not found or no features")
else:
    _n_cities = _pq_df['city'].nunique()
    _n_dam = (_pq_df[TARGET_COL] == 1).sum() if TARGET_COL in _pq_df.columns else 0
    _n_obs = len(_pq_df)
    _n_dates = _pq_df['date'].nunique() if 'date' in _pq_df.columns else 0
    print(f"  Observations: {_n_obs}, Cities: {_n_cities}, Dates: {_n_dates}, Damaged obs: {_n_dam}, Features: {len(_pq_feat)}")

    # ---- 1. PER-PERIOD AUC (pre vs cross vs post) ----
    print(f"\n  [1/2] PER-PERIOD AUC (best feature per period)")
    if 'period_label' in _pq_df.columns:
        for _period in ['prebattle', 'crossbattle', 'postbattle']:
            _pdf = _pq_df[_pq_df['period_label'] == _period]
            if TARGET_COL not in _pdf.columns or _pdf[TARGET_COL].nunique() < 2 or len(_pdf) < 50:
                print(f"    {_period:15s}: insufficient data ({len(_pdf)} obs)")
                continue
            _best_auc = 0
            _best_feat = ''
            for _col in _pq_feat:
                _valid = _pdf[[_col, TARGET_COL]].dropna()
                if len(_valid) < 50 or _valid[TARGET_COL].nunique() < 2:
                    continue
                try:
                    _auc = max(roc_auc_score(_valid[TARGET_COL], _valid[_col]),
                              1 - roc_auc_score(_valid[TARGET_COL], _valid[_col]))
                    if _auc > _best_auc:
                        _best_auc = _auc
                        _best_feat = _col
                except Exception:
                    continue
            print(f"    {_period:15s}: best AUC={_best_auc:.3f} ({_best_feat}), {len(_pdf)} obs")
            log_nb06_result('fusion_indices_card', 'P10', f'best_auc_{_period}', _best_auc, feature=_best_feat)
    else:
        print("    No period_label column")

    # ---- 2. TEMPORAL AUC TRAJECTORY (per timestep) ----
    print(f"\n  [2/2] TEMPORAL AUC TRAJECTORY (per timestep, best feature)")
    if 'timestep' in _pq_df.columns and TARGET_COL in _pq_df.columns:
        _ts_range = sorted(_pq_df['timestep'].unique())
        _ts_auc = []
        for _ts in _ts_range:
            _tdf = _pq_df[_pq_df['timestep'] == _ts]
            if _tdf[TARGET_COL].nunique() < 2 or len(_tdf) < 50:
                continue
            _best = 0
            _bf = ''
            for _col in _pq_feat[:10]:  # top 10 features only for speed
                _valid = _tdf[[_col, TARGET_COL]].dropna()
                if len(_valid) < 50 or _valid[TARGET_COL].nunique() < 2:
                    continue
                try:
                    _a = max(roc_auc_score(_valid[TARGET_COL], _valid[_col]),
                             1 - roc_auc_score(_valid[TARGET_COL], _valid[_col]))
                    if _a > _best:
                        _best = _a
                        _bf = _col
                except Exception:
                    continue
            if _best > 0:
                _ts_auc.append({'timestep': _ts, 'best_auc': _best, 'best_feature': _bf, 'n_obs': len(_tdf)})

        if _ts_auc:
            _ts_df = pd.DataFrame(_ts_auc)
            print(f"    Timestep range: [{int(_ts_df['timestep'].min()):+d}..{int(_ts_df['timestep'].max()):+d}]")
            print(f"    Pre-battle AUC (t<0):  {_ts_df[_ts_df['timestep']<0]['best_auc'].mean():.3f}" if len(_ts_df[_ts_df['timestep']<0]) > 0 else "    Pre-battle: no data")
            print(f"    Post-battle AUC (t>=0): {_ts_df[_ts_df['timestep']>=0]['best_auc'].mean():.3f}" if len(_ts_df[_ts_df['timestep']>=0]) > 0 else "    Post-battle: no data")

            # plot
            import matplotlib.pyplot as plt
            fig, ax = plt.subplots(figsize=(12, 5))
            ax.plot(_ts_df['timestep'], _ts_df['best_auc'], 'o-', color='#2196F3', markersize=4)
            ax.axhline(0.5, color='red', linestyle='--', alpha=0.5, label='Random')
            ax.axvline(0, color='black', linestyle='-', alpha=0.3, label='Battle start')
            ax.set_xlabel('Timestep (t=0 = battle start)')
            ax.set_ylabel('Best single-feature AUC')
            ax.set_title('F5 fusion_indices_card: Temporal AUC trajectory')
            ax.legend()
            plt.tight_layout()
            save_fig(fig, 'fusion_indices_card_temporal_auc', 'per_parquet')
            save_result(_ts_df, 'fusion_indices_card_temporal_auc', 'per_parquet')
            print(f"    Saved temporal AUC plot + CSV")
        else:
            print("    No timesteps with sufficient data")
    else:
        print("    No timestep column")

    print(f"\n  F5 analysis complete")

del _pq_df
gc.collect()
print("  Memory freed")


CELL P22: F5 -- fusion_indices_card
  Spectral indices + CARD (per-scene)
  Loaded fusion_indices_card: 1592501 rows, 11 features, 473.6 MB
  Observations: 1592501, Cities: 19, Dates: 351, Damaged obs: 204541, Features: 11

  [1/2] PER-PERIOD AUC (best feature per period)
    prebattle      : best AUC=0.603 (s2__ndvi), 286002 obs
    crossbattle    : best AUC=0.657 (s2__ndvi), 152157 obs
    postbattle     : best AUC=0.641 (s2__mndwi), 356046 obs

  [2/2] TEMPORAL AUC TRAJECTORY (per timestep, best feature)
    Timestep range: [-7..+73]
    Pre-battle AUC (t<0):  0.609
    Post-battle AUC (t>=0): 0.769
    saved -> fusion_indices_card_temporal_auc.csv (81 rows)
    Saved temporal AUC plot + CSV

  F5 analysis complete
  Memory freed


In [42]:
# @title CELL P23: F6 -- fusion_indices_card_cohdrop (temporal AUC trajectory)
# Parquet: scene_ms | Indices + CARD + COH drop (per-scene)
import gc
import numpy as np
import pandas as pd
from scipy import stats as scipy_stats
from sklearn.metrics import roc_auc_score

print("=" * 70)
print("CELL P23: F6 -- fusion_indices_card_cohdrop")
print("  Indices + CARD + COH drop (per-scene)")
print("=" * 70)

_pq_df, _pq_feat, _pq_wide = get_analysis_df('fusion_indices_card_cohdrop')
if _pq_df is None or len(_pq_feat) == 0:
    print("  SKIP: parquet not found or no features")
else:
    _n_cities = _pq_df['city'].nunique()
    _n_dam = (_pq_df[TARGET_COL] == 1).sum() if TARGET_COL in _pq_df.columns else 0
    _n_obs = len(_pq_df)
    _n_dates = _pq_df['date'].nunique() if 'date' in _pq_df.columns else 0
    print(f"  Observations: {_n_obs}, Cities: {_n_cities}, Dates: {_n_dates}, Damaged obs: {_n_dam}, Features: {len(_pq_feat)}")

    # ---- 1. PER-PERIOD AUC (pre vs cross vs post) ----
    print(f"\n  [1/2] PER-PERIOD AUC (best feature per period)")
    if 'period_label' in _pq_df.columns:
        for _period in ['prebattle', 'crossbattle', 'postbattle']:
            _pdf = _pq_df[_pq_df['period_label'] == _period]
            if TARGET_COL not in _pdf.columns or _pdf[TARGET_COL].nunique() < 2 or len(_pdf) < 50:
                print(f"    {_period:15s}: insufficient data ({len(_pdf)} obs)")
                continue
            _best_auc = 0
            _best_feat = ''
            for _col in _pq_feat:
                _valid = _pdf[[_col, TARGET_COL]].dropna()
                if len(_valid) < 50 or _valid[TARGET_COL].nunique() < 2:
                    continue
                try:
                    _auc = max(roc_auc_score(_valid[TARGET_COL], _valid[_col]),
                              1 - roc_auc_score(_valid[TARGET_COL], _valid[_col]))
                    if _auc > _best_auc:
                        _best_auc = _auc
                        _best_feat = _col
                except Exception:
                    continue
            print(f"    {_period:15s}: best AUC={_best_auc:.3f} ({_best_feat}), {len(_pdf)} obs")
            log_nb06_result('fusion_indices_card_cohdrop', 'P10', f'best_auc_{_period}', _best_auc, feature=_best_feat)
    else:
        print("    No period_label column")

    # ---- 2. TEMPORAL AUC TRAJECTORY (per timestep) ----
    print(f"\n  [2/2] TEMPORAL AUC TRAJECTORY (per timestep, best feature)")
    if 'timestep' in _pq_df.columns and TARGET_COL in _pq_df.columns:
        _ts_range = sorted(_pq_df['timestep'].unique())
        _ts_auc = []
        for _ts in _ts_range:
            _tdf = _pq_df[_pq_df['timestep'] == _ts]
            if _tdf[TARGET_COL].nunique() < 2 or len(_tdf) < 50:
                continue
            _best = 0
            _bf = ''
            for _col in _pq_feat[:10]:  # top 10 features only for speed
                _valid = _tdf[[_col, TARGET_COL]].dropna()
                if len(_valid) < 50 or _valid[TARGET_COL].nunique() < 2:
                    continue
                try:
                    _a = max(roc_auc_score(_valid[TARGET_COL], _valid[_col]),
                             1 - roc_auc_score(_valid[TARGET_COL], _valid[_col]))
                    if _a > _best:
                        _best = _a
                        _bf = _col
                except Exception:
                    continue
            if _best > 0:
                _ts_auc.append({'timestep': _ts, 'best_auc': _best, 'best_feature': _bf, 'n_obs': len(_tdf)})

        if _ts_auc:
            _ts_df = pd.DataFrame(_ts_auc)
            print(f"    Timestep range: [{int(_ts_df['timestep'].min()):+d}..{int(_ts_df['timestep'].max()):+d}]")
            print(f"    Pre-battle AUC (t<0):  {_ts_df[_ts_df['timestep']<0]['best_auc'].mean():.3f}" if len(_ts_df[_ts_df['timestep']<0]) > 0 else "    Pre-battle: no data")
            print(f"    Post-battle AUC (t>=0): {_ts_df[_ts_df['timestep']>=0]['best_auc'].mean():.3f}" if len(_ts_df[_ts_df['timestep']>=0]) > 0 else "    Post-battle: no data")

            # plot
            import matplotlib.pyplot as plt
            fig, ax = plt.subplots(figsize=(12, 5))
            ax.plot(_ts_df['timestep'], _ts_df['best_auc'], 'o-', color='#2196F3', markersize=4)
            ax.axhline(0.5, color='red', linestyle='--', alpha=0.5, label='Random')
            ax.axvline(0, color='black', linestyle='-', alpha=0.3, label='Battle start')
            ax.set_xlabel('Timestep (t=0 = battle start)')
            ax.set_ylabel('Best single-feature AUC')
            ax.set_title('F6 fusion_indices_card_cohdrop: Temporal AUC trajectory')
            ax.legend()
            plt.tight_layout()
            save_fig(fig, 'fusion_indices_card_cohdrop_temporal_auc', 'per_parquet')
            save_result(_ts_df, 'fusion_indices_card_cohdrop_temporal_auc', 'per_parquet')
            print(f"    Saved temporal AUC plot + CSV")
        else:
            print("    No timesteps with sufficient data")
    else:
        print("    No timestep column")

    print(f"\n  F6 analysis complete")

del _pq_df
gc.collect()
print("  Memory freed")


CELL P23: F6 -- fusion_indices_card_cohdrop
  Indices + CARD + COH drop (per-scene)
  Loaded fusion_indices_card_cohdrop: 1269080 rows, 18 features, 424.1 MB
  Observations: 1269080, Cities: 11, Dates: 178, Damaged obs: 166960, Features: 18

  [1/2] PER-PERIOD AUC (best feature per period)
    prebattle      : best AUC=0.604 (s2__ndvi), 232230 obs
    crossbattle    : best AUC=0.679 (s2__mndwi), 111076 obs
    postbattle     : best AUC=0.645 (s2__mndwi), 300241 obs

  [2/2] TEMPORAL AUC TRAJECTORY (per timestep, best feature)
    Timestep range: [-7..+20]
    Pre-battle AUC (t<0):  0.608
    Post-battle AUC (t>=0): 0.741
    saved -> fusion_indices_card_cohdrop_temporal_auc.csv (28 rows)
    Saved temporal AUC plot + CSV

  F6 analysis complete
  Memory freed


In [43]:
# @title CELL P17: A11 -- composite_vs_scenes_bands (temporal AUC trajectory)
# Parquet: composite_vs_scenes_bands | Pre-composite vs per-scene post delta
import gc
import numpy as np
import pandas as pd
from scipy import stats as scipy_stats
from sklearn.metrics import roc_auc_score

print("=" * 70)
print("CELL P17: A11 -- composite_vs_scenes_bands")
print("  Pre-composite vs per-scene post delta")
print("=" * 70)

_pq_df, _pq_feat, _pq_wide = get_analysis_df('composite_vs_scenes_bands')
if _pq_df is None or len(_pq_feat) == 0:
    print("  SKIP: parquet not found or no features")
else:
    _n_cities = _pq_df['city'].nunique()
    _n_dam = (_pq_df[TARGET_COL] == 1).sum() if TARGET_COL in _pq_df.columns else 0
    _n_obs = len(_pq_df)
    _n_dates = _pq_df['date'].nunique() if 'date' in _pq_df.columns else 0
    print(f"  Observations: {_n_obs}, Cities: {_n_cities}, Dates: {_n_dates}, Damaged obs: {_n_dam}, Features: {len(_pq_feat)}")

    # ---- 1. PER-PERIOD AUC (pre vs cross vs post) ----
    print(f"\n  [1/2] PER-PERIOD AUC (best feature per period)")
    if 'period_label' in _pq_df.columns:
        for _period in ['prebattle', 'crossbattle', 'postbattle']:
            _pdf = _pq_df[_pq_df['period_label'] == _period]
            if TARGET_COL not in _pdf.columns or _pdf[TARGET_COL].nunique() < 2 or len(_pdf) < 50:
                print(f"    {_period:15s}: insufficient data ({len(_pdf)} obs)")
                continue
            _best_auc = 0
            _best_feat = ''
            for _col in _pq_feat:
                _valid = _pdf[[_col, TARGET_COL]].dropna()
                if len(_valid) < 50 or _valid[TARGET_COL].nunique() < 2:
                    continue
                try:
                    _auc = max(roc_auc_score(_valid[TARGET_COL], _valid[_col]),
                              1 - roc_auc_score(_valid[TARGET_COL], _valid[_col]))
                    if _auc > _best_auc:
                        _best_auc = _auc
                        _best_feat = _col
                except Exception:
                    continue
            print(f"    {_period:15s}: best AUC={_best_auc:.3f} ({_best_feat}), {len(_pdf)} obs")
            log_nb06_result('composite_vs_scenes_bands', 'P17', f'best_auc_{_period}', _best_auc, feature=_best_feat)
    else:
        print("    No period_label column")

    # ---- 2. TEMPORAL AUC TRAJECTORY (per timestep) ----
    print(f"\n  [2/2] TEMPORAL AUC TRAJECTORY (per timestep, best feature)")
    if 'timestep' in _pq_df.columns and TARGET_COL in _pq_df.columns:
        _ts_range = sorted(_pq_df['timestep'].unique())
        _ts_auc = []
        for _ts in _ts_range:
            _tdf = _pq_df[_pq_df['timestep'] == _ts]
            if _tdf[TARGET_COL].nunique() < 2 or len(_tdf) < 50:
                continue
            _best = 0
            _bf = ''
            for _col in _pq_feat[:10]:  # top 10 features only for speed
                _valid = _tdf[[_col, TARGET_COL]].dropna()
                if len(_valid) < 50 or _valid[TARGET_COL].nunique() < 2:
                    continue
                try:
                    _a = max(roc_auc_score(_valid[TARGET_COL], _valid[_col]),
                             1 - roc_auc_score(_valid[TARGET_COL], _valid[_col]))
                    if _a > _best:
                        _best = _a
                        _bf = _col
                except Exception:
                    continue
            if _best > 0:
                _ts_auc.append({'timestep': _ts, 'best_auc': _best, 'best_feature': _bf, 'n_obs': len(_tdf)})

        if _ts_auc:
            _ts_df = pd.DataFrame(_ts_auc)
            print(f"    Timestep range: [{int(_ts_df['timestep'].min()):+d}..{int(_ts_df['timestep'].max()):+d}]")
            print(f"    Pre-battle AUC (t<0):  {_ts_df[_ts_df['timestep']<0]['best_auc'].mean():.3f}" if len(_ts_df[_ts_df['timestep']<0]) > 0 else "    Pre-battle: no data")
            print(f"    Post-battle AUC (t>=0): {_ts_df[_ts_df['timestep']>=0]['best_auc'].mean():.3f}" if len(_ts_df[_ts_df['timestep']>=0]) > 0 else "    Post-battle: no data")

            # plot
            import matplotlib.pyplot as plt
            fig, ax = plt.subplots(figsize=(12, 5))
            ax.plot(_ts_df['timestep'], _ts_df['best_auc'], 'o-', color='#2196F3', markersize=4)
            ax.axhline(0.5, color='red', linestyle='--', alpha=0.5, label='Random')
            ax.axvline(0, color='black', linestyle='-', alpha=0.3, label='Battle start')
            ax.set_xlabel('Timestep (t=0 = battle start)')
            ax.set_ylabel('Best single-feature AUC')
            ax.set_title('A11 composite_vs_scenes_bands: Temporal AUC trajectory')
            ax.legend()
            plt.tight_layout()
            save_fig(fig, 'composite_vs_scenes_bands_temporal_auc', 'per_parquet')
            save_result(_ts_df, 'composite_vs_scenes_bands_temporal_auc', 'per_parquet')
            print(f"    Saved temporal AUC plot + CSV")
        else:
            print("    No timesteps with sufficient data")
    else:
        print("    No timestep column")

    print(f"\n  A11 analysis complete")

del _pq_df
gc.collect()
print("  Memory freed")


CELL P17: A11 -- composite_vs_scenes_bands
  Pre-composite vs per-scene post delta
  SKIP: parquet not found or no features
  Memory freed


# CELL 16: MANIFEST-DRIVEN AUC COMPARISON

In [44]:
# @title CELL 16: MANIFEST-DRIVEN AUC COMPARISON (lazy, per-parquet)
import pandas as pd
import numpy as np
from sklearn.metrics import roc_auc_score

print("=" * 70)
print("CELL 16: PER-PARQUET AUC COMPARISON (v2 manifest, lazy loading)")
print("=" * 70)

auc_comparison = []

for pq_name, pq_info in sorted(V3_PARQUET_INFO.items()):
    pq_df, feat_cols, is_wide = get_analysis_df(pq_name)
    if pq_df is None or len(feat_cols) == 0:
        continue
    if TARGET_COL not in pq_df.columns or pq_df[TARGET_COL].nunique() < 2:
        del pq_df; gc.collect()
        continue

    valid = pq_df[[TARGET_COL] + feat_cols].dropna(subset=[TARGET_COL])
    _n_cities_active = int(pq_df['city'].nunique()) if 'city' in pq_df.columns else 0

    best_auc = 0.0
    best_feat = ''
    n_good = 0

    for col in feat_cols:
        col_valid = valid[[col, TARGET_COL]].dropna()
        if len(col_valid) < 50 or col_valid[TARGET_COL].nunique() < 2:
            continue
        try:
            auc = roc_auc_score(col_valid[TARGET_COL], col_valid[col])
            auc_best = max(auc, 1 - auc)
            if auc_best > best_auc:
                best_auc = auc_best
                best_feat = col
            if auc_best > 0.55:
                n_good += 1
        except Exception:
            continue

    # manifest-level exclusions (populated by NB05b fusion inner-join / atomic drop)
    _excl_per_tier = pq_info.get('cities_excluded_per_tier', {}) or {}
    _excl_flat = sorted({c for cs in _excl_per_tier.values() for c in (cs or [])})

    auc_comparison.append({
        'parquet': pq_name, 'id': pq_info['id'],
        'format': pq_info.get('format', '?'),
        'n_features': len(feat_cols), 'n_rows': len(valid),
        'n_cities': _n_cities_active, 'n_excluded': len(_excl_flat),
        'excluded_cities': ','.join(_excl_flat) if _excl_flat else '',
        'best_auc': round(best_auc, 4), 'best_feature': best_feat,
        'n_auc_gt_055': n_good,
        'question': pq_info.get('experiment_question', ''),
    })

    log_nb06_result(pq_name, 'cell16', 'best_single_feature_auc', best_auc, feature=best_feat)

    # free memory
    del pq_df, valid
    gc.collect()
    print(f"    -> freed memory")

if auc_comparison:
    auc_df = pd.DataFrame(auc_comparison).sort_values('best_auc', ascending=False)
    print(f"\n  {'Parquet':<35s} {'ID':>3s} {'Fmt':>5s} {'Feat':>5s} {'Rows':>8s} {'Cit':>3s} {'Exc':>3s} {'BestAUC':>8s} {'#>0.55':>6s}  Best Feature / Excluded")
    print(f"  {'-'*35} {'-'*3} {'-'*5} {'-'*5} {'-'*8} {'-'*3} {'-'*3} {'-'*8} {'-'*6}  {'-'*30}")
    for _, row in auc_df.iterrows():
        _tail = row['best_feature'][:40]
        if row['n_excluded'] > 0:
            _tail = f"{_tail}  (excl: {row['excluded_cities']})"
        print(f"  {row['parquet']:<35s} {row['id']:>3s} {row['format']:>5s} {row['n_features']:>5d} {row['n_rows']:>8d} {row['n_cities']:>3d} {row['n_excluded']:>3d} {row['best_auc']:>8.4f} {row['n_auc_gt_055']:>6d}  {_tail}")

    # save CSV for NB13
    csv_path = OUT_DIR / 'cell16_manifest' / 'v2_parquet_auc_comparison.csv'
    csv_path.parent.mkdir(parents=True, exist_ok=True)
    auc_df.to_csv(csv_path, index=False)
    print(f"\n  CSV saved: {csv_path.name}")

    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(figsize=(14, max(6, len(auc_df) * 0.35)))
    colors = ['#2196F3' if 'fusion' in r else '#4CAF50' if r.startswith('scene') else '#FF9800'
              for r in auc_df['parquet']]
    ax.barh(range(len(auc_df)), auc_df['best_auc'], color=colors, edgecolor='white')
    ax.set_yticks(range(len(auc_df)))
    ax.set_yticklabels([f"{r['parquet']} [{r['id']}]" for _, r in auc_df.iterrows()], fontsize=9)
    ax.set_xlabel('Best single-feature AUC')
    ax.set_title('v2 Parquet Comparison: Best Single-Feature AUC')
    ax.axvline(0.5, color='red', linestyle='--', alpha=0.5, label='Random')
    ax.axvline(0.55, color='orange', linestyle='--', alpha=0.5, label='Threshold')
    ax.legend()
    ax.invert_yaxis()
    plt.tight_layout()
    save_fig(fig, 'v2_parquet_auc_comparison', 'cell16_manifest')
    print(f"  Chart saved")
else:
    print("  No parquets with valid AUC results")


CELL 16: PER-PARQUET AUC COMPARISON (v2 manifest, lazy loading)
  Loaded block_accum_card: 63243 rows, 85 features, 34.7 MB
    -> freed memory
  Loaded block_accum_coh: 60076 rows, 18 features, 16.9 MB
    -> freed memory
  Loaded block_accum_ms: 62043 rows, 441 features, 122.4 MB
    -> freed memory
  Loaded block_stats: 63243 rows, 330 features, 96.6 MB
    -> freed memory
  Loaded card_drop: 62043 rows, 7 features, 14.7 MB
    -> freed memory
  Loaded coh_drop: 51800 rows, 7 features, 12.3 MB
    -> freed memory
  Loaded composite_prepost_bands: 62043 rows, 79 features, 32.5 MB
    -> freed memory
  Loaded composite_prepost_landuse: 62043 rows, 4 features, 14.2 MB
    -> freed memory
  Loaded composite_vs_scenes_landuse: 356046 rows, 3 features, 92.4 MB
    -> freed memory
  Loaded fusion_card_cohdrop: 658108 rows, 9 features, 188.7 MB
    -> freed memory
  Loaded fusion_composite_blockstats: 62043 rows, 409 features, 115.4 MB
    -> freed memory
  Loaded fusion_composite_cohdrop: 

# NB06 v18 Scientific Notes

## Methods justification
- **Non-parametric tests** (Mann-Whitney U, Kruskal-Wallis): justified by non-normal distributions (Cell 8)
- **Spearman correlation** (not Pearson): rank-based, robust to non-normality and outliers
- **Feature selection for VIF/heatmap**: by AUC (discriminability), not variance (scale-dependent)
- **Cohen's d with pooled std**: standard parametric effect size, complemented by non-parametric rank-biserial

## UNOSAT labeling scheme
- `damage_binary = -1`: building not in UNOSAT assessment area (NOT "undamaged")
- `damage_binary = 0`: assessed by UNOSAT as undamaged (grade 1-2)
- `damage_binary = 1`: assessed by UNOSAT as damaged (grade 3-4 = severe/destroyed)
- `damage` column: raw UNOSAT EMS-98 grade (1-4, NaN if not assessed)
- `FILTER_UNOSAT_ONLY=True` drops -1 for supervised analysis; `False` keeps all for unsupervised
- `BALANCE_CLASSES=True` creates `df_balanced` with 50/50 per city (for PCA, KDE)

## NaN structure diagnostic (Cell 3B)
- **NaN-encodes-city test**: RF trained on binary NaN mask to predict city identity. If accuracy >> chance (1/n_cities), NaN patterns are a geographic leakage vector. LightGBM MIA in NB09 can exploit this.
- **Listwise vs pairwise dropout**: quantifies how much data PCA/VIF lose vs AUC/Mann-Whitney. If listwise drops >50%, PCA results are on a non-representative subset.
- **Temporal balance**: scene parquets with timestep show pre/post observation count per modality. Severe imbalance means pre-battle baseline features are unreliable.
- **Feature-group NaN variance across cities**: high variance = different cities have different modality coverage = structural NaN that encodes geography.

## Known limitations
- **Bhattacharyya/J-M distance** assumes Gaussian — invalid for categorical (landuse) and sparse (fire) features
- **PCA** drops NaN rows — significant data loss possible with SAR NaN patterns
- **Shapiro-Wilk** at n>5000 rejects almost any distribution — interpret p-values qualitatively
- **Ripley's K**: no edge correction applied — L(r) biased at large distances
- **Moran's I**: KNN k=8 on lon/lat coordinates — distances in degrees are approximate
- **UNOSAT labels**: known to have false negatives (Aimaiti 2022); undamaged label not reliable
- **Class imbalance**: 5091 damaged vs 88156 undamaged (5.5%) — AUC/Mann-Whitney unaffected, but PCA/KDE may be dominated by majority class

## NB03d products incorporated
- Composite spectral indices (NDVI, NDBI, NBR, NDWI, MNDWI, NDRE, CIre, BSI, BAEI)
- dNBR/RBR change detection maps (composites root)
- Per-scene NBR (multispectral/nbr/)
- CARD temporal statistics (baseline/assessment mean/std)
- Coherence baseline statistics (pre-battle baseline)
- Temporal products: COH zscore, COH rolling, CARD rolling, COH post-baseline
- Landuse classification per period (mode extraction for categorical, mean/std for spectral indices)
- Fire detection (active_fire/burn_scar) — when available in parquet


# CELL 17: ROLLING + BLOCK + COH DROP FEATURE ANALYSIS

Loads separate parquets independently (no merge into main df to avoid NaN contamination):
- Rolling stats (P1b): per-window temporal aggregates
- Block stats (P1c): block-based temporal aggregates
- COH drop accumulator (R3): running_min, drop_count, max_drop per building

Each dataset gets its own AUC + Cohen's d analysis.
Results feed into NB13 for cross-dataset comparison and model selection.


In [45]:
# @title CELL 17: ROLLING + BLOCK FEATURE ANALYSIS (P1b + P1c products)
import numpy as np
import pandas as pd
from scipy import stats as scipy_stats
from sklearn.metrics import roc_auc_score
from statsmodels.stats.outliers_influence import variance_inflation_factor
import re

print("=" * 70)
print("CELL 17: ROLLING + BLOCK FEATURE ANALYSIS")
print("=" * 70)

# =============================================================================
# LOAD NEW PARQUETS (per-tier)
# =============================================================================
_tiers = TIER_SELECTION if TIER_SELECTION != "ALL" else [0, 1, 2, 3, 4, 5]

new_parquets = {}

# rolling stats: one per window size, pick default window=7
_ROLL_WINDOW = 7
try:
    _rs = load_v3_parquet(f'rolling_stats_roll{_ROLL_WINDOW}')
    new_parquets['rolling_stats_card'] = _rs
    print(f"  rolling_stats (roll{_ROLL_WINDOW}): {len(_rs)} rows, {len(_rs.columns)} cols, {_rs['city'].nunique()} cities")
except FileNotFoundError as e:
    print(f"  rolling_stats: {e}")

# block stats
try:
    _bs = load_v3_parquet('block_stats')
    new_parquets['block_stats_card'] = _bs
    print(f"  block_stats: {len(_bs)} rows, {len(_bs.columns)} cols, {_bs['city'].nunique()} cities")
except FileNotFoundError as e:
    print(f"  block_stats: {e}")

# COH drop accumulator (from v2 parquets)
try:
    _cd = load_v3_parquet('coh_drop')
    if _cd is not None:
        new_parquets['coh_drop'] = _cd
        print(f"  coh_drop: {len(_cd)} rows, {len(_cd.columns)} cols, {_cd['city'].nunique()} cities")
except Exception as e:
    print(f"  coh_drop: failed ({e})")
    print(f"  coh_drop: {e}")
else:
    print(f"  coh_drop: not found, skipping")

if not new_parquets:
    print("\n  No new parquets available. Skipping cell.")
else:
    # merge damage labels from main df (already loaded in Cell 3)
    label_cols = ['point_id', 'city', TARGET_COL]
    label_df = df[label_cols].copy()

    # =============================================================================
    # PART 1: AUC COMPARISON TABLE (raw P1 vs rolling stats vs blocks)
    # =============================================================================
    print(f"\n{'='*70}")
    print("PART 1: AUC COMPARISON -- Raw P1 vs Rolling Stats vs Blocks")
    print(f"{'='*70}")

    def compute_auc_for_features(feat_df, feature_cols, city_name, target_col):
        """Compute AUC for each feature column. Returns list of dicts."""
        city_data = feat_df[feat_df['city'] == city_name]
        if len(city_data) < MIN_SAMPLES or target_col not in city_data.columns:
            return []
        if city_data[target_col].nunique() < 2:
            return []
        results = []
        for col in feature_cols:
            valid = city_data[[col, target_col]].dropna()
            if len(valid) < MIN_SAMPLES or valid[target_col].nunique() < 2:
                continue
            try:
                auc = roc_auc_score(valid[target_col], valid[col])
                auc_best = max(auc, 1 - auc)
                results.append({'feature': col, 'city': city_name, 'auc': auc_best})
            except:
                pass
        return results

    # raw P1 features from product_prepost (already in df)
    raw_card_cols = [c for c in FEATURE_COLS if 'baseline' in c or 'assessment' in c]
    raw_card_cols = [c for c in raw_card_cols if c.startswith('s1__') and 'roll' not in c and 'blk' not in c]

    all_auc_rows = []

    for city_name in CITIES_TO_PROCESS:
        # raw P1 AUCs
        for r in compute_auc_for_features(df, raw_card_cols, city_name, TARGET_COL):
            r['source'] = 'raw_P1'
            # extract stat name from e.g. s1__vv__assessment__mean
            parts = r['feature'].split('__')
            r['stat'] = parts[-1] if len(parts) >= 4 else ''
            r['pol'] = parts[1] if len(parts) >= 2 else ''
            r['phase'] = parts[2] if len(parts) >= 3 else ''
            r['window'] = 'raw'
            all_auc_rows.append(r)

        # rolling stats AUCs
        if 'rolling_stats_card' in new_parquets:
            rs_df = new_parquets['rolling_stats_card'].merge(label_df, on=['point_id', 'city'], how='inner')
            rs_feat_cols = [c for c in rs_df.columns if c.startswith('s1__') and 'roll' in c]
            for r in compute_auc_for_features(rs_df, rs_feat_cols, city_name, TARGET_COL):
                r['source'] = 'rolling_stats'
                parts = r['feature'].split('__')
                # s1__vv__roll7__assessment__mean
                r['pol'] = parts[1] if len(parts) >= 2 else ''
                m = re.search(r'roll(\d+)', r['feature'])
                r['window'] = f"roll{m.group(1)}" if m else ''
                r['stat'] = parts[-1] if len(parts) >= 5 else ''
                r['phase'] = parts[3] if len(parts) >= 5 else ''
                all_auc_rows.append(r)

        # block stats AUCs
        if 'block_stats_card' in new_parquets:
            bs_df = new_parquets['block_stats_card'].merge(label_df, on=['point_id', 'city'], how='inner')
            bs_feat_cols = [c for c in bs_df.columns if c.startswith('s1__') and 'blk' in c]
            for r in compute_auc_for_features(bs_df, bs_feat_cols, city_name, TARGET_COL):
                r['source'] = 'block_stats'
                parts = r['feature'].split('__')
                r['pol'] = parts[1] if len(parts) >= 2 else ''
                m = re.search(r'(blk(?:_pre)?\d+)', r['feature'])
                r['window'] = m.group(1) if m else ''
                # column = s1__vh__blk01__mean_mean -> last part is {tif_stat}_{zonal_agg}
                last = parts[-1] if len(parts) >= 4 else ''
                sub = last.split('_')
                r['stat'] = sub[0] if sub else ''
                r['zonal'] = sub[1] if len(sub) > 1 else ''
                r['phase'] = r['window']
                all_auc_rows.append(r)

    if all_auc_rows:
        auc_new = pd.DataFrame(all_auc_rows)

        # summary: best AUC per source x window, averaged across cities
        print(f"\n  Mean best-AUC per source x window (across cities, VV assessment/post-invasion features):")
        vv_assess = auc_new[
            (auc_new['pol'].isin(['vv', 'coh_vv'])) &
            (auc_new['phase'].str.startswith('blk') | auc_new['phase'].isin(['assessment']))
        ]
        if len(vv_assess) > 0:
            pivot = vv_assess.groupby(['source', 'window', 'stat'])['auc'].mean().reset_index()
            # best stat per window
            best_per_window = pivot.loc[pivot.groupby(['source', 'window'])['auc'].idxmax()]
            best_per_window = best_per_window.sort_values('auc', ascending=False)
            print(f"\n  {'Source':<18s} {'Window':<12s} {'Best stat':<12s} {'Mean AUC':>8s}")
            print(f"  {'-'*18} {'-'*12} {'-'*12} {'-'*8}")
            for _, row in best_per_window.head(20).iterrows():
                print(f"  {row['source']:<18s} {row['window']:<12s} {row['stat']:<12s} {row['auc']:>8.3f}")

        # =============================================================================
        # PART 2: BLOCK TIMELINE (AUC per block showing damage emergence)
        # =============================================================================
        if 'block_stats_card' in new_parquets:
            print(f"\n{'='*70}")
            print("PART 2: BLOCK TIMELINE -- When does damage become detectable?")
            print(f"{'='*70}")

            block_auc = auc_new[auc_new['source'] == 'block_stats'].copy()
            if len(block_auc) > 0:
                # get block metadata from parquet
                bs_df_meta = new_parquets['block_stats_card']
                meta_cols = [c for c in bs_df_meta.columns if c.startswith('meta__')]

                # per block, mean AUC of mean stat across cities
                block_mean = block_auc[block_auc['stat'] == 'mean'].groupby('window')['auc'].agg(['mean', 'std', 'count']).reset_index()
                block_mean = block_mean.sort_values('window')

                print(f"\n  {'Block':<14s} {'Mean AUC':>8s} {'Std':>6s} {'Cities':>6s} {'Interpretation'}")
                print(f"  {'-'*14} {'-'*8} {'-'*6} {'-'*6} {'-'*30}")
                for _, row in block_mean.iterrows():
                    blk = row['window']
                    interp = ""
                    if blk == 'blk00':
                        interp = "baseline (should be ~0.50)"
                    elif blk.startswith('blk_pre'):
                        interp = "pre-invasion (should be ~0.50)"
                    elif blk == 'blk01':
                        interp = "first 3mo post-invasion"
                    elif blk == 'blk02':
                        interp = "3-6mo post-invasion"
                    elif blk == 'blk03':
                        interp = "6-9mo post-invasion"
                    print(f"  {blk:<14s} {row['mean']:>8.3f} {row['std']:>6.3f} {int(row['count']):>6d} {interp}")

                print(f"\n  Interpretation: AUC should be ~0.50 for baseline/pre-invasion blocks")
                print(f"  (no damage signal), then rise for post-invasion blocks.")
                print(f"  If pre-invasion blocks show AUC >> 0.50, the model detects non-war changes.")
                print(f"  This is Dietrich's Eq. 3 pre-invasion guard rationale.")

        # =============================================================================
        # PART 3: VIF BETWEEN RAW P1 AND ROLLING FEATURES
        # =============================================================================
        if 'rolling_stats_card' in new_parquets:
            print(f"\n{'='*70}")
            print("PART 3: VIF -- Are rolling stats redundant with raw P1 stats?")
            print(f"{'='*70}")

            # pick VV assessment mean/std from raw and each rolling size
            vif_candidates = []
            for col in raw_card_cols:
                if 'vv__assessment__mean' in col or 'vv__assessment__std' in col:
                    vif_candidates.append(col)

            rs_df_full = new_parquets['rolling_stats_card'].merge(label_df, on=['point_id', 'city'], how='inner')
            for ws in [3, 5, 7, 13]:
                for stat in ['mean', 'std']:
                    col = f"s1__vv__roll{ws}__assessment__{stat}"
                    if col in rs_df_full.columns:
                        vif_candidates.append(col)

            # combine into one df for VIF
            combined_cols = [c for c in vif_candidates if c in df.columns or c in rs_df_full.columns]
            vif_df = df[['point_id', 'city'] + [c for c in combined_cols if c in df.columns]].copy()
            rs_extra = [c for c in combined_cols if c not in df.columns and c in rs_df_full.columns]
            if rs_extra:
                vif_df = vif_df.merge(rs_df_full[['point_id', 'city'] + rs_extra], on=['point_id', 'city'], how='inner')

            vif_cols = [c for c in combined_cols if c in vif_df.columns]
            vif_data = vif_df[vif_cols].dropna()

            if len(vif_data) > 100 and len(vif_cols) >= 2:
                print(f"\n  VIF for {len(vif_cols)} features ({len(vif_data)} samples):")
                vif_matrix = vif_data.values.astype(np.float64)
                # add constant
                vif_matrix = np.column_stack([vif_matrix, np.ones(len(vif_matrix))])
                for j, col in enumerate(vif_cols):
                    try:
                        vif_val = variance_inflation_factor(vif_matrix, j)
                        tag = "HIGH REDUNDANCY" if vif_val > 10 else "ok" if vif_val < 5 else "moderate"
                        short_name = col.replace('s1__vv__', '').replace('assessment__', 'a_')
                        print(f"    {short_name:<35s} VIF={vif_val:>8.1f}  ({tag})")
                    except:
                        pass
                print(f"\n  VIF > 10 = highly redundant with other features in set")
                print(f"  If raw P1 and roll7 both have VIF > 10, keep only one in NB09a")
            else:
                print(f"\n  Not enough data for VIF ({len(vif_data)} samples, {len(vif_cols)} cols)")

        # =============================================================================
        # PART 4: COHEN'S D FOR BEST ROLLING FEATURES
        # =============================================================================
        print(f"\n{'='*70}")
        print("PART 4: COHEN'S D -- Effect size for top rolling features")
        print(f"{'='*70}")

        top_features = auc_new.groupby('feature')['auc'].mean().nlargest(15).index.tolist()
        if top_features and 'rolling_stats_card' in new_parquets:
            rs_df_labels = new_parquets['rolling_stats_card'].merge(label_df, on=['point_id', 'city'], how='inner')
            all_feats_df = pd.concat([df, rs_df_labels.drop(columns=[c for c in rs_df_labels.columns if c in df.columns and c not in ['point_id', 'city']], errors='ignore')], axis=1)

            print(f"\n  {'Feature':<50s} {'d':>7s} {'AUC':>7s} {'Effect'}")
            print(f"  {'-'*50} {'-'*7} {'-'*7} {'-'*12}")
            for feat in top_features:
                src_df = df if feat in df.columns else rs_df_labels if feat in rs_df_labels.columns else None
                if src_df is None:
                    if 'block_stats_card' in new_parquets:
                        bs_labels = new_parquets['block_stats_card'].merge(label_df, on=['point_id', 'city'], how='inner')
                        if feat in bs_labels.columns:
                            src_df = bs_labels
                if src_df is None:
                    continue
                valid = src_df[[feat, TARGET_COL]].dropna()
                if len(valid) < 20 or valid[TARGET_COL].nunique() < 2:
                    continue
                d_vals = valid.loc[valid[TARGET_COL] == 1, feat].values
                c_vals = valid.loc[valid[TARGET_COL] == 0, feat].values
                pooled_std = np.sqrt((np.var(d_vals) + np.var(c_vals)) / 2)
                if pooled_std < 1e-10:
                    continue
                cohens_d = abs(np.mean(d_vals) - np.mean(c_vals)) / pooled_std
                mean_auc = auc_new[auc_new['feature'] == feat]['auc'].mean()
                effect = "large" if cohens_d > 0.8 else "medium" if cohens_d > 0.5 else "small"
                short = feat.replace('s1__', '').replace('assessment__', 'a_')[:48]
                print(f"  {short:<50s} {cohens_d:>7.3f} {mean_auc:>7.3f} {effect}")
    else:
        print("\n  No AUC results computed")

# registry
    if all_auc_rows:
        _top_new = auc_new.groupby('feature')['auc'].mean().nlargest(10)
        registry.log_statistics(
            cell_id='cell17_rolling_block',
            analysis_name='rolling_block_auc_comparison',
            parquet_name='rolling_stats+block_stats',
            tier_selection=_tiers,
            cities=CITIES_TO_PROCESS,
            n_buildings=len(df),
            n_features_tested=auc_new['feature'].nunique(),
            summary_metrics={
                'mean_auc_raw': float(auc_new[auc_new['source'] == 'raw_P1']['auc'].mean()) if (auc_new['source'] == 'raw_P1').any() else None,
                'mean_auc_rolling': float(auc_new[auc_new['source'] == 'rolling_stats']['auc'].mean()) if (auc_new['source'] == 'rolling_stats').any() else None,
                'mean_auc_block': float(auc_new[auc_new['source'] == 'block_stats']['auc'].mean()) if (auc_new['source'] == 'block_stats').any() else None,
            },
            top_features=[{'feature': f, 'mean_auc': float(a)} for f, a in _top_new.items()],
            note='Rolling + block feature AUC vs raw P1 (Cell 17)',
            tags=['rolling', 'block', 'temporal', 'auc'],
        )

# =============================================================================
# COH DROP ACCUMULATOR ANALYSIS
# =============================================================================
if 'coh_drop' in new_parquets:
    print(f"\n{'='*70}")
    print("COH DROP ACCUMULATOR ANALYSIS")
    print(f"{'='*70}")

    cd_df = new_parquets['coh_drop'].merge(label_df, on=['point_id', 'city'], how='inner')
    cd_feat_cols = [c for c in new_parquets['coh_drop'].columns
                    if c not in ('point_id', 'city', 'tier', TARGET_COL)]
    print(f"  {len(cd_df)} buildings, {len(cd_feat_cols)} features, {cd_df['city'].nunique()} cities")

    cd_auc_rows = []
    for city_name in sorted(cd_df['city'].unique()):
        for r in compute_auc_for_features(cd_df, cd_feat_cols, city_name, TARGET_COL):
            r['source'] = 'coh_drop'
            cd_auc_rows.append(r)

    if cd_auc_rows:
        cd_auc_df = pd.DataFrame(cd_auc_rows)
        save_result(cd_auc_df, 'coh_drop_auc', 'cell17_rolling_block')

        # top features
        cd_top = cd_auc_df.groupby('feature')['auc'].mean().sort_values(ascending=False)
        print(f"\n  Top COH drop features by mean AUC:")
        for feat, auc_val in cd_top.head(10).items():
            print(f"    {feat:<55s} AUC={auc_val:.3f}")

        # Cohen's d for top features
        for city_name in sorted(cd_df['city'].unique()):
            city_data = cd_df[cd_df['city'] == city_name]
            if city_data[TARGET_COL].nunique() < 2:
                continue
            print(f"\n  {city_name}:")
            for feat in cd_top.head(5).index:
                if feat not in city_data.columns:
                    continue
                d_vals = city_data.loc[city_data[TARGET_COL] == 1, feat].dropna().values
                c_vals = city_data.loc[city_data[TARGET_COL] == 0, feat].dropna().values
                if len(d_vals) < 10 or len(c_vals) < 10:
                    continue
                n1, n2 = len(d_vals), len(c_vals)
                pooled = np.sqrt(((n1-1)*np.var(d_vals,ddof=1) + (n2-1)*np.var(c_vals,ddof=1)) / (n1+n2-2))
                d = (np.mean(d_vals) - np.mean(c_vals)) / pooled if pooled > 0 else 0
                effect = 'large' if abs(d) > 0.8 else 'medium' if abs(d) > 0.5 else 'small' if abs(d) > 0.2 else 'negligible'
                print(f"    {feat:<50s} d={d:+.3f} [{effect}]")

        # add to all_auc_rows for combined comparison
        all_auc_rows.extend(cd_auc_rows)

        # registry
        _top_cd = cd_auc_df.groupby('feature')['auc'].mean().nlargest(10)
        registry.log_statistics(
            cell_id='cell17_coh_drop',
            analysis_name='coh_drop_auc',
            parquet_name='bda_coh_drop',
            tier_selection=_tiers,
            cities=sorted(cd_df['city'].unique()),
            n_buildings=len(cd_df),
            n_features_tested=cd_auc_df['feature'].nunique(),
            summary_metrics={
                'mean_auc': float(cd_auc_df['auc'].mean()),
                'max_auc': float(cd_auc_df['auc'].max()),
                'n_cities': int(cd_df['city'].nunique()),
            },
            top_features=[{'feature': f, 'mean_auc': float(a)} for f, a in _top_cd.items()],
            note='COH drop accumulator features (running_min, drop_count, max_drop)',
            tags=['coh_drop', 'accumulator', 'change_detection', 'auc'],
        )
    else:
        print("  No COH drop AUC results computed")

print(f"\n{'='*70}")
print("CELL 17 COMPLETE")
print(f"{'='*70}")



CELL 17: ROLLING + BLOCK FEATURE ANALYSIS
  rolling_stats (roll7): 63243 rows, 41 cols, 21 cities
  block_stats: 63243 rows, 333 cols, 21 cities
  coh_drop: 51800 rows, 10 cols, 12 cities
  coh_drop: not found, skipping

PART 1: AUC COMPARISON -- Raw P1 vs Rolling Stats vs Blocks

  Mean best-AUC per source x window (across cities, VV assessment/post-invasion features):

  Source             Window       Best stat    Mean AUC
  ------------------ ------------ ------------ --------
  block_stats        blk02        median          0.582
  block_stats        blk03        mean            0.566
  block_stats        blk01        std             0.560
  block_stats        blk_pre01    mean            0.556
  block_stats        blk15        std             0.538
  block_stats        blk08        std             0.535
  block_stats        blk06        min             0.526
  block_stats        blk07        kurtosis        0.524
  block_stats        blk05        median          0.521
  block_st

# CELL 18: SINGLE-SCENE & ACCUMULATOR PRODUCT ANALYSIS
# P7 (RGB pre/post single-scene), P7b (CARD pre/post single-scene), R3 (COH drop accumulator)
# These are NB03e-derived TIF products: one pre + one post scene per city.
# This cell loads them directly from source dirs, samples at building centroids, and runs AUC + Mann-Whitney.


In [46]:
# @title CELL 18: SINGLE-SCENE & ACCUMULATOR PRODUCT ANALYSIS (P7 + P7b + R3)
import numpy as np
import pandas as pd
import rasterio
from pathlib import Path
from scipy import stats as scipy_stats
from sklearn.metrics import roc_auc_score
import re
import json

print("=" * 70)
print("CELL 18: SINGLE-SCENE & ACCUMULATOR PRODUCT ANALYSIS")
print("=" * 70)
print("  P7:  RGB pre/post single-scene (MS_DIR/{city}/rgb_prepost/)")
print("  P7b: CARD pre/post single-scene (SAR_CARD_DIR/{city}/card_prepost/)")
print("  R3:  COH drop accumulator (TEMPORAL_ROOT/{city}/COH/coh_drop_accumulator/)")

# =============================================================================
# FUNCTIONS
# =============================================================================

def sample_raster_at_centroids(tif_path, centroids_x, centroids_y, transform, band=1, dst_crs=None):
    """Sample a raster TIF at building centroid coordinates.
    If dst_crs differs from TIF CRS, reprojects centroids before sampling.
    Returns array of values (NaN where out of bounds or nodata).
    """
    with rasterio.open(tif_path) as src:
        data = src.read(band).astype(np.float32)
        src_transform = src.transform
        nodata = src.nodata
        tif_crs = src.crs
    cx, cy = centroids_x, centroids_y
    if dst_crs and tif_crs and str(dst_crs) != str(tif_crs):
        from pyproj import Transformer
        transformer = Transformer.from_crs(dst_crs, tif_crs, always_xy=True)
        cx, cy = transformer.transform(centroids_x, centroids_y)
    n = len(cx)
    vals = np.full(n, np.nan, dtype=np.float32)
    for i in range(n):
        col, row = ~src_transform * (cx[i], cy[i])
        col, row = int(round(col)), int(round(row))
        if 0 <= row < data.shape[0] and 0 <= col < data.shape[1]:
            v = data[row, col]
            if nodata is not None and v == nodata:
                continue
            if np.isfinite(v):
                vals[i] = v
    return vals


def compute_auc_mw(vals, y, min_samples=20):
    """Compute AUC + Mann-Whitney for a single feature vector vs binary target."""
    valid = np.isfinite(vals)
    v = vals[valid]
    t = y[valid]
    if len(v) < min_samples or len(np.unique(t)) < 2:
        return None
    d_vals = v[t == 1]
    c_vals = v[t == 0]
    if len(d_vals) < 5 or len(c_vals) < 5:
        return None
    try:
        u_stat, p_val = scipy_stats.mannwhitneyu(d_vals, c_vals, alternative='two-sided')
        auc = roc_auc_score(t, v)
        auc_best = max(auc, 1 - auc)
    except Exception:
        return None
    return {
        'auc_raw': float(auc),
        'auc_best': float(auc_best),
        'direction': 'higher=damaged' if auc >= 0.5 else 'lower=damaged',
        'mann_whitney_u': float(u_stat),
        'mann_whitney_p': float(p_val),
        'mean_damaged': float(np.mean(d_vals)),
        'mean_control': float(np.mean(c_vals)),
        'n_valid': int(len(v)),
    }


# =============================================================================
# COLLECT PRODUCTS PER CITY
# =============================================================================

all_rows = []

for CITY in CITIES_TO_PROCESS:
    city_df = df[df['city'] == CITY]
    if len(city_df) < MIN_SAMPLES:
        continue
    if TARGET_COL not in city_df.columns or city_df[TARGET_COL].nunique() < 2:
        continue

    # V3 sample-unit (point_id) coordinates: x_utm/y_utm are the per-point UTM
    # coordinates, equivalent in semantics to V2's centroid_x/centroid_y on
    # building footprints. Fall back to centroid_x/centroid_y (V2 shape) and
    # lon/lat (geographic) for backward compatibility.
    if 'x_utm' in city_df.columns and 'y_utm' in city_df.columns:
        cx = city_df['x_utm'].values
        cy = city_df['y_utm'].values
    elif 'centroid_x' in city_df.columns and 'centroid_y' in city_df.columns:
        cx = city_df['centroid_x'].values
        cy = city_df['centroid_y'].values
    elif 'lon' in city_df.columns and 'lat' in city_df.columns:
        cx = city_df['lon'].values
        cy = city_df['lat'].values
    else:
        print(f"    SKIP: no coordinate columns (x_utm/centroid_x/lon)")
        continue
    y_arr = city_df[TARGET_COL].values.astype(int)

    _ref_path = STACK_ROOT / CITY / 'reference_grid.json'
    _centroid_crs = None
    if _ref_path.exists():
        with open(_ref_path) as _f:
            _ref = json.load(_f)
        _centroid_crs = f"EPSG:{_ref.get('utm_epsg', 4326)}"

    print(f"\n  {CITY} ({len(city_df)} buildings)")

    # --- P7: RGB PRE/POST SINGLE-SCENE ---
    rgb_dir = MS_DIR / CITY / MS_RGB_PREPOST_SUBDIR
    if rgb_dir.exists():
        rgb_tifs = sorted(rgb_dir.glob('s2__rgb_prepost__*.tif'))
        rgb_tifs = [t for t in rgb_tifs if '_viz_' not in t.name]
        for tif in rgb_tifs:
            # extract band count
            with rasterio.open(tif) as src:
                n_bands = src.count
                src_transform = src.transform
            band_labels = ['b04', 'b03', 'b02'] if n_bands == 3 else ['val']
            # parse phase from filename: s2__rgb_prepost__pre__20210915.tif or s2__rgb_prepost__delta.tif
            m = re.match(r's2__rgb_prepost__(\w+?)(?:__\d{8})?\.tif', tif.name)
            if not m:
                continue
            phase = m.group(1)  # pre, post, delta
            for bi, blabel in enumerate(band_labels):
                feat_name = f"s2__rgb_prepost__{phase}__{blabel}"
                vals = sample_raster_at_centroids(tif, cx, cy, src_transform, band=bi+1, dst_crs=_centroid_crs)
                result = compute_auc_mw(vals, y_arr)
                if result is not None:
                    result['city'] = CITY
                    result['feature'] = feat_name
                    result['source'] = 'rgb_prepost'
                    result['product'] = 'P7'
                    all_rows.append(result)
        n_rgb = len([r for r in all_rows if r.get('city') == CITY and r.get('source') == 'rgb_prepost'])
        print(f"    P7 RGB prepost: {len(rgb_tifs)} TIFs, {n_rgb} features tested")
    else:
        print(f"    P7 RGB prepost: not found ({rgb_dir})")

    # --- P7b: CARD PRE/POST SINGLE-SCENE ---
    card_pp_dir = SAR_CARD_DIR / CITY / SAR_CARD_PREPOST_SUBDIR
    if card_pp_dir.exists():
        card_tifs = sorted(card_pp_dir.glob('s1__*.tif'))
        for tif in card_tifs:
            with rasterio.open(tif) as src:
                src_transform = src.transform
            # s1__vv__card_prepost__pre__20220101.tif or s1__vv__card_prepost__delta.tif
            m = re.match(r's1__(vv|vh)__card_prepost__(\w+?)(?:__\d{8})?\.tif', tif.name)
            if not m:
                continue
            pol = m.group(1)
            phase = m.group(2)
            feat_name = f"s1__{pol}__card_prepost__{phase}"
            vals = sample_raster_at_centroids(tif, cx, cy, src_transform, dst_crs=_centroid_crs)
            result = compute_auc_mw(vals, y_arr)
            if result is not None:
                result['city'] = CITY
                result['feature'] = feat_name
                result['source'] = 'card_prepost'
                result['product'] = 'P7b'
                all_rows.append(result)
        n_card = len([r for r in all_rows if r.get('city') == CITY and r.get('source') == 'card_prepost'])
        print(f"    P7b CARD prepost: {len(card_tifs)} TIFs, {n_card} features tested")
    else:
        print(f"    P7b CARD prepost: not found ({card_pp_dir})")

    # --- R3: COH DROP ACCUMULATOR ---
    coh_acc_dir = TEMPORAL_ROOT / CITY / 'COH' / 'coh_drop_accumulator'
    if coh_acc_dir.exists():
        # only numeric products (skip date and landuse transition TIFs)
        coh_products = [
            's1__coh__running_min.tif',
            's1__coh__drop_count.tif',
            's1__coh__max_drop.tif',
            's1__coh__scenes_observed.tif',
        ]
        for fname in coh_products:
            tif = coh_acc_dir / fname
            if not tif.exists():
                continue
            with rasterio.open(tif) as src:
                src_transform = src.transform
            feat_name = fname.replace('.tif', '')
            vals = sample_raster_at_centroids(tif, cx, cy, src_transform, dst_crs=_centroid_crs)
            result = compute_auc_mw(vals, y_arr)
            if result is not None:
                result['city'] = CITY
                result['feature'] = feat_name
                result['source'] = 'coh_accumulator'
                result['product'] = 'R3'
                all_rows.append(result)
        n_coh = len([r for r in all_rows if r.get('city') == CITY and r.get('source') == 'coh_accumulator'])
        print(f"    R3 COH accumulator: {n_coh} features tested")
    else:
        print(f"    R3 COH accumulator: not found ({coh_acc_dir})")

# =============================================================================
# RESULTS
# =============================================================================

if all_rows:
    new_auc_df = pd.DataFrame(all_rows).sort_values(['source', 'city', 'auc_best'], ascending=[True, True, False])
    save_result(new_auc_df, 'single_scene_auc', 'cell18_single_scene')

    # summary table: mean AUC per feature across cities
    print(f"\n{'='*70}")
    print("SUMMARY: Mean AUC per feature (across cities)")
    print(f"{'='*70}")
    feat_summary = new_auc_df.groupby(['source', 'feature']).agg(
        mean_auc=('auc_best', 'mean'),
        std_auc=('auc_best', 'std'),
        n_cities=('city', 'nunique'),
        mean_p=('mann_whitney_p', 'mean'),
    ).sort_values('mean_auc', ascending=False).reset_index()

    print(f"\n  {'Source':<16s} {'Feature':<40s} {'AUC':>7s} {'Std':>6s} {'Cities':>6s} {'Sig'}") 
    print(f"  {'-'*16} {'-'*40} {'-'*7} {'-'*6} {'-'*6} {'-'*4}")
    for _, row in feat_summary.iterrows():
        sig = '*' if row['mean_p'] < 0.05 else ' '
        print(f"  {row['source']:<16s} {row['feature']:<40s} {row['mean_auc']:>7.3f} {row['std_auc']:>6.3f} {int(row['n_cities']):>6d} {sig}")

    # comparison with Cell 4 composite features (if auc_df exists)
    if 'auc_df' in dir() and len(auc_df) > 0 and 'feature' in auc_df.columns:
        print(f"\n{'='*70}")
        print("COMPARISON: Single-scene vs composite/baseline features")
        print(f"{'='*70}")
        composite_mean = auc_df.groupby('feature')['auc_best'].mean()
        single_mean = new_auc_df.groupby('feature')['auc_best'].mean()
        print(f"  Composite features (Cell 4):  mean AUC = {composite_mean.mean():.3f}  (n={len(composite_mean)})")
        print(f"  Single-scene features (P7+P7b+R3): mean AUC = {single_mean.mean():.3f}  (n={len(single_mean)})")
        print(f"  Best composite: {composite_mean.idxmax():<45s} AUC={composite_mean.max():.3f}")
        print(f"  Best single:    {single_mean.idxmax():<45s} AUC={single_mean.max():.3f}")

    # registry
    _top_new = new_auc_df.groupby('feature')['auc_best'].mean().nlargest(10)
    registry.log_statistics(
        cell_id='cell18_single_scene',
        analysis_name='single_scene_accumulator_auc',
        parquet_name='TIF_direct_sampling',
        tier_selection=_tiers,
        cities=CITIES_TO_PROCESS,
        n_buildings=len(df),
        n_features_tested=new_auc_df['feature'].nunique(),
        summary_metrics={
            'mean_auc_rgb_prepost': float(new_auc_df[new_auc_df['source'] == 'rgb_prepost']['auc_best'].mean()) if (new_auc_df['source'] == 'rgb_prepost').any() else None,
            'mean_auc_card_prepost': float(new_auc_df[new_auc_df['source'] == 'card_prepost']['auc_best'].mean()) if (new_auc_df['source'] == 'card_prepost').any() else None,
            'mean_auc_coh_accumulator': float(new_auc_df[new_auc_df['source'] == 'coh_accumulator']['auc_best'].mean()) if (new_auc_df['source'] == 'coh_accumulator').any() else None,
            'n_significant_p05': int((new_auc_df['mann_whitney_p'] < 0.05).sum()),
        },
        top_features=[{'feature': f, 'mean_auc': float(a)} for f, a in _top_new.items()],
        note='Single-scene RGB/CARD pre/post + COH accumulator (Cell 18)',
        tags=['single_scene', 'rgb_prepost', 'card_prepost', 'coh_accumulator', 'auc'],
    )
else:
    print("\n  No single-scene/accumulator products found for any city.")
    print("  Check that NB03e P7, P7b, R3 have been run and products exist in:")
    print(f"    MS_DIR/{{city}}/{MS_RGB_PREPOST_SUBDIR}/")
    print(f"    SAR_CARD_DIR/{{city}}/{SAR_CARD_PREPOST_SUBDIR}/")
    print(f"    TEMPORAL_ROOT/{{city}}/COH/coh_drop_accumulator/")

print(f"\n{'='*70}")
print("CELL 18 COMPLETE")
print(f"{'='*70}")


CELL 18: SINGLE-SCENE & ACCUMULATOR PRODUCT ANALYSIS
  P7:  RGB pre/post single-scene (MS_DIR/{city}/rgb_prepost/)
  P7b: CARD pre/post single-scene (SAR_CARD_DIR/{city}/card_prepost/)
  R3:  COH drop accumulator (TEMPORAL_ROOT/{city}/COH/coh_drop_accumulator/)

  Avdiivka (1186 buildings)
    P7 RGB prepost: 4 TIFs, 12 features tested
    P7b CARD prepost: 6 TIFs, 6 features tested
    R3 COH accumulator: 4 features tested

  Bucha (1662 buildings)
    P7 RGB prepost: 3 TIFs, 9 features tested
    P7b CARD prepost: 6 TIFs, 6 features tested
    R3 COH accumulator: 0 features tested

  Chernihiv (2810 buildings)
    P7 RGB prepost: 3 TIFs, 9 features tested
    P7b CARD prepost: 6 TIFs, 6 features tested
    R3 COH accumulator: not found (/content/drive_f/masterthesis/data/satellite/temporal_products/Chernihiv/COH/coh_drop_accumulator)

  Chornobaivka (142 buildings)
    P7 RGB prepost: 3 TIFs, 9 features tested
    P7b CARD prepost: 6 TIFs, 6 features tested
    R3 COH accumulator: no

In [47]:
# @title CELL 19: SAVE RESULTS
# Save accumulated NB06 results to CSV
registry.save()
print(f"  NB06_RESULTS: {len(NB06_RESULTS)} entries")


  Registry saved: /content/drive_f/masterthesis/results/nb06v3/nb06_results_registry.csv (88 entries)
  NB06_RESULTS: 88 entries
